# 🇲🇱 Mali Knowledge Hub v0.2

**Purpose:** Trusted, evidence-grounded AI support for research, analysis and planning across Mali's institutional knowledge ecosystem.

This notebook contains the reproducible core of the Mali Knowledge Hub, combining document retrieval, external data sources and a multi-agent reasoning pipeline.

### Working principles

- Evidence first
- Precise source and page provenance
- Clear separation of fact, synthesis and inference
- Explicit uncertainty
- No unsupported claims
- Independent verification of substantive claims
- Source-independent architecture for documents and external data
- Reproducible research and analysis

### Core pipeline

**Question → Research → Evidence → Analysis → Factcheck → Grounded Rewrite → Final Answer**

## ▶️ 00 — Start Knowledge Hub

Run this section once after every new Colab runtime.

It initializes the Knowledge Hub environment without ingesting documents,
running tests, launching the application, or modifying the database.

In [ ]:
# ============================================================
# ▶️ 00 — START MALI KNOWLEDGE HUB v0.3
# ============================================================

# ------------------------------------------------------------
# 1. Install dependencies
# ------------------------------------------------------------

!pip install -q openai supabase pymupdf streamlit pandas requests


# ------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------

from google.colab import userdata
from supabase import create_client
from openai import OpenAI

import importlib.util
from pathlib import Path


# ------------------------------------------------------------
# 3. Load secrets
# ------------------------------------------------------------

SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_SECRET_KEY = userdata.get("SUPABASE_SECRET_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
HDX_HAPI_APP_IDENTIFIER = userdata.get("HDX_HAPI_APP_IDENTIFIER")

if not SUPABASE_URL:
    raise ValueError(
        "❌ Missing SUPABASE_URL in Colab Secrets"
    )

if not SUPABASE_SECRET_KEY:
    raise ValueError(
        "❌ Missing SUPABASE_SECRET_KEY in Colab Secrets"
    )

if not OPENAI_API_KEY:
    raise ValueError(
        "❌ Missing OPENAI_API_KEY in Colab Secrets"
    )
if not HDX_HAPI_APP_IDENTIFIER:
    raise ValueError(
        "❌ Missing HDX_HAPI_APP_IDENTIFIER in Colab Secrets"
    )

# ------------------------------------------------------------
# 4. Initialize clients
# ------------------------------------------------------------

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_SECRET_KEY
)

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)


# ------------------------------------------------------------
# 5. Load persistent operational runtime
# ------------------------------------------------------------

RUNTIME_BUCKET = "knowledge-hub-runtime"
RUNTIME_STORAGE_PATH = "knowledge_hub_runtime_v0_3.py"

runtime_local_path = Path(
    "/content/knowledge_hub_runtime_v0_3.py"
)

runtime_bytes = (
    supabase
    .storage
    .from_(RUNTIME_BUCKET)
    .download(RUNTIME_STORAGE_PATH)
)

runtime_local_path.write_bytes(
    runtime_bytes
)


# ------------------------------------------------------------
# 6. Import runtime module
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "knowledge_hub_runtime",
    runtime_local_path
)

knowledge_hub_runtime = (
    importlib.util.module_from_spec(spec)
)

spec.loader.exec_module(
    knowledge_hub_runtime
)

knowledge_hub_runtime.configure_runtime(
    supabase,
    openai_client,
    HDX_HAPI_APP_IDENTIFIER
)

# Expose operational functions directly in notebook
search_knowledge_base = (
    knowledge_hub_runtime.search_knowledge_base
)

research_humanitarian_needs = (
    knowledge_hub_runtime.research_humanitarian_needs
)

run_fongim_sync_extract = (
    knowledge_hub_runtime.run_fongim_sync_extract
)

run_complete_fongim_sync = (
    knowledge_hub_runtime.run_complete_fongim_sync
)


# ------------------------------------------------------------
# 7. Corpus health check
# ------------------------------------------------------------

documents_result = (
    supabase
    .table("documents")
    .select("id,title,status")
    .execute()
)

documents = documents_result.data or []

chunks_result = (
    supabase
    .table("chunks")
    .select("id", count="exact")
    .limit(1)
    .execute()
)

total_chunks = chunks_result.count or 0


# ------------------------------------------------------------
# 8. Runtime health check
# ------------------------------------------------------------

required_runtime_functions = [
    search_knowledge_base,
    research_humanitarian_needs,
    run_fongim_sync_extract,
    run_complete_fongim_sync
]

if not all(
    callable(fn)
    for fn in required_runtime_functions
):
    raise RuntimeError(
        "❌ Operational runtime did not load correctly"
    )


# ------------------------------------------------------------
# 9. Ready
# ------------------------------------------------------------

print()
print("🇲🇱 MALI KNOWLEDGE HUB v0.3")
print("─" * 50)
print("✅ Dependencies loaded")
print("✅ OpenAI connected")
print("✅ Supabase connected")
print("✅ Persistent runtime downloaded")
print("✅ Retrieval engine loaded")
print("✅ OCHA / HDX HAPI connector loaded")
print("✅ FONGIM connector loaded")
print(
    f"✅ Corpus available: "
    f"{len(documents)} documents / "
    f"{total_chunks} chunks"
)
print()
print("🚀 KNOWLEDGE HUB READY")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.0 MB/s eta 0:00:00

🇲🇱 MALI KNOWLEDGE HUB v0.3
──────────────────────────────────────────────────
✅ Dependencies loaded
✅ OpenAI connected
✅ Supabase connected
✅ Persistent runtime downloaded
✅ Retrieval engine loaded
✅ OCHA / HDX HAPI connector loaded
✅ FONGIM connector loaded
✅ Corpus available: 5 documents / 1239 chunks

🚀 KNOWLEDGE HUB READY


## 🔒 00 — Setup & Configuration

Run this section after every fresh Colab runtime.

Normally, you should not need to edit the core code below.

In [ ]:
# ============================================================
# 00 — SETUP & CONFIGURATION
# ============================================================

!pip install -q pymupdf openai supabase streamlit

from google.colab import userdata
from supabase import create_client
from openai import OpenAI

import os
import re
import json
import pymupdf

# ------------------------------------------------------------
# Secrets stored in Colab
# ------------------------------------------------------------

SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_SECRET_KEY = userdata.get("SUPABASE_SECRET_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# ------------------------------------------------------------
# Clients
# ------------------------------------------------------------

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_SECRET_KEY
)

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)

print("✅ Supabase connected")
print("✅ OpenAI connected")
print("🚀 Mali Knowledge Hub core environment ready")

✅ Supabase connected
✅ OpenAI connected
🚀 Mali Knowledge Hub core environment ready


## 🔒 01 — Database & Corpus

Checks the current Supabase corpus after every runtime restart.

This section shows:
- how many documents are stored
- which documents are active
- how many chunks exist in total
- how many chunks belong to each document

In [ ]:
# ============================================================
# 01 — DATABASE & CORPUS
# Exact counts — avoids Supabase's 1000-row result limit
# ============================================================

# Fetch document metadata
documents_result = (
    supabase
    .table("documents")
    .select(
        "id,title,organization,publication_date,"
        "valid_from,valid_until,document_type,"
        "language,geographic_scope,status,version"
    )
    .order("publication_date")
    .execute()
)

documents = documents_result.data


# ------------------------------------------------------------
# Count chunks exactly for each document
# ------------------------------------------------------------

for doc in documents:

    count_result = (
        supabase
        .table("chunks")
        .select("id", count="exact")
        .eq("document_id", doc["id"])
        .execute()
    )

    doc["chunk_count"] = count_result.count or 0


# ------------------------------------------------------------
# Exact corpus totals
# ------------------------------------------------------------

total_chunks_result = (
    supabase
    .table("chunks")
    .select("id", count="exact")
    .execute()
)

total_chunks = total_chunks_result.count or 0

active_documents = [
    d for d in documents
    if d.get("status") == "active"
]

populated_documents = [
    d for d in documents
    if d["chunk_count"] > 0
]


# ------------------------------------------------------------
# Display corpus
# ------------------------------------------------------------

print("=" * 100)
print("🇲🇱 MALI KNOWLEDGE HUB — CORPUS STATUS")
print("=" * 100)

print(f"\n📚 Document records:    {len(documents)}")
print(f"📄 Populated documents: {len(populated_documents)}")
print(f"🧩 Total chunks:        {total_chunks}")

print("\nDOCUMENTS")
print("-" * 100)

for i, doc in enumerate(documents, 1):

    print(f"\n{i}. {doc['title']}")
    print(f"   ID:           {doc['id']}")
    print(f"   Organization: {doc.get('organization')}")
    print(f"   Type:         {doc.get('document_type')}")
    print(f"   Version:      {doc.get('version')}")
    print(f"   Status:       {doc.get('status')}")
    print(f"   Chunks:       {doc['chunk_count']}")

    if doc["chunk_count"] == 0:
        print("   ⚠️ EMPTY DOCUMENT RECORD")

print("\n" + "=" * 100)

if len(documents) != len(populated_documents):
    print(
        f"⚠️ {len(documents) - len(populated_documents)} "
        "empty document record(s) detected"
    )
else:
    print("✅ No empty document records detected")

print("✅ Corpus check complete")

🇲🇱 MALI KNOWLEDGE HUB — CORPUS STATUS

📚 Document records:    5
📄 Populated documents: 5
🧩 Total chunks:        1239

DOCUMENTS
----------------------------------------------------------------------------------------------------

1. Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024-2033)
   ID:           f88ef966-bbbc-4c2e-85b6-b01b6fe929a5
   Organization: Ministère de l’Économie et des Finances — République du Mali
   Type:         Stratégie nationale
   Version:      2024-2033
   Status:       active
   Chunks:       277

2. Vision Mali 2063 — Mali Kura Ɲɛtaasira ka bɛn san 2063 ma
   ID:           495a9123-89f9-448e-986a-48fb27a0774b
   Organization: Ministère de l’Économie et des Finances — République du Mali
   Type:         Vision nationale de développement
   Version:      2063
   Status:       active
   Chunks:       602

3. Projets Structurants Prioritaires pour la mise en œuvre de Mali Kura 2063 et de la SNEDD 2024-2033
   ID:           e6722d26-b6

## 🔒 02 — PDF Processing

Reusable PDF parsing and chunking logic.

This section:
- validates that a PDF can be opened
- checks whether enough text is extractable
- cleans problematic characters
- detects headings and numbered paragraphs
- creates page-level structured chunks

Normally, you should not need to edit this section.

In [ ]:
# ============================================================
# 02 — PDF PROCESSING
# ============================================================

def clean_text(text):
    """
    Clean extracted PDF text before storing it.
    """
    if not text:
        return ""

    # PostgreSQL cannot store null bytes
    text = text.replace("\x00", "")

    # Remove soft hyphens
    text = text.replace("\u00ad", "")

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


def clean_line(line):
    """
    Clean a single extracted line.
    """
    return clean_text(line)


def is_heading(line):
    """
    Detect common heading patterns in French government documents.
    This is deliberately conservative.
    """
    patterns = [
        r"^[IVX]+\.\s*$",
        r"^[IVX]+\.\s+.+",
        r"^\d+\.\d+\.\d+\.?\s+.+",
        r"^\d+\.\d+\.?\s+.+",
        r"^Axe [Ss]tratégique.+",
        r"^Objectif [Gg]lobal.+",
        r"^Objectif [Ss]pécifique.+",
        r"^Introduction$",
        r"^Conclusion$",
        r"^Contexte et Justification$",
        r"^Le diagnostic stratégique$",
        r"^Les Axes stratégiques$",
    ]

    return any(
        re.match(pattern, line)
        for pattern in patterns
    )


def starts_numbered_paragraph(line):
    """
    Detect paragraphs such as:
    137. Le Mali...
    24. Le Gouvernement...
    """
    return bool(
        re.match(r"^\d{1,3}\.\s+.+", line)
    )


def validate_pdf(pdf_path, min_chars_per_page=100):
    """
    Basic integrity and extraction-quality check.

    Returns statistics rather than silently accepting
    a PDF with mostly unreadable pages.
    """

    if not os.path.exists(pdf_path):
        raise FileNotFoundError(
            f"PDF not found: {pdf_path}"
        )

    doc = pymupdf.open(pdf_path)

    page_stats = []

    for page_idx in range(doc.page_count):

        raw_text = doc[page_idx].get_text("text")
        text = clean_text(raw_text)

        page_stats.append({
            "page": page_idx + 1,
            "characters": len(text)
        })

    readable_pages = [
        p for p in page_stats
        if p["characters"] >= min_chars_per_page
    ]

    low_text_pages = [
        p for p in page_stats
        if p["characters"] < min_chars_per_page
    ]

    readable_ratio = (
        len(readable_pages) / doc.page_count
        if doc.page_count
        else 0
    )

    result = {
        "page_count": doc.page_count,
        "readable_pages": len(readable_pages),
        "low_text_pages": len(low_text_pages),
        "readable_ratio": readable_ratio,
        "page_stats": page_stats
    }

    doc.close()

    return result


def chunk_pdf(pdf_path, max_chunk_chars=2200):
    """
    Parse a PDF page by page and create structured chunks.

    Each chunk preserves:
    - page number
    - chunk index
    - detected section title
    - original cleaned content
    """

    doc = pymupdf.open(pdf_path)

    chunks = []
    chunk_index = 0
    current_section = None

    for page_idx in range(doc.page_count):

        page_number = page_idx + 1

        raw_text = doc[page_idx].get_text("text")
        raw_text = clean_text(raw_text)

        lines = [
            clean_line(line)
            for line in raw_text.splitlines()
        ]

        lines = [
            line for line in lines
            if line
        ]

        # Remove simple printed page number
        if lines and lines[0] == str(page_number):
            lines = lines[1:]

        segments = []
        current_segment = []

        for line in lines:

            if is_heading(line):

                if current_segment:
                    segments.append({
                        "text": " ".join(current_segment),
                        "section": current_section
                    })

                    current_segment = []

                current_section = line

                segments.append({
                    "text": line,
                    "section": current_section
                })

                continue

            if starts_numbered_paragraph(line):

                if current_segment:
                    segments.append({
                        "text": " ".join(current_segment),
                        "section": current_section
                    })

                current_segment = [line]

            else:
                current_segment.append(line)

        if current_segment:
            segments.append({
                "text": " ".join(current_segment),
                "section": current_section
            })

        buffer = []
        buffer_section = None

        for segment in segments:

            # Start a new chunk when section changes
            if buffer and segment["section"] != buffer_section:

                chunks.append({
                    "page_number": page_number,
                    "chunk_index": chunk_index,
                    "section_title": buffer_section,
                    "content": "\n\n".join(buffer)
                })

                chunk_index += 1
                buffer = []

            if not buffer:
                buffer_section = segment["section"]

            candidate = "\n\n".join(
                buffer + [segment["text"]]
            )

            # Split oversized chunks
            if len(candidate) > max_chunk_chars and buffer:

                chunks.append({
                    "page_number": page_number,
                    "chunk_index": chunk_index,
                    "section_title": buffer_section,
                    "content": "\n\n".join(buffer)
                })

                chunk_index += 1

                buffer = [segment["text"]]
                buffer_section = segment["section"]

            else:
                buffer.append(segment["text"])

        if buffer:

            chunks.append({
                "page_number": page_number,
                "chunk_index": chunk_index,
                "section_title": buffer_section,
                "content": "\n\n".join(buffer)
            })

            chunk_index += 1

    doc.close()

    return chunks


print("✅ PDF processing functions loaded")

✅ PDF processing functions loaded


## 🔒 03 — Ingestion

Safe reusable document ingestion.

This section:
- validates the PDF first
- aborts if extraction quality is suspicious
- creates the document record
- creates embeddings in batches
- uploads chunks to Supabase
- returns the new document ID

Normally, you should not edit the function itself.

In [ ]:
# ============================================================
# 03 — INGESTION
# ============================================================

def ingest_document(
    pdf_path,
    metadata,
    batch_size=50,
    min_readable_ratio=0.80
):
    """
    Validate, chunk, embed and upload one document.

    Safety principle:
    Do not silently ingest a PDF if most pages appear unreadable.
    """

    # --------------------------------------------------------
    # 1. Validate PDF
    # --------------------------------------------------------

    validation = validate_pdf(pdf_path)

    print(f"📄 Pages: {validation['page_count']}")
    print(
        f"✅ Readable pages: "
        f"{validation['readable_pages']}/{validation['page_count']} "
        f"({validation['readable_ratio']:.1%})"
    )

    if validation["readable_ratio"] < min_readable_ratio:
        raise ValueError(
            "❌ INGESTION STOPPED: PDF extraction quality is too low.\n"
            f"Readable ratio: {validation['readable_ratio']:.1%}\n"
            "Check whether the PDF is incomplete, scanned, corrupted, "
            "or requires OCR."
        )

    # --------------------------------------------------------
    # 2. Chunk PDF
    # --------------------------------------------------------

    chunks = chunk_pdf(pdf_path)

    # Remove empty chunks just in case
    chunks = [
        c for c in chunks
        if c.get("content", "").strip()
    ]

    print(f"🧩 Chunks created: {len(chunks)}")

    if not chunks:
        raise ValueError(
            "❌ INGESTION STOPPED: No usable chunks were created."
        )

    # --------------------------------------------------------
    # 3. Create document record
    # --------------------------------------------------------

    doc_result = (
        supabase
        .table("documents")
        .insert(metadata)
        .execute()
    )

    document_id = doc_result.data[0]["id"]

    print("✅ Document record created:", document_id)

    # --------------------------------------------------------
    # 4. Create embeddings
    # --------------------------------------------------------

    embedded_chunks = []

    try:

        for start in range(0, len(chunks), batch_size):

            batch = chunks[start:start + batch_size]

            texts_for_embedding = []

            for c in batch:

                context = (
                    f"Document: {metadata['title']}\n"
                    f"Organization: {metadata.get('organization', '')}\n"
                    f"Document type: {metadata.get('document_type', '')}\n"
                    f"Version: {metadata.get('version', '')}\n"
                    f"Section: {c['section_title'] or 'Non spécifiée'}\n"
                    f"Page: {c['page_number']}\n\n"
                    f"{c['content']}"
                )

                texts_for_embedding.append(context)

            response = openai_client.embeddings.create(
                model="text-embedding-3-small",
                input=texts_for_embedding
            )

            for c, emb in zip(batch, response.data):

                embedded_chunks.append({
                    "document_id": document_id,
                    "page_number": c["page_number"],
                    "chunk_index": c["chunk_index"],
                    "section_title": c["section_title"],
                    "content": c["content"],
                    "embedding": emb.embedding
                })

            print(
                f"Embedded "
                f"{min(start + batch_size, len(chunks))}/"
                f"{len(chunks)}"
            )

        # ----------------------------------------------------
        # 5. Upload chunks
        # ----------------------------------------------------

        for start in range(0, len(embedded_chunks), batch_size):

            batch = embedded_chunks[start:start + batch_size]

            (
                supabase
                .table("chunks")
                .insert(batch)
                .execute()
            )

            print(
                f"Uploaded "
                f"{min(start + batch_size, len(embedded_chunks))}/"
                f"{len(embedded_chunks)}"
            )

    except Exception as e:

        # ----------------------------------------------------
        # Cleanup if ingestion fails after document creation
        # ----------------------------------------------------

        print("\n⚠️ Ingestion failed.")
        print("Cleaning up incomplete document record...")

        (
            supabase
            .table("documents")
            .delete()
            .eq("id", document_id)
            .execute()
        )

        print("✅ Incomplete document record removed")

        raise e

    # --------------------------------------------------------
    # 6. Success
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("🎉 INGESTION COMPLETE")
    print("=" * 80)

    print("Title:", metadata["title"])
    print("Document ID:", document_id)
    print("Pages:", validation["page_count"])
    print("Chunks:", len(embedded_chunks))

    return document_id


print("✅ Ingestion function loaded")

✅ Ingestion function loaded


In [ ]:
# ============================================================
# 03.1 — GENERIC DOCUMENT INGEST
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. Select uploaded PDF
# ------------------------------------------------------------

pdf_files = list(Path("/content").glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError("❌ No PDF found in /content")

print("PDFs found:")
for i, p in enumerate(pdf_files, start=1):
    print(f"{i}. {p.name}")

# For now: use the most recently uploaded PDF
pdf_path = str(max(pdf_files, key=lambda p: p.stat().st_mtime))

print("\nSelected:")
print(pdf_path)


# ------------------------------------------------------------
# 2. Metadata
# ------------------------------------------------------------

metadata = {
    "title": "Mali — Besoins humanitaires et Plan de Réponse 2026",
    "organization": "OCHA / Équipe Humanitaire Pays — Mali",
    "document_type": "Humanitarian Needs and Response Plan",
    "version": "2026",
    "status": "active"
}


# ------------------------------------------------------------
# 3. Ingest
# ------------------------------------------------------------

document_id = ingest_document(
    pdf_path=pdf_path,
    metadata=metadata
)

print("\n✅ Document ingested")
print("Document ID:", document_id)

FileNotFoundError: ❌ No PDF found in /content

## 🔒 04 — Retrieval

Semantic search across the complete Knowledge Hub corpus.

This section:
- embeds the user's question
- searches all stored chunks by semantic similarity
- preserves document provenance
- enriches every result with document metadata
- returns ranked evidence for downstream analysis

Normally, you should not need to edit this section.

In [ ]:
# ============================================================
# 04 — RETRIEVAL
# ============================================================

def search_knowledge_base(
    question,
    match_count=12,
    filter_document_ids=None,
    similarity_threshold=None
):
    """
    Search the Knowledge Hub document corpus.

    Optional document filtering allows retrieval to be restricted
    to a defined subset of documents before vector ranking.

    Returns ranked chunks enriched with document metadata
    so provenance is preserved throughout the pipeline.
    """

    # --------------------------------------------------------
    # 1. Embed the question
    # --------------------------------------------------------

    embedding_response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )

    query_embedding = embedding_response.data[0].embedding


    # --------------------------------------------------------
    # 2. Vector search in Supabase
    # --------------------------------------------------------

    response = supabase.rpc(
        "match_chunks",
        {
            "query_embedding": query_embedding,
            "match_count": match_count,
            "filter_document_ids": filter_document_ids
        }
    ).execute()

    raw_results = response.data or []


    # --------------------------------------------------------
    # 3. Optional similarity threshold
    # --------------------------------------------------------

    if similarity_threshold is not None:

        raw_results = [
            r
            for r in raw_results
            if (
                r.get("similarity") is not None
                and r.get("similarity") >= similarity_threshold
            )
        ]


    # --------------------------------------------------------
    # 4. Load document metadata once
    # --------------------------------------------------------

    document_ids = list({
        r.get("document_id")
        for r in raw_results
        if r.get("document_id")
    })

    documents_by_id = {}

    if document_ids:

        docs_response = (
            supabase
            .table("documents")
            .select(
                "id,title,organization,publication_date,"
                "valid_from,valid_until,document_type,"
                "language,geographic_scope,status,version"
            )
            .in_("id", document_ids)
            .execute()
        )

        documents_by_id = {
            d["id"]: d
            for d in docs_response.data
        }


    # --------------------------------------------------------
    # 5. Enrich every chunk with provenance
    # --------------------------------------------------------

    results = []

    for r in raw_results:

        doc = documents_by_id.get(
            r.get("document_id"),
            {}
        )

        results.append({
            **r,

            "document_title": doc.get("title"),
            "organization": doc.get("organization"),
            "publication_date": doc.get("publication_date"),
            "valid_from": doc.get("valid_from"),
            "valid_until": doc.get("valid_until"),
            "document_type": doc.get("document_type"),
            "language": doc.get("language"),
            "geographic_scope": doc.get("geographic_scope"),
            "document_status": doc.get("status"),
            "version": doc.get("version")
        })

    return results


print("✅ Retrieval function loaded")

✅ Retrieval function loaded


In [ ]:
# ============================================================
# TEMP TEST — FILTERED DOCUMENT RETRIEVAL
# ============================================================

# 1. Load the document registry
documents = (
    supabase
    .table("documents")
    .select("id,title,document_type")
    .execute()
    .data
)

print("DOCUMENTS")
print("=" * 70)

for doc in documents:
    print(
        doc["document_type"],
        "|",
        doc["title"],
        "|",
        doc["id"]
    )


# 2. Separate HNRP from government documents
hnrp_document_ids = [
    doc["id"]
    for doc in documents
    if doc["document_type"] == "Humanitarian Needs and Response Plan"
]

government_document_ids = [
    doc["id"]
    for doc in documents
    if doc["document_type"] != "Humanitarian Needs and Response Plan"
]


print("\n" + "=" * 70)
print("DOCUMENT GROUPS")
print("=" * 70)

print("Government documents:", len(government_document_ids))
print("HNRP documents:", len(hnrp_document_ids))


# 3. Run a GOVERNMENT-ONLY retrieval
results = search_knowledge_base(
    """
    What are Mali's long-term development priorities,
    strategic objectives and structural development agenda?
    """,
    match_count=15,
    filter_document_ids=government_document_ids
)


print("\n" + "=" * 70)
print("GOVERNMENT-ONLY RETRIEVAL")
print("=" * 70)

print("Results:", len(results))

for i, result in enumerate(results, 1):
    print(
        f"{i:02d}.",
        result.get("document_title"),
        "|",
        result.get("document_type"),
        "| similarity:",
        round(result.get("similarity", 0), 3)
    )


# 4. Safety check
returned_ids = {
    result.get("document_id")
    for result in results
}

unexpected_ids = (
    returned_ids
    - set(government_document_ids)
)

print("\n" + "=" * 70)
print("FILTER CHECK")
print("=" * 70)

if unexpected_ids:
    print("❌ FILTER FAILED")
    print("Unexpected document IDs:", unexpected_ids)
else:
    print("✅ FILTER WORKS")
    print("All retrieved chunks come from government documents.")

## 🔒 05 — Answer Engine

Generates answers exclusively from retrieved Knowledge Hub evidence.

The Answer Engine:
- receives structured evidence from Retrieval
- assigns every passage a stable evidence ID
- preserves document, page and section provenance
- prohibits unsupported claims
- distinguishes supported answers from evidence gaps
- requires citations for substantive claims

No external knowledge is used by default.

In [ ]:
# ============================================================
# 05 — ANSWER ENGINE
# ============================================================

def build_evidence_bundle(results):
    """
    Convert retrieval results into a structured evidence bundle.

    Each passage receives a stable evidence ID.
    """

    evidence = []

    for i, r in enumerate(results, 1):

        evidence.append({
            "evidence_id": f"E{i}",
            "document_id": r.get("document_id"),
            "document_title": r.get("document_title"),
            "document_type": r.get("document_type"),
            "organization": r.get("organization"),
            "version": r.get("version"),
            "page": r.get("page_number"),
            "section": r.get("section_title"),
            "similarity": r.get("similarity"),
            "content": r.get("content")
        })

    return evidence


def generate_grounded_answer(
    question,
    match_count=12,
    model="gpt-5-mini"
):
    """
    Retrieve evidence and generate a grounded answer.

    Returns both the answer and the evidence used,
    allowing provenance to be inspected afterwards.
    """

    # --------------------------------------------------------
    # 1. Retrieve evidence
    # --------------------------------------------------------

    results = search_knowledge_base(
        question,
        match_count=match_count
    )

    evidence = build_evidence_bundle(results)


    # --------------------------------------------------------
    # 2. Format evidence explicitly
    # --------------------------------------------------------

    evidence_text = ""

    for e in evidence:

        evidence_text += f"""
[{e['evidence_id']}]
DOCUMENT: {e['document_title']}
DOCUMENT_ID: {e['document_id']}
TYPE: {e['document_type']}
ORGANIZATION: {e['organization']}
VERSION: {e['version']}
PAGE: {e['page']}
SECTION: {e['section']}

CONTENT:
{e['content']}

---
"""


    # --------------------------------------------------------
    # 3. Grounded analyst prompt
    # --------------------------------------------------------

    system_prompt = """
You are the analytical engine of the Mali Knowledge Hub.

Your task is to answer questions using ONLY the evidence passages
provided to you.

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent facts, policies, targets, relationships or citations.
3. Every substantive factual claim must be supported by the evidence.
4. Cite claims using the exact evidence IDs provided, for example [E3].
5. Never attribute a claim to a document unless that evidence passage
   explicitly comes from that document.
6. Distinguish carefully between:
   - current policy or strategic priorities
   - diagnosis of existing problems
   - targets
   - past results
   - recommendations
   - scenarios or hypothetical futures
7. Scenario passages describe hypothetical or alternative futures.
   They must NEVER be used as evidence for current conditions,
   current government policy, current priorities, or observed results.
   If relevant, they may only be discussed explicitly as scenarios.
8. If the evidence is insufficient, explicitly say so.
9. If documents differ in emphasis, preserve that distinction.
10. Prefer synthesis over simply listing retrieved passages.

At the end provide:

EVIDENCE STATUS: SUPPORTED
or
EVIDENCE STATUS: PARTIAL
or
EVIDENCE STATUS: INSUFFICIENT

Use the language of the user's question.
"""


    user_prompt = f"""
QUESTION:

{question}

EVIDENCE:

{evidence_text}

Answer the question based exclusively on this evidence.
"""


    # --------------------------------------------------------
    # 4. Generate answer
    # --------------------------------------------------------

    response = openai_client.responses.create(
        model=model,
        instructions=system_prompt,
        input=user_prompt
    )

    answer = response.output_text


    return {
        "question": question,
        "answer": answer,
        "evidence": evidence
    }


print("✅ Answer Engine loaded")

✅ Answer Engine loaded


In [ ]:
# ============================================================
# 05.1 — GROUNDED ANSWER TEST
# ============================================================

question = """
How do the humanitarian needs identified for Mali in 2026 relate to the country's longer-term development priorities, and where are the main areas of alignment or tension?
"""

test = generate_grounded_answer(
    question,
    match_count=12
)

print("=" * 100)
print("ANSWER")
print("=" * 100)

print(test["answer"])


print("\n\n" + "=" * 100)
print("CITATION MAP")
print("=" * 100)

for e in test["evidence"]:

    print(
        f"\n[{e['evidence_id']}] "
        f"{e['document_title']} | "
        f"p. {e['page']} | "
        f"{e['section']}"
    )

ANSWER
Summary

- Strategic alignment: The HNRP 2026 is explicitly designed to be complementary to Mali’s longer‑term development frameworks (CSRE 2022–2031, Vision 2063, SNEDD 2024–2033, UNSDCF). It states that humanitarian priorities are articulated around development programmes and the government’s vision, and that shock analyses, target maps and an inventory of activities not covered by the HNRP (e.g., durable infrastructure, institutional strengthening, local economic development) will be shared with development partners to clarify roles and avoid gaps or duplication [E1].

- Operational alignment: The humanitarian plan prioritizes measures (protection, access to essential services, risk reduction) that overlap with development objectives of restoring services and stabilizing communities; it also commits to localization and strengthening national/local actors, which supports transition from relief to development-led solutions [E3, E6, E11].

Main areas of alignment

1. Shared obje

## 🧪 06 — Testing & Debugging

Reusable diagnostics for checking Knowledge Hub quality.

Use this section when:
- retrieval results look suspicious
- citations appear incorrect
- a document seems over- or under-represented
- an answer may be using evidence incorrectly

Known limitation in v0.1:
**Evidence semantics are not yet structurally classified.**
The system may retrieve scenarios, diagnoses, targets and policy priorities together.
The Answer Engine is instructed to distinguish them, but this is not yet guaranteed by metadata.

### 🧪 06.1 — Retrieval & Provenance Test

Safe diagnostic test.

Checks whether semantic retrieval returns relevant evidence from across the corpus and whether each retrieved chunk retains its correct document identity, page and section.*kursiver Text*

### 🧪 06.2 — Grounded Answer Test

End-to-end test of:

**Question → Retrieval → Evidence Bundle → Answer**

After generation, this cell displays both the answer and the exact evidence map used to produce it.

### Known limitations — v0.1

**1. Evidence semantics**  
Retrieved passages are not yet structurally classified as policy priorities,
diagnoses, targets, results, recommendations or scenarios.

**2. Cross-document retrieval**  
Global top-k semantic retrieval may concentrate results in one highly relevant
or larger document. Relevant evidence from other documents may therefore be
absent from the evidence bundle even when it would materially improve the answer.

## 🚀 07 — Application

User-facing Streamlit interface for the Mali Knowledge Hub.

The application:
- accepts natural-language questions
- searches the closed institutional corpus
- generates evidence-grounded answers
- displays citations and source passages
- keeps the underlying retrieval and provenance visible for verification

### 🚀 07.1 — Build Application

Creates the Streamlit application from the Knowledge Hub core.

In [ ]:
# ============================================================
# 07.1 — BUILD STREAMLIT APPLICATION
# ============================================================

app_code = r'''
import os
import re
import json
import unicodedata
import importlib.util
import time
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import streamlit as st

from supabase import create_client
from openai import OpenAI


# ============================================================
# CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Mali Knowledge Hub",
    page_icon="🇲🇱",
    layout="wide"
)


def get_secret(name):
    try:
        return st.secrets[name]
    except (KeyError, FileNotFoundError):
        return os.environ[name]


SUPABASE_URL = get_secret("SUPABASE_URL")
SUPABASE_SECRET_KEY = get_secret("SUPABASE_SECRET_KEY")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
HDX_HAPI_APP_IDENTIFIER = get_secret(
    "HDX_HAPI_APP_IDENTIFIER"
)


supabase = create_client(
    SUPABASE_URL,
    SUPABASE_SECRET_KEY
)

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)


# ============================================================
# OPERATIONAL RUNTIME
# ============================================================

RUNTIME_BUCKET = "knowledge-hub-runtime"
RUNTIME_STORAGE_PATH = "knowledge_hub_runtime_v0_3.py"

runtime_local_path = Path(
    "/tmp/knowledge_hub_runtime_v0_3.py"
)

runtime_bytes = (
    supabase
    .storage
    .from_(RUNTIME_BUCKET)
    .download(RUNTIME_STORAGE_PATH)
)

runtime_local_path.write_bytes(
    runtime_bytes
)

spec = importlib.util.spec_from_file_location(
    "knowledge_hub_runtime",
    runtime_local_path
)

knowledge_hub_runtime = (
    importlib.util.module_from_spec(spec)
)

spec.loader.exec_module(
    knowledge_hub_runtime
)

knowledge_hub_runtime.configure_runtime(
    supabase,
    openai_client,
    HDX_HAPI_APP_IDENTIFIER
)


# ============================================================
# OPERATIONAL SOURCE FUNCTIONS
# ============================================================

search_knowledge_base = (
    knowledge_hub_runtime.search_knowledge_base
)

research_humanitarian_needs = (
    knowledge_hub_runtime.research_humanitarian_needs
)

run_fongim_sync_extract = (
    knowledge_hub_runtime.run_fongim_sync_extract
)


# ============================================================
# SOURCE REGISTRY
# ============================================================

SOURCE_REGISTRY = {
    "Government Framework Documents": [
        "Vision Mali 2063 — Mali Kura Ɲɛtaasira ka bɛn san 2063 ma",
        "Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024–2033)",
        "Projets Structurants Prioritaires pour la mise en œuvre de Mali Kura 2063 et de la SNEDD 2024–2033",
        "Phasage des Projets Structurants Prioritaires"
    ],
    "Humanitarian Needs Assessment": [
        "Mali — Besoins humanitaires et Plan de Réponse 2026 (OCHA / Équipe Humanitaire Pays)"
    ],
    "OCHA Database": [
        "Live structured humanitarian-needs data queried through HDX HAPI"
    ],
    "International NGO Activities": [
        "Structured FONGIM project, location, sector and organization data from the synchronized Knowledge Hub operational mirror"
    ]
}


def is_source_inventory_question(question):
    """
    Detect questions about the Knowledge Hub's own source inventory.

    This must be broader than exact phrase matching because users may ask:
    - What are your sources?
    - What data do you use?
    - Which databases can you access?
    - What's in the Knowledge Hub?
    - Where does your information come from?
    """

    q = normalize_text(question).strip()

    if not q:
        return False

    exact_or_near_exact = [
        "what are your sources",
        "what are the sources",
        "what sources do you use",
        "which sources do you use",
        "what sources do you have",
        "which sources do you have",
        "what sources do you have access to",
        "which sources do you have access to",
        "what data do you use",
        "which data do you use",
        "what data do you have",
        "which data do you have",
        "what data can you access",
        "which data can you access",
        "what databases do you use",
        "which databases do you use",
        "what databases can you access",
        "which databases can you access",
        "what documents do you have",
        "which documents do you have",
        "what documents can you access",
        "which documents can you access",
        "what evidence do you use",
        "which evidence do you use",
        "where does your information come from",
        "where do your data come from",
        "where does your data come from",
        "what is in the knowledge hub",
        "whats in the knowledge hub",
        "what does the knowledge hub contain",
        "quelles sont tes sources",
        "quelles sont vos sources",
        "quelles sources utilises tu",
        "quelles sources utilisez vous",
        "quelles donnees utilises tu",
        "quelles donnees utilisez vous",
        "a quelles sources as tu acces",
        "a quelles sources avez vous acces",
        "welche quellen hast du",
        "welche quellen nutzt du",
        "auf welche quellen hast du zugriff",
        "welche daten nutzt du",
        "welche daten hast du",
        "auf welche daten hast du zugriff",
        "welche dokumente hast du",
        "auf welche dokumente hast du zugriff"
    ]

    if any(
        phrase in q
        for phrase in exact_or_near_exact
    ):
        return True

    # Robust semantic-style rule for short platform-inventory questions.
    source_terms = [
        "source",
        "sources",
        "data source",
        "data sources",
        "database",
        "databases",
        "document",
        "documents",
        "evidence",
        "quellen",
        "quelle",
        "daten",
        "dokumente",
        "sources",
        "donnees",
        "documents"
    ]

    inventory_terms = [
        "your",
        "you use",
        "you have",
        "you access",
        "available",
        "access to",
        "knowledge hub",
        "tes ",
        "vos ",
        "utilises",
        "utilisez",
        "acces",
        "hast du",
        "nutzt du",
        "zugriff",
        "verfugbar"
    ]

    has_source_term = any(
        term in q
        for term in source_terms
    )

    has_inventory_term = any(
        term in q
        for term in inventory_terms
    )

    # Keep this deliberately limited to short questions so that
    # substantive research questions mentioning "sources" are not hijacked.
    if (
        has_source_term
        and has_inventory_term
        and len(q.split()) <= 14
    ):
        return True

    return False


def source_inventory_answer():
    return """
I currently have access to **four source families**:

**1. Government Framework Documents**
- Vision Mali 2063 — *Mali Kura Ɲɛtaasira ka bɛn san 2063 ma*
- Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024–2033)
- Projets Structurants Prioritaires for implementation of Mali Kura 2063 and SNEDD 2024–2033
- Phasage des Projets Structurants Prioritaires

**2. Humanitarian Needs Assessment**
- *Mali — Besoins humanitaires et Plan de Réponse 2026* (OCHA / Équipe Humanitaire Pays)

**3. OCHA Database**
- Live structured humanitarian-needs data queried through **HDX HAPI**. The Hub retrieves only records relevant to the question and preserves the source geography and categories.

**4. International NGO Activities**
- Structured **FONGIM** project data, including projects, locations, sectors and organizations. The app queries the synchronized Knowledge Hub operational mirror of the FONGIM source data.

The language model itself is **not** treated as a source. For analytical questions, the Hub selects the relevant source families and retrieves fresh evidence for that question.
""".strip()


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_text(value):
    value = str(value or "")

    value = unicodedata.normalize(
        "NFKD",
        value
    )

    value = "".join(
        c
        for c in value
        if not unicodedata.combining(c)
    )

    return value.lower().strip()


# ============================================================
# SUPABASE PAGINATION
# ============================================================

def fetch_all_rows(
    table_name,
    columns="*",
    eq_filters=None,
    page_size=1000
):

    if eq_filters is None:
        eq_filters = {}

    rows = []
    start = 0

    while True:

        query = (
            supabase
            .table(table_name)
            .select(columns)
        )

        for key, value in eq_filters.items():

            query = query.eq(
                key,
                value
            )

        response = (
            query
            .range(
                start,
                start + page_size - 1
            )
            .execute()
        )

        batch = response.data or []

        rows.extend(batch)

        if len(batch) < page_size:
            break

        start += page_size

    return rows


# ============================================================
# GEOGRAPHY RESOLVER
# ============================================================

@st.cache_data(ttl=3600)
def get_fongim_geographies():

    rows = fetch_all_rows(
        "fongim_project_locations",
        columns="region,cercle",
        eq_filters={
            "is_present_in_source": True
        }
    )

    regions = sorted({
        row.get("region")
        for row in rows
        if row.get("region")
    })

    cercles = sorted({
        row.get("cercle")
        for row in rows
        if row.get("cercle")
    })

    return {
        "regions": regions,
        "cercles": cercles
    }


def find_name_in_question(
    question,
    candidates
):

    q = normalize_text(question)

    matches = []

    for candidate in candidates:

        normalized_candidate = normalize_text(
            candidate
        )

        pattern = (
            r"(?<!\w)"
            + re.escape(normalized_candidate)
            + r"(?!\w)"
        )

        if re.search(
            pattern,
            q
        ):
            matches.append(candidate)

    matches = sorted(
        matches,
        key=lambda x: len(str(x)),
        reverse=True
    )

    return matches


def resolve_geography(question):

    geographies = get_fongim_geographies()

    region_matches = find_name_in_question(
        question,
        geographies["regions"]
    )

    cercle_matches = find_name_in_question(
        question,
        geographies["cercles"]
    )

    q = normalize_text(question)

    explicit_region = None
    explicit_cercle = None

    for region in region_matches:

        r = normalize_text(region)

        if (
            f"region de {r}" in q
            or f"region du {r}" in q
            or f"region d'{r}" in q
            or f"region {r}" in q
        ):
            explicit_region = region
            break

    for cercle in cercle_matches:

        c = normalize_text(cercle)

        if (
            f"cercle de {c}" in q
            or f"cercle du {c}" in q
            or f"cercle d'{c}" in q
            or f"cercle {c}" in q
        ):
            explicit_cercle = cercle
            break

    if explicit_cercle:

        return {
            "region": explicit_region,
            "cercle": explicit_cercle,
            "assumption": None
        }

    if explicit_region:

        return {
            "region": explicit_region,
            "cercle": None,
            "assumption": None
        }

    if region_matches:

        region = region_matches[0]

        assumption = None

        if any(
            normalize_text(c)
            == normalize_text(region)
            for c in cercle_matches
        ):

            assumption = (
                f"Interpreted '{region}' as the region, "
                f"not the cercle."
            )

        return {
            "region": region,
            "cercle": None,
            "assumption": assumption
        }

    if cercle_matches:

        return {
            "region": None,
            "cercle": cercle_matches[0],
            "assumption": None
        }

    return {
        "region": None,
        "cercle": None,
        "assumption": None
    }


# ============================================================
# SOURCE ROUTER
# ============================================================

HUMANITARIAN_KEYWORDS = [
    "humanitarian",
    "humanitaire",
    "besoin",
    "needs",
    "people in need",
    "personnes dans le besoin",
    "pin",
    "food security",
    "securite alimentaire",
    "nutrition",
    "wash",
    "eha",
    "health",
    "sante",
    "protection",
    "gbv",
    "violence basee sur le genre",
    "mine action",
    "deplacement",
    "deplace",
    "displacement"
]


FONGIM_KEYWORDS = [
    "project",
    "projet",
    "intervention",
    "programme",
    "organization",
    "organisation",
    "ong",
    "ngo",
    "actor",
    "acteur",
    "partner",
    "partenaire",
    "coverage",
    "couverture",
    "who works",
    "qui intervient",
    "sector",
    "secteur",
    "presence",
    "présence"
]


def contains_any_keyword(
    question,
    keywords
):

    q = normalize_text(question)

    return any(
        normalize_text(keyword) in q
        for keyword in keywords
    )


GOVERNMENT_KEYWORDS = [
    "government", "gouvernement", "etat", "state",
    "policy", "politique", "strategy", "strategie",
    "strategic", "priority", "priorite", "priorities",
    "vision", "snedd", "mali 2063", "mali kura",
    "projet structurant", "structural project",
    "local development", "developpement local",
    "decentralisation", "decentralization",
    "governance", "gouvernance"
]


CROSS_SOURCE_KEYWORDS = [
    "compare", "comparison", "comparer", "comparaison",
    "gap", "gaps", "lacune", "lacunes",
    "alignment", "alignement", "coordination",
    "nexus", "hdp", "humanitarian-development-peace",
    "humanitarian development peace",
    "opportunity", "opportunities",
    "opportunite", "opportunites",
    "mismatch", "tension", "synergy", "synergies"
]


PLANNER_SOURCE_ENUM = [
    "GOVERNMENT_DOCS",
    "HNRP",
    "HAPI",
    "FONGIM"
]


def plan_sources_semantically(question, geography=None):
    """
    Small structured planner.

    Its job is ONLY to identify information needs and source families.
    It must not answer the question, infer facts, or reason about
    geographic relationships.
    """

    started = time.perf_counter()

    system_prompt = """
You are the source planner for the Mali Knowledge Hub.

Your ONLY task is to decide which connected source families must be
queried to answer the user's question.

CONNECTED SOURCE FAMILIES:

GOVERNMENT_DOCS
- Mali government strategy, policy, planning and structural priorities
- Vision Mali 2063
- SNEDD 2024-2033
- Projets Structurants Prioritaires
- Phasage des Projets Structurants Prioritaires

HNRP
- Mali humanitarian needs and response planning document
- humanitarian priorities, response objectives, modalities and planning

HAPI
- current Knowledge Hub connector for structured humanitarian-needs data
- use for quantitative / structured humanitarian needs

FONGIM
- structured operational project data
- organizations / NGOs / actors
- projects and interventions
- sectors
- project locations
- who is working where
- existing activities / operational presence

RULES:
1. Select every source family materially needed to answer the question.
2. Do NOT select a source merely because it might be interesting.
3. A question can require multiple source families.
4. "Who is working", "who is active", actors, NGOs, organizations,
   interventions, projects, activities, operational presence or coverage
   require FONGIM.
5. Humanitarian needs may require HNRP and/or HAPI:
   - HNRP for narrative priorities and response planning.
   - HAPI for structured humanitarian-needs figures.
6. Government priorities, policies, strategies, structural causes or
   planned structural responses require GOVERNMENT_DOCS.
7. Do not claim that a source contains a particular fact.
8. Do not reason about parent/child geography. Geography resolution is
   handled separately by deterministic code.
9. Do not answer the user's substantive question.
10. Return JSON only, matching the requested schema.
"""

    schema = {
        "type": "object",
        "properties": {
            "intents": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "topic": {"type": "string"},
                        "source_families": {
                            "type": "array",
                            "items": {
                                "type": "string",
                                "enum": PLANNER_SOURCE_ENUM
                            }
                        }
                    },
                    "required": [
                        "topic",
                        "source_families"
                    ],
                    "additionalProperties": False
                }
            },
            "geo_entities": {
                "type": "array",
                "items": {"type": "string"}
            },
            "is_meta_question": {
                "type": "boolean"
            }
        },
        "required": [
            "intents",
            "geo_entities",
            "is_meta_question"
        ],
        "additionalProperties": False
    }

    response = openai_client.responses.create(
        model="gpt-5.6-luna",
        reasoning={"effort": "none"},
        max_output_tokens=500,
        input=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": question
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "mali_source_plan",
                "schema": schema,
                "strict": True
            }
        }
    )

    raw = response.output_text
    plan = json.loads(raw)

    selected = set()

    for intent in plan.get("intents", []):
        for source in intent.get("source_families", []):
            if source in PLANNER_SOURCE_ENUM:
                selected.add(source)

    source_plan = {
        "government_docs": "GOVERNMENT_DOCS" in selected,
        "hnrp_docs": "HNRP" in selected,
        "hapi": "HAPI" in selected,
        "fongim": "FONGIM" in selected
    }

    # Safe fallback: never allow a malformed/empty substantive plan
    # to silently suppress all evidence retrieval.
    if (
        not plan.get("is_meta_question", False)
        and not any(source_plan.values())
    ):
        source_plan = {
            "government_docs": True,
            "hnrp_docs": True,
            "hapi": False,
            "fongim": False
        }

    return {
        "source_plan": source_plan,
        "planner_output": plan,
        "seconds": round(
            time.perf_counter() - started,
            3
        )
    }




# ============================================================
# DOCUMENT FAMILY CLASSIFICATION
# ============================================================

def classify_document_family(item):

    title = normalize_text(
        item.get("document_title")
    )

    organization = normalize_text(
        item.get("organization")
    )

    if (
        "besoins humanitaires" in title
        or "plan de reponse" in title
        or "humanitarian" in title
        or "ocha" in organization
    ):

        return (
            "Humanitarian Response Plan / HNRP"
        )

    return "Government strategies"


# ============================================================
# DOCUMENT EVIDENCE
# ============================================================

@st.cache_data(ttl=3600)
def get_document_groups():
    """
    Resolve document families dynamically from the document registry.

    The HNRP is identified from existing document metadata.
    Every other document in the current corpus belongs to the
    government/development family.
    """

    documents = (
        supabase
        .table("documents")
        .select("id,title,document_type,organization")
        .execute()
        .data
        or []
    )

    hnrp_document_ids = []
    government_document_ids = []

    for document in documents:

        document_id = document.get("id")

        if not document_id:
            continue

        document_type = normalize_text(
            document.get("document_type")
        )

        title = normalize_text(
            document.get("title")
        )

        organization = normalize_text(
            document.get("organization")
        )

        is_hnrp = (
            document_type
            == normalize_text(
                "Humanitarian Needs and Response Plan"
            )
            or "besoins humanitaires" in title
            or "plan de reponse" in title
            or "humanitarian needs" in title
            or (
                "humanitarian" in title
                and "ocha" in organization
            )
        )

        if is_hnrp:
            hnrp_document_ids.append(
                str(document_id)
            )
        else:
            government_document_ids.append(
                str(document_id)
            )

    return {
        "government":
            government_document_ids,

        "hnrp":
            hnrp_document_ids
    }


def build_document_evidence(
    question,
    use_government=True,
    use_hnrp=True,
    government_count=8,
    hnrp_count=8
):

    function_started = time.perf_counter()

    document_groups = get_document_groups()

    government_document_ids = document_groups["government"]
    hnrp_document_ids = document_groups["hnrp"]

    trace = {
        "government_docs": {
            "requested": bool(use_government),
            "status": "NOT_REQUESTED",
            "seconds": 0.0,
            "records": 0
        },
        "hnrp_docs": {
            "requested": bool(use_hnrp),
            "status": "NOT_REQUESTED",
            "seconds": 0.0,
            "records": 0
        }
    }

    def retrieve_government():
        if not use_government or not government_document_ids:
            return [], 0.0

        started = time.perf_counter()

        government_query = f"""
{question}

Retrieve evidence specifically about Mali's national development
strategy, long-term policy priorities, government objectives,
Vision Mali 2063, SNEDD 2024-2033, structural projects and
development planning.

Focus on the government/development evidence most relevant to the
user's question.
"""

        results = search_knowledge_base(
            government_query,
            match_count=government_count,
            filter_document_ids=government_document_ids
        )

        return results, time.perf_counter() - started

    def retrieve_hnrp():
        if not use_hnrp or not hnrp_document_ids:
            return [], 0.0

        started = time.perf_counter()

        hnrp_query = f"""
{question}

Retrieve evidence specifically from Mali's humanitarian needs
and response planning documents, including humanitarian needs,
response priorities, humanitarian objectives and HNRP 2026.

Focus on the humanitarian planning evidence most relevant to the
user's question.
"""

        results = search_knowledge_base(
            hnrp_query,
            match_count=hnrp_count,
            filter_document_ids=hnrp_document_ids
        )

        return results, time.perf_counter() - started

    if use_government and use_hnrp:
        with ThreadPoolExecutor(max_workers=2) as executor:
            government_future = executor.submit(
                retrieve_government
            )
            hnrp_future = executor.submit(
                retrieve_hnrp
            )
            government_results, government_seconds = (
                government_future.result()
            )
            hnrp_results, hnrp_seconds = (
                hnrp_future.result()
            )
    else:
        government_results, government_seconds = (
            retrieve_government()
        )
        hnrp_results, hnrp_seconds = (
            retrieve_hnrp()
        )

    if use_government:
        trace["government_docs"].update({
            "status": (
                "SUCCESS_WITH_RESULTS"
                if government_results
                else "SUCCESS_ZERO_RESULTS"
            ),
            "seconds": round(government_seconds, 3),
            "records": len(government_results)
        })

    if use_hnrp:
        trace["hnrp_docs"].update({
            "status": (
                "SUCCESS_WITH_RESULTS"
                if hnrp_results
                else "SUCCESS_ZERO_RESULTS"
            ),
            "seconds": round(hnrp_seconds, 3),
            "records": len(hnrp_results)
        })

    combined = government_results + hnrp_results

    seen = set()
    evidence = []

    for result in combined:
        dedupe_key = (
            result.get("document_id"),
            result.get("page_number"),
            result.get("section_title"),
            result.get("content")
        )

        if dedupe_key in seen:
            continue

        seen.add(dedupe_key)

        evidence.append({
            "source_type": "knowledge_base_document",
            "source_family": classify_document_family(result),
            "document_id": result.get("document_id"),
            "document_title": result.get("document_title"),
            "document_type": result.get("document_type"),
            "organization": result.get("organization"),
            "version": result.get("version"),
            "page": result.get("page_number"),
            "section": result.get("section_title"),
            "similarity": result.get("similarity"),
            "content": result.get("content")
        })

    trace["total_seconds"] = round(
        time.perf_counter() - function_started,
        3
    )

    return {
        "evidence": evidence,
        "trace": trace
    }


# ============================================================
# HAPI REDUCER
# ============================================================

def reduce_hapi_humanitarian_evidence(
    evidence_items,
    max_sector_examples=8
):
    """
    Reduce HAPI humanitarian-needs records while preserving
    geography, denominator and analytical scope.

    Principles:
    1. Use aggregate population_category='total' whenever available.
    2. Never sum Admin2 observations into an Admin1 total.
    3. Prefer true Admin1 observations for region-level analysis.
    4. If only Admin2 observations exist, preserve complete
       intersectoral coverage and use sector figures only as
       locality-specific observations.
    5. Never present the largest Admin2 sector observation as a
       region-wide sector ranking.
    """

    if not evidence_items:
        return []

    # --------------------------------------------------------
    # 1. KEEP AGGREGATE POPULATION RECORDS
    # --------------------------------------------------------

    total_rows = [
        item
        for item in evidence_items
        if normalize_text(
            item.get("population_category")
        ) == "total"
    ]

    # Conservative fallback:
    # do not attempt analytical ranking from demographic slices.
    if not total_rows:
        fallback = []

        for item in evidence_items[:30]:
            fallback.append({
                **item,
                "passage": (
                    str(item.get("passage") or "")
                    + " NOTE: No population_category='total' "
                    "record was available in this result set. "
                    "This record is a population subgroup and "
                    "must not be interpreted as the total number "
                    "of people in need."
                )
            })

        return fallback


    # --------------------------------------------------------
    # 2. IDENTIFY GEOGRAPHIC STRUCTURE
    # --------------------------------------------------------

    admin1_rows = [
        item
        for item in total_rows
        if item.get("admin_level") == 1
    ]

    admin2_rows = [
        item
        for item in total_rows
        if item.get("admin_level") == 2
    ]


    # --------------------------------------------------------
    # 3. IF TRUE ADMIN1 DATA EXIST, USE THEM
    # --------------------------------------------------------

    if admin1_rows:

        intersectoral = [
            item
            for item in admin1_rows
            if normalize_text(
                item.get("sector_name")
            ) == "intersectoral"
        ]

        sector_rows = [
            item
            for item in admin1_rows
            if normalize_text(
                item.get("sector_name")
            ) != "intersectoral"
        ]

        # One aggregate observation per sector.
        # If duplicates exist, keep the largest documented value
        # but preserve the original geography and period.
        best_by_sector = {}

        for item in sector_rows:

            sector = (
                item.get("sector_name")
                or "Unspecified sector"
            )

            current = best_by_sector.get(
                sector
            )

            if (
                current is None
                or (
                    item.get("value") or 0
                ) > (
                    current.get("value") or 0
                )
            ):
                best_by_sector[
                    sector
                ] = item

        ranked_sector_rows = sorted(
            best_by_sector.values(),
            key=lambda x:
                x.get("value") or 0,
            reverse=True
        )

        selected = (
            intersectoral
            + ranked_sector_rows[
                :max_sector_examples
            ]
        )

        result = []

        for item in selected:

            result.append({
                **item,
                "passage": (
                    str(item.get("passage") or "")
                    + " This is an Admin1-level aggregate "
                    "observation for the stated geography and "
                    "reference period."
                )
            })

        return result


    # --------------------------------------------------------
    # 4. OTHERWISE PRESERVE ADMIN2 STRUCTURE
    # --------------------------------------------------------

    working_rows = (
        admin2_rows
        if admin2_rows
        else total_rows
    )

    intersectoral = [
        item
        for item in working_rows
        if normalize_text(
            item.get("sector_name")
        ) == "intersectoral"
    ]

    # Keep all intersectoral locality totals.
    intersectoral = sorted(
        intersectoral,
        key=lambda x: (
            x.get("admin1_name") or "",
            x.get("admin2_name") or ""
        )
    )

    intersectoral_with_scope = []

    for item in intersectoral:

        intersectoral_with_scope.append({
            **item,
            "passage": (
                str(item.get("passage") or "")
                + " This is an Admin2/locality observation. "
                "It must not be summed with other Admin2 "
                "observations to manufacture an Admin1 total."
            )
        })


    # --------------------------------------------------------
    # 5. SECTOR OBSERVATIONS
    # --------------------------------------------------------

    sector_rows = [
        item
        for item in working_rows
        if normalize_text(
            item.get("sector_name")
        ) != "intersectoral"
    ]

    # For each sector, retain the largest documented locality
    # observation purely as an illustrative severity signal.
    # This is NOT a regional sector total.
    best_by_sector = {}

    for item in sector_rows:

        sector = (
            item.get("sector_name")
            or "Unspecified sector"
        )

        current = best_by_sector.get(
            sector
        )

        if (
            current is None
            or (
                item.get("value") or 0
            ) > (
                current.get("value") or 0
            )
        ):
            best_by_sector[
                sector
            ] = item


    sector_examples = sorted(
        best_by_sector.values(),
        key=lambda x:
            x.get("value") or 0,
        reverse=True
    )[:max_sector_examples]


    sector_examples_with_scope = []

    for item in sector_examples:

        sector_examples_with_scope.append({
            **item,
            "passage": (
                str(item.get("passage") or "")
                + " This is the largest recorded Admin2 "
                "observation for this sector within the returned "
                "result set. It is NOT an Admin1 total and must "
                "not be interpreted as a region-wide sector "
                "caseload or as proof that this sector is the "
                "region's most severe need."
            )
        })


    # --------------------------------------------------------
    # 6. COMBINE
    # --------------------------------------------------------

    return (
        intersectoral_with_scope
        + sector_examples_with_scope
    )

@st.cache_data(ttl=900, show_spinner=False)
def build_hapi_evidence(
    geography
):

    raw = research_humanitarian_needs(
        admin1_name=(
            geography.get("region")
            if not geography.get("cercle")
            else None
        ),
        admin2_name=geography.get(
            "cercle"
        ),
        population_status="INN",
        latest_only=True,
        limit=10000
    )

    reduced = (
        reduce_hapi_humanitarian_evidence(
            raw
        )
    )

    evidence = []

    for item in reduced:

        evidence.append({
            **item,

            "source_family":
                "OCHA humanitarian data",

            "content":
                item.get("passage"),

            "document_title":
                item.get(
                    "dataset_title"
                )
                or item.get(
                    "document_title"
                )
                or "HDX HAPI",

            "page":
                None,

            "section":
                item.get(
                    "sector_name"
                ),

            "organization":
                item.get(
                    "provider_name"
                )
                or "OCHA / HDX",

            "version":
                None
        })

    return {
        "raw_count": len(raw),
        "evidence": evidence
    }


# ============================================================
# FONGIM STRUCTURED RESEARCH
# ============================================================

def get_rows_for_project_ids(
    table_name,
    columns,
    project_ids,
    chunk_size=300
):

    if not project_ids:
        return []

    rows = []

    for start in range(
        0,
        len(project_ids),
        chunk_size
    ):

        chunk = project_ids[
            start:start + chunk_size
        ]

        response = (
            supabase
            .table(table_name)
            .select(columns)
            .in_(
                "fongim_project_id",
                chunk
            )
            .eq(
                "is_present_in_source",
                True
            )
            .execute()
        )

        rows.extend(
            response.data or []
        )

    return rows


@st.cache_data(ttl=900, show_spinner=False)
def research_fongim(
    geography
):

    location_filters = {
        "is_present_in_source": True
    }

    if geography.get("region"):
        location_filters["region"] = geography["region"]

    if geography.get("cercle"):
        location_filters["cercle"] = geography["cercle"]

    locations = fetch_all_rows(
        "fongim_project_locations",
        columns=(
            "fongim_project_id,"
            "region,"
            "cercle,"
            "commune_raw,"
            "last_synced_at"
        ),
        eq_filters=location_filters
    )

    project_ids = sorted({
        row.get("fongim_project_id")
        for row in locations
        if row.get("fongim_project_id") is not None
    })

    if not project_ids:
        return {
            "project_count": 0,
            "location_count": 0,
            "evidence": []
        }

    projects = get_rows_for_project_ids(
        "fongim_projects",
        (
            "fongim_project_id,"
            "project_name,"
            "start_date,"
            "end_date,"
            "status,"
            "project_type,"
            "beneficiaries,"
            "donor,"
            "funding_amount_raw,"
            "last_synced_at"
        ),
        project_ids
    )

    sectors = get_rows_for_project_ids(
        "fongim_project_sectors",
        (
            "fongim_project_id,"
            "sector,"
            "sector_other_raw"
        ),
        project_ids
    )

    project_orgs = get_rows_for_project_ids(
        "fongim_project_organizations",
        (
            "fongim_project_id,"
            "fongim_organization_id"
        ),
        project_ids
    )

    organization_ids = sorted({
        row.get("fongim_organization_id")
        for row in project_orgs
        if row.get("fongim_organization_id") is not None
    })

    organizations = []

    if organization_ids:
        response = (
            supabase
            .table("fongim_organizations")
            .select(
                "fongim_organization_id,"
                "organization_name"
            )
            .in_(
                "fongim_organization_id",
                organization_ids
            )
            .eq(
                "is_present_in_source",
                True
            )
            .execute()
        )

        organizations = response.data or []

    org_name_by_id = {
        row.get("fongim_organization_id"):
        row.get("organization_name")
        for row in organizations
    }


    # --------------------------------------------------------
    # PROJECT-LEVEL RELATIONAL MAPS
    # --------------------------------------------------------

    sectors_by_project = defaultdict(set)

    for row in sectors:
        project_id = row.get("fongim_project_id")
        sector = row.get("sector") or "Unspecified"

        if project_id is not None:
            sectors_by_project[project_id].add(sector)

    orgs_by_project = defaultdict(set)

    for row in project_orgs:
        project_id = row.get("fongim_project_id")
        org_id = row.get("fongim_organization_id")

        org_name = (
            org_name_by_id.get(org_id)
            or str(org_id)
        )

        if project_id is not None:
            orgs_by_project[project_id].add(org_name)


    # --------------------------------------------------------
    # SECTOR COUNTS
    # --------------------------------------------------------

    sector_projects = defaultdict(set)

    for project_id, project_sectors in sectors_by_project.items():
        for sector in project_sectors:
            sector_projects[sector].add(project_id)

    top_sectors = sorted(
        (
            (sector, len(ids))
            for sector, ids in sector_projects.items()
        ),
        key=lambda x: x[1],
        reverse=True
    )


    # --------------------------------------------------------
    # CERCLE COUNTS
    # --------------------------------------------------------

    circle_projects = defaultdict(set)

    for row in locations:
        cercle = row.get("cercle")

        if cercle:
            circle_projects[cercle].add(
                row.get("fongim_project_id")
            )

    top_cercles = sorted(
        (
            (cercle, len(ids))
            for cercle, ids in circle_projects.items()
        ),
        key=lambda x: x[1],
        reverse=True
    )


    # --------------------------------------------------------
    # ORGANIZATION COUNTS
    # --------------------------------------------------------

    org_projects = defaultdict(set)

    for project_id, project_org_names in orgs_by_project.items():
        for org_name in project_org_names:
            org_projects[org_name].add(project_id)

    top_orgs = sorted(
        (
            (org, len(ids))
            for org, ids in org_projects.items()
        ),
        key=lambda x: x[1],
        reverse=True
    )


    # --------------------------------------------------------
    # CROSS-DIMENSIONAL ORGANIZATION × SECTOR ANALYSIS
    # --------------------------------------------------------

    org_sector_projects = defaultdict(
        lambda: defaultdict(set)
    )

    sector_org_projects = defaultdict(
        lambda: defaultdict(set)
    )

    for project_id in project_ids:

        project_org_names = orgs_by_project.get(
            project_id,
            set()
        )

        project_sectors = sectors_by_project.get(
            project_id,
            set()
        )

        for org_name in project_org_names:
            for sector in project_sectors:

                org_sector_projects[
                    org_name
                ][
                    sector
                ].add(project_id)

                sector_org_projects[
                    sector
                ][
                    org_name
                ].add(project_id)


    org_sector_breadth = sorted(
        (
            (
                org_name,
                len(sector_map),
                sum(
                    len(ids)
                    for ids in sector_map.values()
                )
            )
            for org_name, sector_map
            in org_sector_projects.items()
        ),
        key=lambda x: (
            -x[1],
            -x[2],
            x[0]
        )
    )


    # --------------------------------------------------------
    # GEOGRAPHIC SCOPE
    # --------------------------------------------------------

    scope_parts = []

    if geography.get("region"):
        scope_parts.append(
            f"region={geography['region']}"
        )

    if geography.get("cercle"):
        scope_parts.append(
            f"cercle={geography['cercle']}"
        )

    geographic_scope = (
        ", ".join(scope_parts)
        if scope_parts
        else "Mali"
    )

    latest_sync = max(
        [
            str(row.get("last_synced_at"))
            for row in projects
            if row.get("last_synced_at")
        ],
        default=None
    )


    # --------------------------------------------------------
    # BUILD STRUCTURED EVIDENCE
    # --------------------------------------------------------

    evidence = []

    evidence.append({
        "source_type": "fongim_structured",
        "source_family": "FONGIM intervention data",
        "document_title": "FONGIM operational project data",
        "document_type": "structured_operational_data",
        "organization": "FONGIM",
        "version": None,
        "page": None,
        "section": geographic_scope,
        "content": (
            f"FONGIM records {len(project_ids)} unique projects "
            f"with at least one recorded location in "
            f"{geographic_scope}, represented by "
            f"{len(locations)} project-location records. "
            f"Latest synchronization timestamp among "
            f"these project records: {latest_sync}. "
            f"These counts describe recorded project presence; "
            f"they do not demonstrate funding adequacy, "
            f"population coverage, implementation quality "
            f"or impact."
        )
    })


    if top_sectors:

        sector_text = "; ".join(
            f"{sector}: {count} projects"
            for sector, count in top_sectors[:10]
        )

        evidence.append({
            "source_type": "fongim_structured",
            "source_family": "FONGIM intervention data",
            "document_title": "FONGIM operational project data",
            "document_type": "structured_operational_data",
            "organization": "FONGIM",
            "version": None,
            "page": None,
            "section": "Sector profile",
            "content": (
                f"Among FONGIM projects with at least one "
                f"recorded location in {geographic_scope}, "
                f"the number of unique projects associated "
                f"with each leading sector is: "
                f"{sector_text}. "
                f"A project may be associated with more than "
                f"one sector, so sector counts must not be summed "
                f"to derive a project total."
            )
        })


    if top_cercles:

        cercle_text = "; ".join(
            f"{cercle}: {count} projects"
            for cercle, count in top_cercles[:10]
        )

        evidence.append({
            "source_type": "fongim_structured",
            "source_family": "FONGIM intervention data",
            "document_title": "FONGIM operational project data",
            "document_type": "structured_operational_data",
            "organization": "FONGIM",
            "version": None,
            "page": None,
            "section": "Recorded geographic presence",
            "content": (
                f"Unique FONGIM projects with a recorded "
                f"location in each cercle within the selected "
                f"scope: {cercle_text}. "
                f"This is a count of recorded project presence, "
                f"not a measure of needs coverage or resources."
            )
        })


    if top_orgs:

        org_text = "; ".join(
            f"{org}: {count} projects"
            for org, count in top_orgs[:10]
        )

        evidence.append({
            "source_type": "fongim_structured",
            "source_family": "FONGIM intervention data",
            "document_title": "FONGIM operational project data",
            "document_type": "structured_operational_data",
            "organization": "FONGIM",
            "version": None,
            "page": None,
            "section": "Organizations",
            "content": (
                f"Primary organizations associated with "
                f"FONGIM projects in {geographic_scope}: "
                f"{org_text}. "
                f"Counts refer to projects linked to each "
                f"primary organization in the source."
            )
        })


    # --------------------------------------------------------
    # ORGANIZATION × SECTOR EVIDENCE
    # --------------------------------------------------------

    if org_sector_breadth:

        org_sector_lines = []

        for org_name, sector_count, _ in org_sector_breadth:

            sector_map = org_sector_projects[
                org_name
            ]

            sector_details = sorted(
                (
                    (
                        sector,
                        len(ids)
                    )
                    for sector, ids
                    in sector_map.items()
                ),
                key=lambda x: (
                    -x[1],
                    x[0]
                )
            )

            sector_text = ", ".join(
                f"{sector} ({count} projects)"
                for sector, count
                in sector_details
            )

            org_sector_lines.append(
                f"{org_name}: "
                f"{sector_count} distinct sectors — "
                f"{sector_text}"
            )

        evidence.append({
            "source_type": "fongim_structured",
            "source_family": "FONGIM intervention data",
            "document_title": "FONGIM operational project data",
            "document_type": "structured_operational_data",
            "organization": "FONGIM",
            "version": None,
            "page": None,
            "section": "Organization-sector relationships",
            "content": (
                f"Project-level organization-sector relationships "
                f"for FONGIM projects in {geographic_scope}. "
                f"Each organization below is linked to the listed "
                f"sector because at least one project associated "
                f"with that organization is also associated with "
                f"that sector. Counts in parentheses are unique "
                f"projects for that organization-sector combination. "
                f"Distinct-sector counts describe recorded thematic "
                f"breadth, not organizational effectiveness or "
                f"quality. "
                + "; ".join(org_sector_lines)
            )
        })


    # --------------------------------------------------------
    # SECTOR × ORGANIZATION EVIDENCE
    # --------------------------------------------------------

    if sector_org_projects:

        sector_org_lines = []

        for sector, org_map in sorted(
            sector_org_projects.items(),
            key=lambda item: (
                -len(sector_projects.get(
                    item[0],
                    set()
                )),
                item[0]
            )
        ):

            org_details = sorted(
                (
                    (
                        org_name,
                        len(ids)
                    )
                    for org_name, ids
                    in org_map.items()
                ),
                key=lambda x: (
                    -x[1],
                    x[0]
                )
            )

            org_text = ", ".join(
                f"{org_name} ({count} projects)"
                for org_name, count
                in org_details
            )

            sector_org_lines.append(
                f"{sector}: {org_text}"
            )

        evidence.append({
            "source_type": "fongim_structured",
            "source_family": "FONGIM intervention data",
            "document_title": "FONGIM operational project data",
            "document_type": "structured_operational_data",
            "organization": "FONGIM",
            "version": None,
            "page": None,
            "section": "Sector-organization relationships",
            "content": (
                f"Project-level sector-organization relationships "
                f"for FONGIM projects in {geographic_scope}. "
                f"Organizations are listed under a sector when "
                f"at least one project associated with that "
                f"organization is also associated with that sector. "
                f"Counts in parentheses are unique projects for "
                f"that sector-organization combination. "
                + "; ".join(sector_org_lines)
            )
        })


    project_examples = sorted(
        [
            p
            for p in projects
            if p.get("project_name")
        ],
        key=lambda x:
            str(x.get("project_name"))
    )[:8]

    if project_examples:

        examples_text = "; ".join(
            (
                f"{p.get('project_name')}"
                + (
                    f" [{p.get('status')}]"
                    if p.get("status")
                    else ""
                )
            )
            for p in project_examples
        )

        evidence.append({
            "source_type": "fongim_structured",
            "source_family": "FONGIM intervention data",
            "document_title": "FONGIM operational project data",
            "document_type": "structured_operational_data",
            "organization": "FONGIM",
            "version": None,
            "page": None,
            "section": "Illustrative project records",
            "content": (
                f"Illustrative project records from the "
                f"selected FONGIM result set: "
                f"{examples_text}. "
                f"These examples are illustrative and are "
                f"not a ranking of projects."
            )
        })


    return {
        "project_count": len(project_ids),
        "location_count": len(locations),
        "evidence": evidence
    }

# ============================================================
# UNIFIED EVIDENCE LEDGER
# ============================================================

def build_unified_evidence(
    document_evidence,
    hapi_evidence,
    fongim_evidence
):

    combined = (
        document_evidence
        + hapi_evidence
        + fongim_evidence
    )

    ledger = []

    for index, item in enumerate(
        combined,
        1
    ):

        ledger.append({
            **item,
            "evidence_id":
                f"E{index:02d}"
        })

    return ledger


# ============================================================
# EVIDENCE SERIALIZATION
# ============================================================

def evidence_to_prompt(
    ledger
):

    blocks = []

    for item in ledger:

        blocks.append(
            f"""
[{item['evidence_id']}]
SOURCE FAMILY: {item.get('source_family')}
SOURCE TYPE: {item.get('source_type')}
SOURCE: {item.get('document_title')}
ORGANIZATION: {item.get('organization')}
TYPE: {item.get('document_type')}
VERSION: {item.get('version')}
PAGE: {item.get('page')}
SECTION: {item.get('section')}

EVIDENCE:
{item.get('content')}

---
"""
        )

    return "\n".join(
        blocks
    )


# ============================================================
# FOUR-SOURCE RESEARCH
# ============================================================

def run_four_source_research(question):

    total_started = time.perf_counter()

    geo_started = time.perf_counter()
    geography = resolve_geography(question)
    geography_seconds = time.perf_counter() - geo_started

    planner_result = plan_sources_semantically(
        question,
        geography
    )

    source_plan = planner_result["source_plan"]
    routing_seconds = planner_result["seconds"]

    document_result = {
        "evidence": [],
        "trace": {
            "government_docs": {
                "requested": bool(source_plan["government_docs"]),
                "status": (
                    "NOT_REQUESTED"
                    if not source_plan["government_docs"]
                    else "SUCCESS_ZERO_RESULTS"
                ),
                "seconds": 0.0,
                "records": 0
            },
            "hnrp_docs": {
                "requested": bool(source_plan["hnrp_docs"]),
                "status": (
                    "NOT_REQUESTED"
                    if not source_plan["hnrp_docs"]
                    else "SUCCESS_ZERO_RESULTS"
                ),
                "seconds": 0.0,
                "records": 0
            },
            "total_seconds": 0.0
        }
    }

    hapi_result = {
        "raw_count": 0,
        "evidence": []
    }

    fongim_result = {
        "project_count": 0,
        "location_count": 0,
        "evidence": []
    }

    source_trace = {
        "hapi": {
            "requested": bool(source_plan["hapi"]),
            "status": (
                "NOT_REQUESTED"
                if not source_plan["hapi"]
                else "SUCCESS_ZERO_RESULTS"
            ),
            "seconds": 0.0,
            "records": 0
        },
        "fongim": {
            "requested": bool(source_plan["fongim"]),
            "status": (
                "NOT_REQUESTED"
                if not source_plan["fongim"]
                else "SUCCESS_ZERO_RESULTS"
            ),
            "seconds": 0.0,
            "records": 0
        }
    }

    jobs = {}

    def timed_hapi():
        started = time.perf_counter()
        value = build_hapi_evidence(geography)
        return value, time.perf_counter() - started

    def timed_fongim():
        started = time.perf_counter()
        value = research_fongim(geography)
        return value, time.perf_counter() - started

    with ThreadPoolExecutor(max_workers=3) as executor:

        if (
            source_plan["government_docs"]
            or source_plan["hnrp_docs"]
        ):
            jobs["documents"] = executor.submit(
                build_document_evidence,
                question,
                source_plan["government_docs"],
                source_plan["hnrp_docs"]
            )

        if source_plan["hapi"]:
            jobs["hapi"] = executor.submit(
                timed_hapi
            )

        if source_plan["fongim"]:
            jobs["fongim"] = executor.submit(
                timed_fongim
            )

        if "documents" in jobs:
            document_result = jobs["documents"].result()

        if "hapi" in jobs:
            hapi_result, hapi_seconds = jobs["hapi"].result()
            source_trace["hapi"].update({
                "status": (
                    "SUCCESS_WITH_RESULTS"
                    if hapi_result.get("evidence")
                    else "SUCCESS_ZERO_RESULTS"
                ),
                "seconds": round(hapi_seconds, 3),
                "records": int(hapi_result.get("raw_count", 0))
            })

        if "fongim" in jobs:
            fongim_result, fongim_seconds = jobs["fongim"].result()
            source_trace["fongim"].update({
                "status": (
                    "SUCCESS_WITH_RESULTS"
                    if fongim_result.get("evidence")
                    else "SUCCESS_ZERO_RESULTS"
                ),
                "seconds": round(fongim_seconds, 3),
                "records": int(fongim_result.get("project_count", 0)),
                "location_records": int(
                    fongim_result.get("location_count", 0)
                )
            })

    reduction_started = time.perf_counter()

    ledger = build_unified_evidence(
        document_result["evidence"],
        hapi_result["evidence"],
        fongim_result["evidence"]
    )

    family_counts = defaultdict(int)

    for item in ledger:
        family_counts[
            item.get("source_family")
        ] += 1

    evidence_reduction_seconds = (
        time.perf_counter() - reduction_started
    )

    execution_trace = {
        "resolved_question": question,
        "geography": {
            "resolved": geography,
            "seconds": round(geography_seconds, 3)
        },
        "routing": {
            "source_plan": source_plan,
            "planner_output": planner_result.get(
                "planner_output",
                {}
            ),
            "seconds": routing_seconds
        },
        "sources": {
            "government_docs": document_result["trace"][
                "government_docs"
            ],
            "hnrp_docs": document_result["trace"][
                "hnrp_docs"
            ],
            "hapi": source_trace["hapi"],
            "fongim": source_trace["fongim"]
        },
        "document_bundle_seconds": document_result["trace"].get(
            "total_seconds",
            0.0
        ),
        "evidence_reduction_seconds": round(
            evidence_reduction_seconds,
            3
        )
    }

    return {
        "geography": geography,
        "source_plan": source_plan,
        "ledger": ledger,
        "family_counts": dict(family_counts),
        "hapi_raw_count": hapi_result["raw_count"],
        "fongim_project_count": fongim_result["project_count"],
        "research_seconds": round(
            time.perf_counter() - total_started,
            2
        ),
        "execution_trace": execution_trace
    }


# ============================================================
# ANSWER ENGINE
# ============================================================

def generate_grounded_answer(
    question,
    model="gpt-5.6-luna"
):

    answer_started = time.perf_counter()

    research = (
        run_four_source_research(
            question
        )
    )

    ledger = research[
        "ledger"
    ]

    evidence_text = (
        evidence_to_prompt(
            ledger
        )
    )

    system_prompt = """
You are the analytical synthesis layer of the Mali Knowledge Hub.

Answer the user's question exclusively from the supplied evidence
ledger. Never use outside knowledge.

CORE RULE:
Reason across evidence. Do not reason beyond evidence.

EPISTEMIC RULES:

1. Every substantive factual claim must be supported by one or more
   exact evidence IDs, e.g. [E03] or [E03, E07].

2. Preserve source attribution, reference periods and geographic
   scope.

3. Never turn broader geographic evidence into a narrower geographic
   claim. If evidence is for Mopti region and the question concerns
   Bandiagara, say explicitly that it is broader regional context.

4. Never invent facts, policies, projects, interventions, causal
   relationships, geographic aggregates, totals, rankings, coverage,
   implementation status, impact or citations.

5. Government strategy documents establish stated priorities,
   objectives, diagnoses, targets or scenarios. They do not by
   themselves demonstrate implementation or impact.

6. Humanitarian planning documents and structured humanitarian data
   must not be treated as equivalent evidence when their reference
   periods, definitions or geographic levels differ.

7. Never sum Admin2 observations to manufacture an Admin1 total unless
   the evidence explicitly provides such an aggregate.

8. FONGIM project counts describe recorded project presence. They do
   not establish funding adequacy, needs coverage, service quality,
   effectiveness or impact.

9. A FONGIM project may have multiple sectors and locations. Never sum
   sector or location counts to reconstruct the number of projects.

10. Do not infer a programming gap merely because one FONGIM sector
    has fewer recorded projects than another.

11. If a relationship across humanitarian needs, government
    priorities and operational interventions is only thematic, say
    that it is thematic rather than causal.

12. You may connect concepts across sources only when those concepts
    themselves are documented in the evidence.

13. If the evidence cannot answer part of the question, state briefly
    and precisely what cannot be established.

14. Use the language of the user's question.

DEFAULT RESPONSE:
Write for a busy policy or operational adviser. Be concise,
analytical and decision-useful. Do not reproduce the evidence ledger.

Use this structure unless the question clearly requires another form:

**Bottom line**
Answer the actual question directly in 2-4 sentences.

**What the evidence shows**
Give 3-5 concise bullets with the most decision-relevant findings.
Combine related evidence instead of repeating it.

**Important limitations**
Give only 1-3 limitations that materially affect interpretation or
action. Omit this section if there are no material limitations.

LENGTH:
- Default target: 350-550 words maximum.
- Simple single-source questions: usually 150-300 words.
- Do not add separate sections called "Source facts", "Analytical
  synthesis", "Cautious inference", "Evidence limitations" or
  "Next steps" unless the user explicitly asks for that detail.
- Do not repeat the same evidence in multiple sections.
- Do not offer additional work at the end unless necessary to answer
  the question.

Citations belong directly after the claims they support. Do not add a
generic bibliography.
"""

    user_prompt = f"""
QUESTION

{question}


EVIDENCE LEDGER

{evidence_text}


Produce a concise evidence-grounded analytical answer.
"""

    synthesis_started = time.perf_counter()

    response = (
        openai_client
        .responses
        .create(
            model=model,
            reasoning={"effort": "none"},
            max_output_tokens=1800,
            instructions=system_prompt,
            input=user_prompt
        )
    )

    return {
        "answer":
            response.output_text,

        "evidence":
            ledger,

        "geography":
            research[
                "geography"
            ],

        "source_plan":
            research[
                "source_plan"
            ],

        "family_counts":
            research[
                "family_counts"
            ],

        "hapi_raw_count":
            research[
                "hapi_raw_count"
            ],

        "fongim_project_count":
            research[
                "fongim_project_count"
            ],

        "research_seconds":
            research.get(
                "research_seconds"
            ),

        "synthesis_seconds":
            round(
                time.perf_counter()
                - synthesis_started,
                2
            ),

        "execution_trace":
            research.get(
                "execution_trace",
                {}
            ),

        "total_seconds":
            round(
                time.perf_counter()
                - answer_started,
                2
            )
    }


# ============================================================
# CONVERSATIONAL CONTEXT
# ============================================================

def likely_context_dependent_followup(
    question,
    messages
):

    if not messages:
        return False

    q = normalize_text(question).strip()

    if not q:
        return False

    markers = [
        "what about ",
        "how about ",
        "and ",
        "but ",
        "compare that",
        "compare this",
        "compare it",
        "same for ",
        "same question",
        "that ",
        "this ",
        "those ",
        "these ",
        "them ",
        "it ",
        "why is that",
        "why does that",
        "what does that",
        "and there",
        "and in "
    ]

    if any(
        q.startswith(marker)
        for marker in markers
    ):
        return True

    return len(q.split()) <= 5


def resolve_conversational_question(
    question,
    messages,
    model="gpt-5-mini"
):
    """Convert a follow-up into a standalone research question."""

    prior_user_turns = []

    for message in messages:
        if message.get("role") != "user":
            continue

        standalone = (
            message.get("standalone_question")
            or message.get("content")
        )

        if standalone:
            prior_user_turns.append(standalone)

    if not prior_user_turns:
        return question.strip()

    context_text = "\n".join(
        f"{i + 1}. {item}"
        for i, item in enumerate(prior_user_turns[-6:])
    )

    instructions = """
You rewrite conversational follow-up questions for an evidence-grounded
research system.

Use the previous USER questions only to resolve conversational references.

Return ONE standalone research question that preserves the user's intent.

Rules:
1. Do not answer the question.
2. Do not add facts, assumptions, entities or claims except what is
   strictly necessary to resolve the reference.
3. Do not use outside knowledge.
4. Previous assistant answers are not evidence and must not be used.
5. If the new question is already standalone, return it essentially unchanged.
6. Use the language of the new question.
7. Return only the rewritten question.
"""

    input_text = f"""
PREVIOUS USER QUESTIONS

{context_text}

NEW USER QUESTION

{question}
"""

    response = (
        openai_client
        .responses
        .create(
            model=model,
            instructions=instructions,
            input=input_text
        )
    )

    return (
        response.output_text
        or question
    ).strip()


# ============================================================
# USER INTERFACE
# ============================================================

st.markdown(
    """
<style>
    .block-container {
        max-width: 1180px;
        padding-top: 3.1rem;
        padding-bottom: 7rem;
    }

    .kh-hero {
        position: relative;
        overflow: hidden;
        border: 1px solid #dce9f6;
        border-radius: 22px;
        padding: 2.45rem 2.6rem 2.15rem 2.6rem;
        margin-bottom: 1.05rem;
        background:
            radial-gradient(circle at 88% 18%, rgba(74, 163, 232, 0.18), transparent 27%),
            linear-gradient(135deg, #f8fbff 0%, #ffffff 58%, #eef7ff 100%);
        box-shadow: 0 8px 30px rgba(20, 63, 103, 0.06);
    }

    .kh-hero::after {
        content: "";
        position: absolute;
        width: 290px;
        height: 290px;
        border-radius: 50%;
        right: -95px;
        top: -125px;
        background: rgba(42, 137, 216, 0.055);
        pointer-events: none;
    }

    .kh-title {
        position: relative;
        z-index: 1;
        font-size: 3.28rem;
        line-height: 1.02;
        font-weight: 800;
        letter-spacing: -0.04em;
        margin: 0 0 0.58rem 0;
        color: #0b2b55;
    }

    .kh-subtitle {
        position: relative;
        z-index: 1;
        font-size: 1.12rem;
        line-height: 1.55;
        color: #4b637b;
        margin: 0;
        max-width: 880px;
    }

    .kh-search-label {
        font-size: 0.93rem;
        font-weight: 780;
        color: #284662;
        margin: 0.65rem 0 0.38rem 0;
    }

    .kh-examples-label {
        font-size: 0.92rem;
        font-weight: 760;
        color: #334a63;
        margin-top: 1.15rem;
        margin-bottom: 0.35rem;
    }

    .kh-example-card {
        border: 1px solid #d9e6f2;
        border-radius: 15px;
        padding: 1rem 1rem 0.95rem 1rem;
        min-height: 132px;
        background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%);
        box-shadow: 0 4px 16px rgba(26, 79, 126, 0.045);
        margin-bottom: 0.45rem;
    }

    .kh-example-kicker {
        font-size: 0.8rem;
        font-weight: 850;
        letter-spacing: 0.04em;
        color: #237dc4;
        margin-bottom: 0.48rem;
    }

    .kh-example-text {
        font-size: 0.96rem;
        line-height: 1.42;
        color: #17334f;
        min-height: 66px;
    }

    .kh-how-copy {
        color: #6b7b8d;
        font-size: 0.92rem;
        margin-top: -0.1rem;
        margin-bottom: 0.55rem;
    }

    .kh-section-intro {
        margin: 1.45rem 0 0.45rem 0;
        color: #466078;
        font-size: 0.9rem;
        font-weight: 680;
    }

    div[data-testid="stTextArea"] textarea {
        min-height: 92px !important;
        border-radius: 16px !important;
        border: 1.5px solid #bddaf1 !important;
        background: #ffffff !important;
        font-size: 1.05rem !important;
        line-height: 1.45 !important;
        padding: 1rem 1.05rem !important;
        box-shadow: 0 4px 18px rgba(36, 137, 216, 0.06);
    }

    div[data-testid="stTextArea"] textarea:focus {
        border-color: #2489d8 !important;
        box-shadow: 0 0 0 3px rgba(36, 137, 216, 0.10) !important;
    }

    div[data-testid="stExpander"] {
        border: 1px solid #cfdfef !important;
        border-radius: 16px !important;
        background: linear-gradient(180deg, #ffffff 0%, #fbfdff 100%) !important;
        box-shadow: 0 5px 18px rgba(20, 63, 103, 0.045);
        margin-bottom: 0.8rem;
        overflow: hidden;
    }

    div[data-testid="stExpander"] details > summary {
        min-height: 68px;
        padding: 0.75rem 1rem !important;
    }

    div[data-testid="stExpander"] details > summary p {
        font-size: 1.08rem !important;
        font-weight: 780 !important;
        color: #12365d !important;
    }

    div[data-testid="stChatMessage"] {
        border-radius: 16px;
        padding: 0.15rem 0.25rem;
    }

    div[data-testid="stChatInput"] textarea {
        font-size: 1rem;
    }

    button[kind="primary"] {
        background: #2489d8 !important;
        border-color: #2489d8 !important;
        color: white !important;
        border-radius: 12px;
        font-weight: 700;
        min-height: 46px;
    }

    button[kind="primary"]:hover {
        background: #1678c6 !important;
        border-color: #1678c6 !important;
    }

    .kh-topnav {
        margin: 0 0 1.15rem 0;
        padding: 0.4rem;
        border: 1px solid #dce9f6;
        border-radius: 15px;
        background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%);
        box-shadow: 0 4px 16px rgba(26, 79, 126, 0.045);
    }

    [data-testid="stChatInput"] {
        max-width: 1180px;
        margin-left: auto;
        margin-right: auto;
    }

    [data-testid="stChatInput"] > div {
        border: 1.5px solid #8fc4ec !important;
        border-radius: 16px !important;
        background: #ffffff !important;
        box-shadow: 0 5px 18px rgba(22, 120, 198, 0.11) !important;
    }

    [data-testid="stChatInput"] > div:focus-within {
        border-color: #1678c6 !important;
        box-shadow: 0 0 0 3px rgba(22,120,198,.10), 0 6px 20px rgba(22,120,198,.13) !important;
    }

    @media (max-width: 760px) {
        .block-container {
            padding-top: 1.6rem;
        }

        .kh-hero {
            padding: 1.55rem 1.3rem 1.4rem 1.3rem;
        }

        .kh-title {
            font-size: 2.35rem;
        }

        .kh-subtitle {
            font-size: 1rem;
        }
    }
</style>

<div class="kh-hero">
    <div class="kh-title">Mali Knowledge Hub</div>
    <div class="kh-subtitle">
        AI-powered knowledge and analysis platform for Humanitarian,
        Development and Peace stakeholders in Mali
    </div>
</div>
""",
    unsafe_allow_html=True
)


if "kh_messages" not in st.session_state:
    st.session_state["kh_messages"] = []

if "kh_page" not in st.session_state:
    st.session_state["kh_page"] = "Knowledge Hub"


def reset_conversation():
    st.session_state["kh_messages"] = []


def render_execution_trace(result):

    trace = result.get("execution_trace") or {}

    if not trace:
        return

    with st.expander("🧪 Retrieval trace"):

        geography_trace = trace.get("geography", {})
        routing_trace = trace.get("routing", {})
        sources = trace.get("sources", {})

        st.markdown("**Resolved question**")
        st.code(
            trace.get("resolved_question", ""),
            language=None
        )

        resolved_geo = geography_trace.get("resolved", {})

        st.markdown("**Resolved geography**")
        st.json(resolved_geo)

        st.markdown("**Source plan**")
        st.json(
            routing_trace.get("source_plan", {})
        )

        planner_output = routing_trace.get(
            "planner_output",
            {}
        )

        if planner_output:
            st.markdown("**Structured planner output**")
            st.json(planner_output)

        st.markdown("**Source execution**")

        source_rows = []

        labels = {
            "government_docs": "Government documents",
            "hnrp_docs": "HNRP",
            "hapi": "OCHA / HAPI",
            "fongim": "FONGIM"
        }

        for key in [
            "government_docs",
            "hnrp_docs",
            "hapi",
            "fongim"
        ]:
            item = sources.get(key, {})

            source_rows.append({
                "Source": labels[key],
                "Status": item.get(
                    "status",
                    "UNKNOWN"
                ),
                "Records": item.get(
                    "records",
                    0
                ),
                "Seconds": item.get(
                    "seconds",
                    0.0
                )
            })

        st.dataframe(
            source_rows,
            use_container_width=True,
            hide_index=True
        )

        st.markdown("**Pipeline timing**")

        total_seconds = result.get("total_seconds", 0.0)
        research_seconds = result.get("research_seconds", 0.0)
        synthesis_seconds = result.get("synthesis_seconds", 0.0)

        timing_rows = [
            {
                "Stage": "Geography resolution",
                "Seconds": geography_trace.get(
                    "seconds",
                    0.0
                )
            },
            {
                "Stage": "Source routing",
                "Seconds": routing_trace.get(
                    "seconds",
                    0.0
                )
            },
            {
                "Stage": "Government retrieval",
                "Seconds": sources.get(
                    "government_docs",
                    {}
                ).get(
                    "seconds",
                    0.0
                )
            },
            {
                "Stage": "HNRP retrieval",
                "Seconds": sources.get(
                    "hnrp_docs",
                    {}
                ).get(
                    "seconds",
                    0.0
                )
            },
            {
                "Stage": "HAPI",
                "Seconds": sources.get(
                    "hapi",
                    {}
                ).get(
                    "seconds",
                    0.0
                )
            },
            {
                "Stage": "FONGIM",
                "Seconds": sources.get(
                    "fongim",
                    {}
                ).get(
                    "seconds",
                    0.0
                )
            },
            {
                "Stage": "Evidence normalization",
                "Seconds": trace.get(
                    "evidence_reduction_seconds",
                    0.0
                )
            },
            {
                "Stage": "Total research wall time",
                "Seconds": research_seconds
            },
            {
                "Stage": "Final synthesis",
                "Seconds": synthesis_seconds
            },
            {
                "Stage": "Total response",
                "Seconds": total_seconds
            }
        ]

        st.dataframe(
            timing_rows,
            use_container_width=True,
            hide_index=True
        )

        st.caption(
            "Source calls execute in parallel where possible, so "
            "individual source times do not sum to research wall time."
        )


def render_source_summary(result):

    family_counts = result.get("family_counts", {})

    source_labels = [
        "Government strategies",
        "Humanitarian Response Plan / HNRP",
        "OCHA humanitarian data",
        "FONGIM intervention data"
    ]

    used = []

    for family in source_labels:
        count = family_counts.get(family, 0)

        if count:
            used.append(f"**{family}** · {count}")

    if used:
        st.markdown(" · ".join(used))

    if result.get("hapi_raw_count"):
        st.caption(
            "OCHA: "
            f"{result['hapi_raw_count']} source records reduced to the "
            "relevant evidence set while preserving the documented "
            "geographic structure."
        )

    if result.get("fongim_project_count"):
        st.caption(
            "FONGIM: "
            f"{result['fongim_project_count']} unique projects matched "
            "the selected geographic scope. Project presence does not "
            "by itself establish coverage or impact."
        )


def render_evidence_inspector(result):

    with st.expander("Inspect evidence"):

        for item in result.get("evidence", []):

            page = (
                f" · p. {item['page']}"
                if item.get("page") is not None
                else ""
            )

            label = (
                f"{item['evidence_id']} — "
                f"{item.get('source_family')}"
                f"{page}"
            )

            with st.expander(label):

                if item.get("document_title"):
                    st.markdown(
                        "**Source:** "
                        f"{item['document_title']}"
                    )

                if item.get("organization"):
                    st.markdown(
                        "**Organization:** "
                        f"{item['organization']}"
                    )

                if item.get("section"):
                    st.markdown(
                        "**Section / dimension:** "
                        f"{item['section']}"
                    )

                if item.get("version"):
                    st.markdown(
                        "**Version:** "
                        f"{item['version']}"
                    )

                st.markdown("**Evidence:**")
                st.write(item.get("content"))


st.markdown('<div class="kh-topnav">', unsafe_allow_html=True)
nav1, nav2, nav3 = st.columns([1.05, 1, 1.15], gap="small")

with nav1:
    if st.button("Knowledge Hub", type="primary" if st.session_state["kh_page"] == "Knowledge Hub" else "secondary", use_container_width=True, key="nav_hub"):
        st.session_state["kh_page"] = "Knowledge Hub"
        st.rerun()

with nav2:
    if st.button("How it works", type="primary" if st.session_state["kh_page"] == "How it works" else "secondary", use_container_width=True, key="nav_how"):
        st.session_state["kh_page"] = "How it works"
        st.rerun()

with nav3:
    if st.button("Available sources", type="primary" if st.session_state["kh_page"] == "Available sources" else "secondary", use_container_width=True, key="nav_sources"):
        st.session_state["kh_page"] = "Available sources"
        st.rerun()

st.markdown('</div>', unsafe_allow_html=True)

if st.session_state["kh_page"] == "How it works":
    with st.container(border=False):

        st.markdown(
            '<div class="kh-how-copy">From your question to an evidence-based answer — in 6 steps</div>',
            unsafe_allow_html=True
        )

        st.markdown(
            """
    <div style="
        border:1px solid #d9e6f2;
        border-radius:14px;
        padding:0.7rem;
        background:#fbfdff;
        margin:0.4rem 0 1.1rem 0;
    ">
        <img
            src="..."
            style="width:100%; height:auto; display:block; border-radius:8px;"
        />
    </div>
    """,
            unsafe_allow_html=True
        )

        step1, step2, step3 = st.columns(3, gap="medium")

        with step1:
            st.markdown(
                """
    **1 · Understand**
    The Hub identifies the geography, topic and types of evidence needed.

    **2 · Retrieve**
    It searches the document corpus and, where relevant, queries live OCHA data and structured FONGIM records.
    """
            )

        with step2:
            st.markdown(
                """
    **3 · Select**
    Only relevant passages and structured records are passed into the analysis. The AI does not answer from general knowledge.

    **4 · Preserve**
    Geographic levels, reporting periods, source categories and source limitations are retained instead of being silently merged.
    """
            )

        with step3:
            st.markdown(
                """
    **5 · Analyse**
    The AI compares evidence across sources to identify documented alignments, gaps and tensions.

    **6 · Respond**
    The answer is generated with evidence references, and the underlying evidence items remain inspectable.
    """
            )

        st.info(
            "The AI interprets retrieved evidence. It is not itself the source of truth."
        )

        st.caption(
            "Core rule: Reason across evidence. Do not reason beyond evidence."
        )

elif st.session_state["kh_page"] == "Available sources":
    with st.container(border=False):

        st.caption(
            "Explore the documents and structured data currently integrated in the Knowledge Hub."
        )

        source_col1, source_col2 = st.columns(2, gap="large")

        with source_col1:

            st.markdown(
                """
    <div class="kh-example-card">
        <div class="kh-example-kicker">🏛️ GOVERNMENT FRAMEWORK DOCUMENTS</div>
        <div class="kh-example-text">
            National, regional, local and sectoral strategies and plans
        </div>
    </div>
    """,
                unsafe_allow_html=True
            )

            with st.popover(
                "View available documents",
                use_container_width=True
            ):
                st.markdown(
                    """
    - **Vision Mali 2063 — Mali Kura Ɲɛtaasira ka bɛn san 2063 ma**
    - **Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024–2033)**
    - **Projets Structurants Prioritaires pour la mise en œuvre de Mali Kura 2063 et de la SNEDD 2024–2033**
    - **Phasage des Projets Structurants Prioritaires**
    """
                )

            st.markdown(
                """
    <div class="kh-example-card">
        <div class="kh-example-kicker">👥 HUMANITARIAN NEEDS ASSESSMENT</div>
        <div class="kh-example-text">
            Humanitarian needs and response planning documents
        </div>
    </div>
    """,
                unsafe_allow_html=True
            )

            with st.popover(
                "View available documents",
                use_container_width=True
            ):
                st.markdown(
                    """
    - **Mali — Besoins humanitaires et Plan de Réponse 2026**
    """
                )

        with source_col2:

            st.markdown(
                """
    <div class="kh-example-card">
        <div class="kh-example-kicker">🗄️ OCHA DATABASE</div>
        <div class="kh-example-text">
            Live structured humanitarian needs data via HDX HAPI
        </div>
    </div>
    """,
                unsafe_allow_html=True
            )

            st.link_button(
                "Go to HDX HAPI ↗",
                "https://hapi.humdata.org/",
                use_container_width=True
            )

            st.markdown(
                """
    <div class="kh-example-card">
        <div class="kh-example-kicker">🌐 INTERNATIONAL NGO ACTIVITIES</div>
        <div class="kh-example-text">
            Structured project information through the FONGIM database
        </div>
    </div>
    """,
                unsafe_allow_html=True
            )

            st.link_button(
                "Open FONGIM interactive map ↗",
                "https://www.fongim.org/carte-interactive",
                use_container_width=True
            )

else:
    control_left, control_right = st.columns([5, 1])
    with control_right:
        if st.button("↻ New chat", use_container_width=True):
            reset_conversation()
            st.rerun()

    example_prompt = None
    typed_prompt = None

    if not st.session_state["kh_messages"]:

        st.markdown(
            '<div class="kh-search-label">Ask the Knowledge Hub</div>',
            unsafe_allow_html=True
        )

        landing_question = st.text_area(
            "Question",
            key="kh_landing_question",
            placeholder=(
                "Ask a question about Mali — search, compare evidence, "
                "or analyse priorities, needs and interventions..."
            ),
            label_visibility="collapsed"
        )

        if st.button(
            "Analyse evidence  →",
            type="primary",
            use_container_width=True,
            key="kh_landing_submit"
        ):
            typed_prompt = landing_question

        st.markdown(
            '<div class="kh-examples-label">Or try one of these example questions:</div>',
            unsafe_allow_html=True
        )

        example_col1, example_col2, example_col3 = st.columns(
            3,
            gap="medium"
        )

        with example_col1:
            st.markdown(
                """
    <div class="kh-example-card">
        <div class="kh-example-kicker">🔎 SEARCH</div>
        <div class="kh-example-text">
            What are the Government's priorities for local development
            in Kayes?
        </div>
    </div>
    """,
                unsafe_allow_html=True
            )

            if st.button(
                "Ask this →",
                key="example_search",
                use_container_width=True
            ):
                example_prompt = (
                    "What are the Government's priorities for local "
                    "development in Kayes?"
                )

        with example_col2:
            st.markdown(
                """
    <div class="kh-example-card">
        <div class="kh-example-kicker">↔ COMPARE</div>
        <div class="kh-example-text">
            In Mopti, how do humanitarian needs compare with current
            NGO interventions?
        </div>
    </div>
    """,
                unsafe_allow_html=True
            )

            if st.button(
                "Ask this →",
                key="example_compare",
                use_container_width=True
            ):
                example_prompt = (
                    "In Mopti, how do humanitarian needs compare with "
                    "current NGO interventions?"
                )

        with example_col3:
            st.markdown(
                """
    <div class="kh-example-card">
        <div class="kh-example-kicker">💡 ANALYSE &amp; PLAN</div>
        <div class="kh-example-text">
            Where are the main gaps and opportunities for stronger
            Humanitarian-Development-Peace coordination in Gao?
        </div>
    </div>
    """,
                unsafe_allow_html=True
            )

            if st.button(
                "Ask this →",
                key="example_plan",
                use_container_width=True
            ):
                example_prompt = (
                    "Where are the main gaps and opportunities for stronger "
                    "Humanitarian-Development-Peace coordination in Gao?"
                )
    else:

        for message in st.session_state["kh_messages"]:

            if message.get("role") == "user":

                with st.chat_message("user"):
                    st.markdown(
                        message.get("content", "")
                    )

            elif message.get("role") == "assistant":

                with st.chat_message("assistant"):

                    st.markdown(
                        message.get("content", "")
                    )

                    result = message.get("result")

                    if result:

                        geography = result.get("geography", {})

                        if geography.get("assumption"):
                            st.info(
                                geography["assumption"]
                            )

                        with st.expander("Sources used"):
                            render_source_summary(result)

                            if result.get("total_seconds") is not None:
                                st.caption(
                                    "Response time: "
                                    f"{result['total_seconds']:.1f}s "
                                    "· source research: "
                                    f"{result.get('research_seconds', 0):.1f}s"
                                )

                        render_execution_trace(result)
                        render_evidence_inspector(result)

        typed_prompt = st.chat_input(
            "Ask a follow-up question..."
        )

    current_prompt = typed_prompt or example_prompt


    if current_prompt:

        current_prompt = current_prompt.strip()

        if current_prompt:

            prior_messages = list(
                st.session_state[
                    "kh_messages"
                ]
            )

            st.session_state[
                "kh_messages"
            ].append(
                {
                    "role": "user",
                    "content": current_prompt,
                    "standalone_question": current_prompt
                }
            )

            if is_source_inventory_question(
                current_prompt
            ):

                st.session_state[
                    "kh_messages"
                ].append(
                    {
                        "role": "assistant",
                        "content": source_inventory_answer(),
                        "result": None
                    }
                )

            else:

                standalone_question = current_prompt

                if likely_context_dependent_followup(
                    current_prompt,
                    prior_messages
                ):
                    standalone_question = (
                        resolve_conversational_question(
                            current_prompt,
                            prior_messages
                        )
                    )

                st.session_state[
                    "kh_messages"
                ][-1][
                    "standalone_question"
                ] = standalone_question

                with st.chat_message("user"):
                    st.markdown(current_prompt)

                with st.chat_message("assistant"):
                    with st.status(
                        "Searching and analysing evidence...",
                        expanded=True
                    ) as search_status:

                        st.write("Planning research and selecting sources...")
                        st.write("Retrieving documents and structured data...")
                        st.write("Cross-checking evidence and preparing the answer...")

                        try:

                            result = generate_grounded_answer(
                                standalone_question
                            )

                            st.session_state[
                                "kh_messages"
                            ].append(
                                {
                                    "role": "assistant",
                                    "content": result["answer"],
                                    "result": result
                                }
                            )

                            search_status.update(
                                label="Analysis complete",
                                state="complete",
                                expanded=False
                            )

                        except Exception as exc:

                            search_status.update(
                                label="Analysis failed",
                                state="error",
                                expanded=True
                            )

                            st.session_state[
                                "kh_messages"
                            ].append(
                                {
                                    "role": "assistant",
                                    "content": (
                                        "I could not complete this analysis. "
                                        f"Knowledge Hub error: {exc}"
                                    ),
                                    "result": None
                                }
                            )

            st.rerun()'''

# ============================================================
# WRITE APPLICATION
# ============================================================

with open(
    "/content/app.py",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        app_code
    )


# ============================================================
# SYNTAX VALIDATION
# ============================================================

compile(
    app_code,
    "/content/app.py",
    "exec"
)


print(
    "✅ Four-source Streamlit application created"
)

print(
    "✅ Python syntax validation passed"
)

print(
    "📄 /content/app.py"
)

✅ Four-source Streamlit application created
✅ Python syntax validation passed
📄 /content/app.py


#### 🚀 07.1a — CREATE DEPLOYMENT REQUIREMENTS

In [ ]:
# 07.1a — CREATE DEPLOYMENT REQUIREMENTS

requirements = """streamlit
supabase
openai
"""

with open("/content/requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("✅ requirements.txt created")
print("📄 /content/requirements.txt")

✅ requirements.txt created
📄 /content/requirements.txt


In [ ]:
from google.colab import files

files.download("/content/app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 07.1b — PACKAGE DEPLOYMENT FILES

from pathlib import Path
import zipfile

deploy_dir = Path("/content/mali-knowledge-hub")
deploy_dir.mkdir(exist_ok=True)

# Copy app.py
app_source = Path("/content/app.py")
app_target = deploy_dir / "app.py"
app_target.write_text(
    app_source.read_text(encoding="utf-8"),
    encoding="utf-8"
)

# Copy requirements.txt
requirements_source = Path("/content/requirements.txt")
requirements_target = deploy_dir / "requirements.txt"
requirements_target.write_text(
    requirements_source.read_text(encoding="utf-8"),
    encoding="utf-8"
)

# Create ZIP
zip_path = Path("/content/mali-knowledge-hub.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(app_target, arcname="app.py")
    z.write(requirements_target, arcname="requirements.txt")

print("✅ Deployment package created")
print("📁 /content/mali-knowledge-hub/")
print("📦 /content/mali-knowledge-hub.zip")

✅ Deployment package created
📁 /content/mali-knowledge-hub/
📦 /content/mali-knowledge-hub.zip


### 🚀 07.2 — Launch / Restart Application

Starts the Streamlit app and exposes it through LocalTunnel.

Important:
- Colab runtimes are temporary.
- The public URL changes after a restart.
- This cell is intentionally kept running while the tunnel is active.
- Your corpus remains safe in Supabase even if the tunnel stops.

In [ ]:
# ============================================================
# 07.2 — LAUNCH / RESTART APPLICATION
# Streamlit + Cloudflare Tunnel
# ============================================================

import subprocess
import time
import os
import re

# ------------------------------------------------------------
# 1. Pass credentials to Streamlit
# ------------------------------------------------------------

os.environ["SUPABASE_URL"] = SUPABASE_URL
os.environ["SUPABASE_SECRET_KEY"] = SUPABASE_SECRET_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


# ------------------------------------------------------------
# 2. Stop old processes
# ------------------------------------------------------------

subprocess.run(
    "pkill -f 'streamlit run' || true",
    shell=True
)

subprocess.run(
    "pkill -f 'cloudflared tunnel' || true",
    shell=True
)

subprocess.run(
    "pkill -f 'localtunnel' || true",
    shell=True
)

time.sleep(2)


# ------------------------------------------------------------
# 3. Check app.py
# ------------------------------------------------------------

app_path = "/content/app.py"

if not os.path.exists(app_path):
    raise FileNotFoundError(
        "❌ /content/app.py not found. Run Section 07.1 first."
    )

print("✅ app.py found")


# ------------------------------------------------------------
# 4. Start Streamlit
# ------------------------------------------------------------

log_path = "/content/streamlit.log"

streamlit_log = open(
    log_path,
    "w",
    encoding="utf-8"
)

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        app_path,
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

print("🚀 Starting Streamlit...")

time.sleep(7)


# ------------------------------------------------------------
# 5. Health check
# ------------------------------------------------------------

health = subprocess.run(
    "curl -s -o /dev/null -w '%{http_code}' "
    "http://localhost:8501",
    shell=True,
    capture_output=True,
    text=True
)

if health.stdout != "200":

    print("❌ Streamlit failed to start")
    subprocess.run(
        f"tail -40 {log_path}",
        shell=True
    )

    raise RuntimeError(
        "Streamlit health check failed."
    )

print("✅ Streamlit is running on port 8501")


# ------------------------------------------------------------
# 6. Install Cloudflare Tunnel
# ------------------------------------------------------------

if not os.path.exists("/usr/local/bin/cloudflared"):

    print("📦 Installing Cloudflare Tunnel...")

    !wget -q \
      https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
      -O /usr/local/bin/cloudflared

    !chmod +x /usr/local/bin/cloudflared

    print("✅ Cloudflare Tunnel installed")


# ------------------------------------------------------------
# 7. Launch public tunnel
# ------------------------------------------------------------

print("\n🌍 Starting Cloudflare Tunnel...")
print("Waiting for public URL...\n")

!cloudflared tunnel --url http://localhost:8501 --no-autoupdate

✅ app.py found
🚀 Starting Streamlit...
✅ Streamlit is running on port 8501
📦 Installing Cloudflare Tunnel...
✅ Cloudflare Tunnel installed

🌍 Starting Cloudflare Tunnel...
Waiting for public URL...

2026-08-28T16:35:26Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-28T16:35:26Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-28T16:35:30Z INF +--------------------------------------------------------------------------------------------+
20

In [ ]:
# 07.2b — Open Streamlit via Colab proxy

from google.colab import output

frontend_url = output.eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("✅ Mali Knowledge Hub frontend:")
print(frontend_url)

✅ Mali Knowledge Hub frontend:
https://8501-m-s-kkb-usc1c0-tkqzwy451p2-c.us-central1-0.prod.colab.dev


In [ ]:
# 07.2c — Start Streamlit in background

import subprocess
import time
import os
import requests

app_path = "/content/app.py"

if not os.path.exists(app_path):
    raise FileNotFoundError(
        "❌ /content/app.py not found. Run Section 07.1 first."
    )

# Stop any stale Streamlit process
subprocess.run(
    "pkill -f 'streamlit run' || true",
    shell=True
)

time.sleep(1)

log_path = "/content/streamlit.log"

streamlit_log = open(
    log_path,
    "w",
    encoding="utf-8"
)

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        app_path,
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
    start_new_session=True
)

print("🚀 Starting Streamlit...")

time.sleep(7)

try:
    r = requests.get(
        "http://127.0.0.1:8501",
        timeout=5
    )

    print("✅ Streamlit status:", r.status_code)

except Exception:
    print("❌ Streamlit failed")
    print(open(log_path).read())
    raise

🚀 Starting Streamlit...
✅ Streamlit status: 200


In [ ]:
# 07.2d — Open Streamlit via Colab proxy

from google.colab import output

frontend_url = output.eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("✅ Mali Knowledge Hub frontend:")
print(frontend_url)

✅ Mali Knowledge Hub frontend:
https://8501-m-s-kkb-usc1c0-tkqzwy451p2-c.us-central1-0.prod.colab.dev


## 🛠️ 08 — Workspace

Safe working area for day-to-day Knowledge Hub development.

Use this section for:
- exploratory questions
- retrieval experiments
- new document ingestion
- temporary diagnostics
- benchmark development
- experimental code

Code in this section may be changed or deleted freely.

**Rule:** Experimental work starts here.  
Changes are moved into Sections 00–07 only after they have been tested and are intended to become part of the stable Knowledge Hub.

### 🧪 Experiment 01 — Cross-document retrieval diversification

**Observed failure:**  
For a question spanning Vision → Strategy → Projects → Implementation, global top-12 retrieval returned 12/12 passages from Vision Mali 2063.

**Hypothesis:**  
Relevant passages from the other documents may exist slightly below the global top-12 and are being crowded out by Vision 2063.

**Experiment:**  
Retrieve the global top-30 candidates and inspect their document distribution before applying any diversification logic.

**Rule:**  
No changes to the stable Retrieval layer until the experiment demonstrates an improvement.

In [ ]:
# ============================================================
# 08 — EXPERIMENT 01
# TOP-30 CANDIDATE POOL
# ============================================================

experiment_question = """
How does the Government of Mali intend to translate the Mali 2063
vision into concrete development priorities and investments over
the next ten years?
"""

candidates = search_knowledge_base(
    experiment_question,
    match_count=30
)

print("=" * 100)
print("EXPERIMENT 01 — GLOBAL TOP-30 CANDIDATES")
print("=" * 100)

print("\nQUESTION:")
print(experiment_question.strip())


# ------------------------------------------------------------
# 1. Document distribution
# ------------------------------------------------------------

distribution = {}

for r in candidates:

    title = (
        r.get("document_title")
        or "UNKNOWN DOCUMENT"
    )

    distribution[title] = (
        distribution.get(title, 0) + 1
    )


print("\n" + "=" * 100)
print("DOCUMENT DISTRIBUTION")
print("=" * 100)

for title, count in sorted(
    distribution.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{count:>2} × {title}")


# ------------------------------------------------------------
# 2. Inspect all candidates
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RANKED CANDIDATES")
print("=" * 100)

for i, r in enumerate(candidates, 1):

    print(
        f"\n#{i:02d} | "
        f"{r.get('similarity', 0):.3f} | "
        f"{r.get('document_title')} | "
        f"p. {r.get('page_number')} | "
        f"{r.get('section_title')}"
    )

    content = (
        r.get("content", "")
        .replace("\n", " ")
        .strip()
    )

    print(content[:300])


print("\n" + "=" * 100)
print("✅ Candidate-pool inspection complete")

### 🧪 Experiment 02 — Query decomposition

**Observed failure:**  
A broad strategic question returned 30/30 chunks from Vision Mali 2063.

**Hypothesis:**  
The problem is not simply lack of retrieval diversity. A single broad query embedding may collapse several distinct research intents into one semantic representation.

**Experiment:**  
Split the original research question into several narrower retrieval queries representing different information needs:

1. long-term vision
2. medium-term strategy
3. concrete structural projects/investments
4. implementation phasing

Then compare which documents each subquery retrieves.

**Goal:**  
Determine whether query decomposition can recover SNEDD, Projets Structurants and Phasage without introducing a Document Router.

In [ ]:
# ============================================================
# 08 — EXPERIMENT 02
# QUERY DECOMPOSITION
# ============================================================

original_question = """
How does the Government of Mali intend to translate the Mali 2063
vision into concrete development priorities and investments over
the next ten years?
"""

subqueries = {
    "Q1_LONG_TERM_VISION": """
    What are the long-term strategic objectives and development
    priorities of Vision Mali 2063?
    """,

    "Q2_MEDIUM_TERM_STRATEGY": """
    What are the development priorities and strategic objectives
    of the SNEDD 2024-2033 for the first decade of Mali 2063?
    """,

    "Q3_CONCRETE_PROJECTS": """
    What priority structural projects and concrete investments
    are planned to implement Mali 2063 and the SNEDD 2024-2033?
    """,

    "Q4_IMPLEMENTATION_PHASING": """
    How are the priority structural projects implementing
    Mali 2063 and the SNEDD phased and sequenced over time,
    especially during 2024-2033?
    """
}

results_by_query = {}

TOP_K_PER_QUERY = 8


print("=" * 100)
print("EXPERIMENT 02 — QUERY DECOMPOSITION")
print("=" * 100)

print("\nORIGINAL QUESTION:")
print(original_question.strip())


# ------------------------------------------------------------
# Run each subquery
# ------------------------------------------------------------

for query_name, query_text in subqueries.items():

    print("\n\n" + "=" * 100)
    print(query_name)
    print("=" * 100)

    print("\nQUERY:")
    print(query_text.strip())

    results = search_knowledge_base(
        query_text,
        match_count=TOP_K_PER_QUERY
    )

    results_by_query[query_name] = results


    # --------------------------------------------------------
    # Document distribution
    # --------------------------------------------------------

    distribution = {}

    for r in results:

        title = (
            r.get("document_title")
            or "UNKNOWN DOCUMENT"
        )

        distribution[title] = (
            distribution.get(title, 0) + 1
        )


    print("\nDOCUMENT DISTRIBUTION:")
    print("-" * 100)

    for title, count in sorted(
        distribution.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        print(f"{count:>2} × {title}")


    # --------------------------------------------------------
    # Ranked results
    # --------------------------------------------------------

    print("\nTOP RESULTS:")
    print("-" * 100)

    for i, r in enumerate(results, 1):

        print(
            f"\n#{i:02d} | "
            f"{r.get('similarity', 0):.3f} | "
            f"{r.get('document_title')} | "
            f"p. {r.get('page_number')} | "
            f"{r.get('section_title')}"
        )

        content = (
            r.get("content", "")
            .replace("\n", " ")
            .strip()
        )

        print(content[:350])


# ------------------------------------------------------------
# Combined corpus coverage
# ------------------------------------------------------------

print("\n\n" + "=" * 100)
print("COMBINED DOCUMENT COVERAGE")
print("=" * 100)

combined_distribution = {}

for query_name, results in results_by_query.items():

    for r in results:

        title = (
            r.get("document_title")
            or "UNKNOWN DOCUMENT"
        )

        combined_distribution[title] = (
            combined_distribution.get(title, 0) + 1
        )

for title, count in sorted(
    combined_distribution.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{count:>2} × {title}")


print("\n" + "=" * 100)
print("✅ Query-decomposition experiment complete")

**Result:** Query decomposition successfully broke the single-document
retrieval monopoly and recovered all four relevant framework documents.

The original broad query returned only Vision Mali 2063. Once decomposed
into distinct information needs — long-term vision, medium-term strategy,
concrete investments and implementation phasing — retrieval correctly
shifted toward the documents most relevant to each information need.

This supports **query representation** as the primary failure mechanism
observed in Experiment 01 and justifies testing automatic research-question
decomposition before introducing document routing.

**New observed limitation:** Retrieval of implementation phasing also surfaced
structurally weak chunks such as table-of-contents entries and very short
section headings. This suggests a separate future issue related to chunk
quality / structural filtering, rather than query representation.

### 🧪 Experiment 03 — Automatic Research Planner

**Observed result from Experiment 02:**  
Manual query decomposition successfully recovered all four relevant framework documents.

**Hypothesis:**  
An LLM can decompose a complex user question into distinct research queries
without being told which documents to retrieve.

**Experiment:**  
Give the planner only the user's original question and ask it to identify
the minimum set of distinct information needs required to answer it.

Each generated research query is then independently submitted to the existing
semantic retrieval system.

**Goal:**  
Determine whether automatic question decomposition can reproduce the retrieval
coverage achieved through manual decomposition.

**Constraint:**  
The planner must reason about information needs, not select or route to specific
documents.

In [ ]:
# ============================================================
# 08 — EXPERIMENT 03
# AUTOMATIC RESEARCH PLANNER
# ============================================================

import json


# ------------------------------------------------------------
# 1. Original user question
# ------------------------------------------------------------

planner_question = """
How does the Government of Mali intend to translate the Mali 2063
vision into concrete development priorities and investments over
the next ten years?
"""


# ------------------------------------------------------------
# 2. Research planner
# ------------------------------------------------------------

def plan_research_queries(question, model="gpt-5-mini"):

    instructions = """
You are a research-planning component for an institutional
knowledge base.

Your task is NOT to answer the user's question.

Instead, identify the distinct information needs that must be
researched in order to answer the question well.

Generate the smallest useful set of retrieval queries.

RULES:

1. Do not answer the user's question.
2. Do not select, name or recommend documents.
3. Do not assume which source contains the answer.
4. Decompose only when the question contains genuinely distinct
   information needs.
5. Avoid redundant or overlapping retrieval queries.
6. Each query must be understandable independently.
7. Each query should be optimized for semantic retrieval from
   government and institutional documents.
8. Preserve important entities, policy names, dates and time
   periods contained in the user's question.
9. Generate between 1 and 5 retrieval queries.
10. Return valid JSON only.

Return exactly this structure:

{
  "research_queries": [
    {
      "label": "short descriptive label",
      "query": "standalone retrieval query"
    }
  ]
}
"""

    response = openai_client.responses.create(
        model=model,
        instructions=instructions,
        input=question
    )

    raw = response.output_text.strip()

    # Remove accidental Markdown fences if the model adds them
    if raw.startswith("```"):
        raw = raw.strip("`")

        if raw.startswith("json"):
            raw = raw[4:].strip()

    return json.loads(raw)


# ------------------------------------------------------------
# 3. Generate research plan
# ------------------------------------------------------------

research_plan = plan_research_queries(
    planner_question
)

print("=" * 100)
print("EXPERIMENT 03 — AUTOMATIC RESEARCH PLANNER")
print("=" * 100)

print("\nORIGINAL QUESTION:")
print(planner_question.strip())

print("\nGENERATED RESEARCH PLAN:")
print("-" * 100)

for i, item in enumerate(
    research_plan["research_queries"],
    1
):
    print(f"\nQ{i} — {item['label']}")
    print(item["query"])


# ------------------------------------------------------------
# 4. Run retrieval for each generated query
# ------------------------------------------------------------

TOP_K_PER_QUERY = 8

planner_results = {}


for i, item in enumerate(
    research_plan["research_queries"],
    1
):

    query_name = f"Q{i}_{item['label']}"

    results = search_knowledge_base(
        item["query"],
        match_count=TOP_K_PER_QUERY
    )

    planner_results[query_name] = results

    print("\n\n" + "=" * 100)
    print(query_name)
    print("=" * 100)

    print("\nQUERY:")
    print(item["query"])


    # --------------------------------------------------------
    # Document distribution
    # --------------------------------------------------------

    distribution = {}

    for r in results:

        title = (
            r.get("document_title")
            or "UNKNOWN DOCUMENT"
        )

        distribution[title] = (
            distribution.get(title, 0) + 1
        )


    print("\nDOCUMENT DISTRIBUTION:")
    print("-" * 100)

    for title, count in sorted(
        distribution.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        print(f"{count:>2} × {title}")


    # --------------------------------------------------------
    # Ranked results
    # --------------------------------------------------------

    print("\nTOP RESULTS:")
    print("-" * 100)

    for rank, r in enumerate(results, 1):

        print(
            f"\n#{rank:02d} | "
            f"{r.get('similarity', 0):.3f} | "
            f"{r.get('document_title')} | "
            f"p. {r.get('page_number')} | "
            f"{r.get('section_title')}"
        )

        content = (
            r.get("content", "")
            .replace("\n", " ")
            .strip()
        )

        print(content[:350])


# ------------------------------------------------------------
# 5. Combined document coverage
# ------------------------------------------------------------

print("\n\n" + "=" * 100)
print("COMBINED DOCUMENT COVERAGE")
print("=" * 100)

combined_distribution = {}

for results in planner_results.values():

    for r in results:

        title = (
            r.get("document_title")
            or "UNKNOWN DOCUMENT"
        )

        combined_distribution[title] = (
            combined_distribution.get(title, 0) + 1
        )


for title, count in sorted(
    combined_distribution.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{count:>2} × {title}")


print("\n" + "=" * 100)
print("✅ Automatic research-planner experiment complete")

**Result:** Automatic query decomposition did not reproduce the retrieval
coverage achieved by manual decomposition.

The planner generated plausible research queries, but they drifted toward
document discovery and broader implementation topics rather than decomposing
the substantive information needs contained in the original question.

Combined retrieval remained strongly concentrated in Vision Mali 2063
(21/24 results), with only limited recovery of Projets Structurants (2)
and SNEDD (1), and no Phasage evidence.

**Interpretation:** The failure does not invalidate query decomposition.
Experiment 02 demonstrated that decomposition can solve the retrieval
concentration problem. Instead, Experiment 03 identifies planner design as
the next layer to improve.

**Design requirement:** The planner should decompose the user's substantive
question into atomic information needs while preserving scope. It should not
search for documents, broaden the question into adjacent research topics,
or assume a corpus structure.

### 🧪 Experiment 04 — Automatic Research Planner v2

**Observed failure in Experiment 03:**  
The automatic planner produced plausible queries, but drifted toward document
discovery and adjacent implementation topics. Retrieval remained concentrated
in Vision Mali 2063.

**Hypothesis:**  
Automatic decomposition can reproduce the benefits of manual decomposition if
the planner is instructed to identify atomic substantive information needs
rather than documents, source types or adjacent research topics.

**Experiment:**  
Run the exact same user question through a revised planner prompt.

No changes are made to:
- the original question
- the embedding model
- semantic retrieval
- the corpus
- top-k per query

Only the planner instructions change.

**Goal:**  
Test whether planner prompt design — rather than the Research Planner
architecture itself — explains the failure observed in Experiment 03.

In [ ]:
# ============================================================
# 08 — EXPERIMENT 04
# AUTOMATIC RESEARCH PLANNER v2
# ============================================================

import json


# ------------------------------------------------------------
# 1. EXACT SAME ORIGINAL QUESTION
# ------------------------------------------------------------

planner_question_v2 = """
How do the humanitarian needs identified for Mali in 2026 relate to the country's longer-term development priorities, and where are the main areas of alignment or tension?
"""


# ------------------------------------------------------------
# 2. Revised research planner
# ------------------------------------------------------------

def plan_research_queries_v2(question, model="gpt-5-mini"):

    instructions = """
You are a research-planning component for a closed institutional
knowledge base.

Your task is NOT to answer the user's question.

Your task is to decompose the user's substantive question into the
smallest useful set of atomic information needs for semantic retrieval.

IMPORTANT DISTINCTION:

Do NOT search for documents or source types.

BAD:
- "Find national development plans implementing the vision."
- "Find public investment programme documents."
- "Find government implementation frameworks."

GOOD:
- "What are the medium-term development priorities for 2024-2033?"
- "What concrete investments are planned to implement these priorities?"
- "How are these investments phased or sequenced over the relevant period?"

The retrieval system will determine which documents contain the evidence.
You only define WHAT INFORMATION is needed.

RULES:

1. Do not answer the user's question.

2. Do not name, select, recommend or search for documents.

3. Do not formulate queries as requests to "find", "identify" or
   "locate" documents, plans, reports, frameworks or sources.

4. Decompose the question into distinct SUBSTANTIVE information needs.

5. Preserve the user's scope.
   Do not expand into adjacent topics merely because they might be
   relevant.

6. Every research query must directly contribute to answering the
   original question.

7. Preserve important entities, policy names, dates and time periods
   explicitly mentioned by the user.

8. Avoid redundant or substantially overlapping queries.

9. Use standalone natural-language questions optimized for semantic
   retrieval from institutional text.

10. Prefer the smallest sufficient decomposition.
    Generate between 1 and 5 queries.

11. Before including a query, internally ask:
    "Would the final answer be materially incomplete without evidence
    answering this query?"
    If not, omit it.

Return valid JSON only.

Return exactly:

{
  "research_queries": [
    {
      "label": "short descriptive label",
      "query": "standalone substantive retrieval question"
    }
  ]
}
"""

    response = openai_client.responses.create(
        model=model,
        instructions=instructions,
        input=question
    )

    raw = response.output_text.strip()

    if raw.startswith("```"):
        raw = raw.strip("`")

        if raw.startswith("json"):
            raw = raw[4:].strip()

    return json.loads(raw)


# ------------------------------------------------------------
# 3. Generate research plan
# ------------------------------------------------------------

research_plan_v2 = plan_research_queries_v2(
    planner_question_v2
)

print("=" * 100)
print("EXPERIMENT 04 — AUTOMATIC RESEARCH PLANNER v2")
print("=" * 100)

print("\nORIGINAL QUESTION:")
print(planner_question_v2.strip())

print("\nGENERATED RESEARCH PLAN:")
print("-" * 100)

for i, item in enumerate(
    research_plan_v2["research_queries"],
    1
):
    print(f"\nQ{i} — {item['label']}")
    print(item["query"])


# ------------------------------------------------------------
# 4. Run EXACT SAME retrieval setup as Experiment 03
# ------------------------------------------------------------

TOP_K_PER_QUERY = 8

planner_results_v2 = {}


for i, item in enumerate(
    research_plan_v2["research_queries"],
    1
):

    query_name = f"Q{i}_{item['label']}"

    results = search_knowledge_base(
        item["query"],
        match_count=TOP_K_PER_QUERY
    )

    planner_results_v2[query_name] = results

    print("\n\n" + "=" * 100)
    print(query_name)
    print("=" * 100)

    print("\nQUERY:")
    print(item["query"])


    # --------------------------------------------------------
    # Document distribution
    # --------------------------------------------------------

    distribution = {}

    for r in results:

        title = (
            r.get("document_title")
            or "UNKNOWN DOCUMENT"
        )

        distribution[title] = (
            distribution.get(title, 0) + 1
        )


    print("\nDOCUMENT DISTRIBUTION:")
    print("-" * 100)

    for title, count in sorted(
        distribution.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        print(f"{count:>2} × {title}")


    # --------------------------------------------------------
    # Ranked results
    # --------------------------------------------------------

    print("\nTOP RESULTS:")
    print("-" * 100)

    for rank, r in enumerate(results, 1):

        print(
            f"\n#{rank:02d} | "
            f"{r.get('similarity', 0):.3f} | "
            f"{r.get('document_title')} | "
            f"p. {r.get('page_number')} | "
            f"{r.get('section_title')}"
        )

        content = (
            r.get("content", "")
            .replace("\n", " ")
            .strip()
        )

        print(content[:350])


# ------------------------------------------------------------
# 5. Combined document coverage
# ------------------------------------------------------------

print("\n\n" + "=" * 100)
print("COMBINED DOCUMENT COVERAGE")
print("=" * 100)

combined_distribution_v2 = {}

for results in planner_results_v2.values():

    for r in results:

        title = (
            r.get("document_title")
            or "UNKNOWN DOCUMENT"
        )

        combined_distribution_v2[title] = (
            combined_distribution_v2.get(title, 0) + 1
        )


for title, count in sorted(
    combined_distribution_v2.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{count:>2} × {title}")


print("\n" + "=" * 100)
print("✅ Automatic research-planner v2 experiment complete")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# NOTEBOOK INDEX / CODE MAP
# ============================================================

import json

try:
    from google.colab import _message
except ImportError:
    raise RuntimeError("This helper is intended for Google Colab.")

notebook = _message.blocking_request(
    "get_ipynb",
    timeout_sec=10
)

cells = notebook.get("ipynb", {}).get("cells", [])

print("=" * 100)
print("MALI KNOWLEDGE HUB — NOTEBOOK INDEX")
print("=" * 100)

for i, cell in enumerate(cells, start=1):

    cell_type = cell.get("cell_type", "unknown")
    source = "".join(cell.get("source", []))

    # First meaningful line
    lines = [
        line.strip()
        for line in source.splitlines()
        if line.strip()
    ]

    if not lines:
        continue

    first_line = lines[0]

    print(f"\nCELL {i:03d} | {cell_type.upper()}")
    print("-" * 100)

    # Markdown: show first few lines
    if cell_type == "markdown":
        for line in lines[:5]:
            print(line[:180])

    # Code: show headings + defined functions/classes + important assignments
    elif cell_type == "code":

        headings = [
            line for line in lines
            if line.startswith("#")
            and (
                "===" in line
                or "—" in line
                or line.startswith("# 0")
                or line.startswith("# 1")
            )
        ]

        functions = [
            line for line in lines
            if line.startswith("def ")
            or line.startswith("class ")
        ]

        important = [
            line for line in lines
            if any(
                key in line
                for key in [
                    "supabase =",
                    "openai_client =",
                    "ingest_document(",
                    "search_knowledge_base(",
                    "answer_question(",
                    "validate_pdf(",
                    "chunk_pdf("
                ]
            )
        ]

        shown = []

        for group in [headings, functions, important]:
            for line in group:
                if line not in shown:
                    shown.append(line)

        if shown:
            for line in shown[:20]:
                print(line[:180])
        else:
            print(first_line[:180])

print("\n" + "=" * 100)
print("✅ Notebook index complete")

### 🧪 Experiment 05 — Evidence-first cross-document synthesis

**Observed result from Experiment 04:**  
Automatic query decomposition successfully identified the two distinct evidence
needs required to answer the original question:

1. Humanitarian needs in Mali in 2026
2. Mali's longer-term development priorities

Retrieval for these evidence needs recovered the relevant document families.
However, the third query — asking directly about alignment and tensions —
again retrieved only the humanitarian response plan.

**Hypothesis:**  
Cross-document comparison should not itself be treated as a retrieval task.
Instead, the system should retrieve evidence separately for the underlying
factual dimensions and then synthesize this evidence to answer the original
analytical question.

**Test:**  
Retrieve evidence separately for:
- humanitarian needs (Q1)
- longer-term development priorities (Q2)

Then combine both evidence sets and generate an answer to the original question
using only this evidence, without retrieving evidence for the comparative
question itself.

In [ ]:
# ============================================================
# 08 — EXPERIMENT 05
# EVIDENCE-FIRST CROSS-DOCUMENT SYNTHESIS
# ============================================================

original_question = """
How do the humanitarian needs identified for Mali in 2026 relate to the
country's longer-term development priorities, and where are the main areas
of alignment or tension?
"""

# ------------------------------------------------------------
# 1. Retrieve the TWO factual evidence needs separately
# ------------------------------------------------------------

q1 = """
What humanitarian needs are identified for Mali in 2026, described by
sector, geographic hotspots, numbers and profiles of affected populations,
severity/scale, and primary drivers?
"""

q2 = """
What are Mali's stated longer-term development priorities and objectives,
including major sectors, geographic and population targets, and core policy
or investment priorities?
"""

q1_results = search_knowledge_base(q1, match_count=12)
q2_results = search_knowledge_base(q2, match_count=12)

# ------------------------------------------------------------
# 2. Combine evidence
# ------------------------------------------------------------

combined_results = q1_results + q2_results

print("=" * 100)
print("DOCUMENT COVERAGE")
print("=" * 100)

from collections import Counter

coverage = Counter(
    r["document_title"]
    for r in combined_results
)

for doc, count in coverage.most_common():
    print(f"{count:2} × {doc}")

# ------------------------------------------------------------
# 3. Build evidence bundle from Q1 + Q2 only
# ------------------------------------------------------------

evidence = build_evidence_bundle(combined_results)

evidence_text = ""

for e in evidence:

    evidence_text += f"""
[{e['evidence_id']}]
DOCUMENT: {e['document_title']}
DOCUMENT_ID: {e['document_id']}
TYPE: {e['document_type']}
ORGANIZATION: {e['organization']}
VERSION: {e['version']}
PAGE: {e['page']}
SECTION: {e['section']}

CONTENT:
{e['content']}

---
"""


# ------------------------------------------------------------
# 4. Grounded synthesis prompt
# ------------------------------------------------------------

system_prompt = """
You are the analytical engine of the Mali Knowledge Hub.

Your task is to answer questions using ONLY the evidence passages
provided to you.

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent facts, policies, targets, relationships or citations.
3. Every substantive factual claim must be supported by the evidence.
4. Cite claims using the exact evidence IDs provided, for example [E3].
5. Never attribute a claim to a document unless that evidence passage
   explicitly comes from that document.
6. Distinguish carefully between:
   - current policy or strategic priorities
   - diagnosis of existing problems
   - targets
   - past results
   - recommendations
   - scenarios or hypothetical futures
7. Scenario passages describe hypothetical or alternative futures.
   They must NEVER be used as evidence for current conditions,
   current government policy, current priorities, or observed results.
   If relevant, they may only be discussed explicitly as scenarios.
8. If the evidence is insufficient, explicitly say so.
9. If documents differ in emphasis, preserve that distinction.
10. Prefer synthesis over simply listing retrieved passages.

IMPORTANT FOR THIS EXPERIMENT:
The evidence was retrieved separately for:
- humanitarian needs
- longer-term development priorities

Use these evidence sets together to answer the original analytical question.
Do not assume alignment or tension unless supported by comparing the evidence.

At the end provide:

EVIDENCE STATUS: SUPPORTED
or
EVIDENCE STATUS: PARTIAL
or
EVIDENCE STATUS: INSUFFICIENT

Use the language of the user's question.
"""


user_prompt = f"""
QUESTION:

{original_question}

EVIDENCE:

{evidence_text}

Answer the question based exclusively on this evidence.
"""


# ------------------------------------------------------------
# 5. Generate answer
# ------------------------------------------------------------

response = openai_client.responses.create(
    model="gpt-5-mini",
    instructions=system_prompt,
    input=user_prompt
)

answer = response.output_text


# ------------------------------------------------------------
# 6. Display answer + citation map
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ANSWER — Q1 + Q2 EVIDENCE ONLY")
print("=" * 100)
print(answer)

print("\n" + "=" * 100)
print("CITATION MAP")
print("=" * 100)

for e in evidence:
    print(
        f"[{e['evidence_id']}] "
        f"{e['document_title']} | "
        f"p. {e['page']} | "
        f"{e['section']}"
    )

## 09 — Research Agent

Transforms a user question into a structured research task and retrieves the most relevant evidence across the Knowledge Hub.

**Output:** Evidence Ledger

In [ ]:
# ============================================================
# 09.1 — RESEARCH PLANNER
# ============================================================

def plan_research(question, model="gpt-5-mini"):
    """
    Converts a user question into the smallest sufficient set
    of factual retrieval questions.

    The planner does NOT answer the question.
    """

    instructions = """
You are the Research Planner of the Mali Knowledge Hub.

Your task is NOT to answer the user's question.

Your task is to identify the MINIMUM factual evidence that must be
retrieved before another agent can answer it.

The downstream Analysis Agent will perform comparison, synthesis,
interpretation and inference.

RULES:

1. Do not answer the user's question.

2. Do not search for, identify, recommend or select documents.

3. Generate retrieval questions only for factual information that is
   explicitly necessary to answer the user's original question.

4. Do not expand the scope by adding potentially useful background
   information unless the user explicitly requested it.

5. Analytical operations are NOT retrieval needs.

   Do NOT create separate retrieval queries for:
   - comparison
   - alignment
   - tensions
   - gaps
   - implications
   - synthesis
   - assessment
   - recommendations

   The Analysis Agent will derive these from the retrieved evidence.

6. Do not create separate queries for subtopics that can naturally be
   captured inside a broader factual information need.

   Example:
   If the user asks about humanitarian needs, displacement, food
   insecurity and service access may be aspects of that evidence.
   They should NOT automatically become separate research queries.

7. Keep each retrieval question focused.
   Do not turn it into a long checklist of every possible indicator,
   sector, actor or data point.

8. Preserve explicitly stated:
   - geography
   - dates/time periods
   - actors
   - policy names
   - substantive scope

9. Prefer the smallest sufficient decomposition.
   Usually 1–3 queries.
   Maximum 5 only when the original question genuinely contains
   multiple independent factual components.

10. Before adding each query, ask:
    "Would the final answer be materially impossible or incomplete
    without separately retrieving this information?"

    If no, omit it.

11. Each query must be a concise, standalone natural-language question
    suitable for semantic retrieval.

Return valid JSON only in exactly this structure:

{
  "research_queries": [
    {
      "label": "short descriptive label",
      "query": "concise standalone factual retrieval question"
    }
  ]
}
"""

    response = openai_client.responses.create(
        model=model,
        instructions=instructions,
        input=question
    )

    raw = response.output_text.strip()

    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)

    return json.loads(raw)


print("✅ Research Planner loaded")

✅ Research Planner loaded


In [ ]:
test_question = """
How do the humanitarian needs identified for Mali in 2026 relate to the
country's longer-term development priorities, and where are the main areas
of alignment or tension?
"""

plan = plan_research(test_question)

print(json.dumps(plan, indent=2, ensure_ascii=False))

{
  "research_queries": [
    {
      "label": "Mali 2026 humanitarian needs (HNO/HRP)",
      "query": "What humanitarian needs for Mali are documented in official 2026 humanitarian publications (e.g., 2026 Humanitarian Needs Overview and 2026 Humanitarian Response Plan), specifying numbers of people in need, priority sectors, geographic distribution (regions/prefectures/localities), severity/acute needs, and the stated humanitarian strategic objectives and response priorities?"
    },
    {
      "label": "Mali national longer-term development priorities (national plan/strategy)",
      "query": "What are Mali's official longer-term development priorities as set out in the most recent national development plan or strategy (name and coverage years), including the plan's stated vision, priority sectors and objectives, geographic or population targets, key programs/flagship policies, and implementation timeline?"
    }
  ]
}


In [ ]:
# ============================================================
# 09.2 — RESEARCH AGENT
# ============================================================

def run_research_agent(
    question,
    top_k_per_query=10,
    model="gpt-5-mini"
):
    """
    Plans the research task, executes semantic retrieval for
    each factual information need, and deduplicates evidence.

    Does NOT answer the user's question.
    """

    # --------------------------------------------------------
    # 1. Create research plan
    # --------------------------------------------------------

    research_plan = plan_research(
        question,
        model=model
    )

    research_queries = research_plan["research_queries"]

    # --------------------------------------------------------
    # 2. Retrieve evidence for each query
    # --------------------------------------------------------

    retrieved = []

    for query_index, item in enumerate(
        research_queries,
        start=1
    ):
        query_id = f"Q{query_index}"

        results = search_knowledge_base(
            item["query"],
            match_count=top_k_per_query
        )

        for rank, r in enumerate(results, start=1):
            retrieved.append({
                **r,
                "research_query_id": query_id,
                "research_query_label": item["label"],
                "research_query": item["query"],
                "retrieval_rank": rank
            })

    # --------------------------------------------------------
    # 3. Deduplicate chunks
    # --------------------------------------------------------

    unique = {}

    for r in retrieved:

        chunk_key = (
            r.get("id")
            or r.get("chunk_id")
            or (
                r.get("document_id"),
                r.get("page_number"),
                r.get("content")
            )
        )

        retrieval_context = {
            "query_id": r["research_query_id"],
            "label": r["research_query_label"],
            "query": r["research_query"],
            "rank": r["retrieval_rank"]
        }

        if chunk_key not in unique:

            unique[chunk_key] = {
                **r,
                "retrieved_for": [retrieval_context]
            }

        else:

            unique[chunk_key]["retrieved_for"].append(
                retrieval_context
            )

    unique_results = list(unique.values())

    # Highest semantic similarity first
    unique_results.sort(
        key=lambda x: x.get("similarity") or 0,
        reverse=True
    )

    # --------------------------------------------------------
    # 4. Return structured research output
    # --------------------------------------------------------

    return {
        "question": question,
        "research_plan": research_plan,
        "results": unique_results,
        "raw_retrieval_count": len(retrieved),
        "unique_evidence_count": len(unique_results)
    }


print("✅ Research Agent loaded")

✅ Research Agent loaded


In [ ]:
research = run_research_agent(
    test_question,
    top_k_per_query=10
)

print("Research queries:")
for i, q in enumerate(
    research["research_plan"]["research_queries"],
    1
):
    print(f"{i}. {q['label']}")
    print(f"   {q['query']}")
    print()

print("Raw retrievals:", research["raw_retrieval_count"])
print("Unique evidence:", research["unique_evidence_count"])

print("\nDocument coverage:")

from collections import Counter

coverage = Counter(
    r.get("document_title")
    for r in research["results"]
)

for document, count in coverage.items():
    print(f"- {document}: {count}")


Research queries:
1. Humanitarian needs in Mali (2026)
   What humanitarian needs were identified for Mali in 2026 in official humanitarian documents (e.g., the 2026 Humanitarian Needs Overview and 2026 Humanitarian Response Plan), including the numbers of people in need, priority sectors, severity levels, and geographic distribution by region/district and priority populations?

2. Mali longer-term development priorities
   What are Mali's official longer-term development priorities as set out in national development strategy documents and the UN Sustainable Development Cooperation Framework (titles and timeframes), including stated priority sectors, strategic objectives, geographic or population focus, and any quantified targets or timelines?

Raw retrievals: 20
Unique evidence: 20

Document coverage:
- Mali — Besoins humanitaires et Plan de Réponse 2026: 10
- Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024-2033): 5
- Projets Structurants Prioritaires pour

## 10 — Evidence Ledger

Creates a standardized evidence layer with:
- Evidence ID
- Source document
- Page / chunk
- Passage
- Supported proposition
- Evidence type
- Retrieval provenance

In [ ]:
# ============================================================
# 10.1 — EVIDENCE LEDGER
# ============================================================

def build_evidence_ledger(research_output):
    """
    Converts Research Agent output into a standardized evidence ledger.

    Each evidence item receives a stable evidence ID and preserves
    source provenance and retrieval context.
    """

    ledger = []

    for i, r in enumerate(
        research_output["results"],
        start=1
    ):

        ledger.append({
            "evidence_id": f"E{i:02d}",

            # Source provenance
            "source_type": "knowledge_base_document",
            "document_id": r.get("document_id"),
            "document_title": r.get("document_title"),
            "document_type": r.get("document_type"),
            "organization": r.get("organization"),
            "publication_date": r.get("publication_date"),
            "version": r.get("version"),

            # Location inside source
            "page": r.get("page_number"),
            "section": r.get("section_title"),
            "chunk_id": r.get("chunk_id") or r.get("id"),

            # Retrieval information
            "similarity": r.get("similarity"),
            "retrieved_for": r.get("retrieved_for"),

            # Evidence content
            "passage": r.get("content"),

            # To be enriched later
            "evidence_type": None,
            "supports": None
        })

    return ledger


print("✅ Evidence Ledger loaded")

✅ Evidence Ledger loaded


In [ ]:
ledger = build_evidence_ledger(research)

print("Evidence items:", len(ledger))
print()

for item in ledger[:5]:
    print(
        item["evidence_id"],
        "|",
        item["document_title"],
        "| page",
        item["page"],
        "| similarity",
        round(item["similarity"], 3)
    )

    print(
        item["passage"][:300]
        .replace("\n", " ")
    )

    print("─" * 80)

Evidence items: 20

E01 | Mali — Besoins humanitaires et Plan de Réponse 2026 | page 15 | similarity 0.829
2.3 Stratégie de réponse  humanitaire US $577,9 requis pour soutenir 3,8M personnes ciblées En 2026, la communauté humanitaire ciblera 3,8 millions de personnes parmi les 5,1 millions de personnes en besoin avec un budget de 577,9 millions USD. L’aggravation combinée des violences, de l’insécurité, 
────────────────────────────────────────────────────────────────────────────────
E02 | Mali — Besoins humanitaires et Plan de Réponse 2026 | page 5 | similarity 0.815
1.1 Aperçu de la crise  L’évolution de la situation humanitaire en 2025 laisse entrevoir une année 2026 durant laquelle les populations continueront de subir l’effet combiné de facteurs sécuritaires, climatiques, sanitaires et socio- économiques. Les opérations militaires en cours dans le nord et le
────────────────────────────────────────────────────────────────────────────────
E03 | Mali — Besoins humanitaires et Plan d

## 11 — Analysis Agent

Synthesizes the Evidence Ledger into an analytical draft.

The analyst may distinguish between:
- [FACT] directly supported by evidence
- [SYNTHESIS] combining several pieces of evidence
- [INFERENCE] analytical interpretation based on the evidence

The analyst cannot introduce unsupported external facts.

In [ ]:
# ============================================================
# 11 — ANALYSIS AGENT v2.1
# ============================================================

def run_analysis_agent(
    question,
    evidence_ledger,
    model="gpt-5-mini"
):
    """
    Produce an evidence-grounded analytical draft from the
    Evidence Ledger.

    v2.1:
    - synthesis and inference allowed
    - no novel substantive content
    - no invented interventions/outcomes/causal pathways
    - no ranking or geographic concentration from selected records
    - strict structured-data handling
    """

    # --------------------------------------------------------
    # FORMAT EVIDENCE
    # --------------------------------------------------------

    evidence_blocks = []

    for item in evidence_ledger:
        evidence_id = item.get("evidence_id")
        source_type = item.get("source_type")

        if source_type == "hdx_hapi":
            block = f"""
EVIDENCE {evidence_id}
SOURCE TYPE: HDX HAPI
RESOURCE: {item.get("document_title")}
ORGANIZATION: {item.get("organization")}
REFERENCE PERIOD: {item.get("reference_period")}
GEOGRAPHY LEVEL: {item.get("admin_level")}
ADMIN1: {item.get("admin1_name")}
ADMIN2: {item.get("admin2_name")}
SECTOR: {item.get("sector")}
POPULATION STATUS: {item.get("population_status")}
VALUE: {item.get("value")}
UNIT: {item.get("unit")}
PASSAGE:
{item.get("passage")}

SYSTEM INTERPRETATION CONSTRAINT:
{item.get("aggregation_note")}
"""
        else:
            block = f"""
EVIDENCE {evidence_id}
SOURCE TYPE: KNOWLEDGE BASE DOCUMENT
DOCUMENT: {item.get("document_title")}
DOCUMENT TYPE: {item.get("document_type")}
ORGANIZATION: {item.get("organization")}
PUBLICATION DATE: {item.get("publication_date")}
PAGE: {item.get("page")}
SECTION: {item.get("section")}
PASSAGE:
{item.get("passage")}
"""

        evidence_blocks.append(block)

    formatted_evidence = "\n\n".join(evidence_blocks)

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    instructions = """
You are the Analysis Agent of the Mali Knowledge Hub.

Your task is to answer the user's analytical question using ONLY
the supplied Evidence Ledger.

You may synthesize and infer.

You may NOT introduce substantive information that is absent from
the supplied evidence.


============================================================
CORE PRINCIPLE
============================================================

Reason across evidence.

Do not reason beyond evidence.

A useful analytical answer may:

- combine documented facts
- compare documented facts
- identify thematic overlap
- identify tensions
- identify gaps in the supplied evidence
- draw cautious analytical implications

But synthesis and inference do NOT grant permission to invent new
substantive content.


============================================================
EPISTEMIC LABELS
============================================================

Every analytical statement must use one of:

[FACT]
[SYNTHESIS]
[INFERENCE]

[FACT]
A statement explicitly supported by one or more evidence items.

[SYNTHESIS]
A relationship, comparison, pattern or thematic connection derived
from multiple supported facts.

[INFERENCE]
A cautious analytical implication that goes beyond explicit source
wording but is logically grounded in the supplied evidence.

Do not disguise inference as fact.


============================================================
NOVEL CONTENT PROHIBITION
============================================================

THIS IS A HARD RULE.

Do not introduce a concrete substantive element unless it is
explicitly present in the Evidence Ledger.

This includes:

- interventions
- programmes
- projects
- implementation activities
- policy measures
- institutional actions
- actors
- organizations
- locations
- sectors
- statistics
- target groups
- outcomes
- causal mechanisms
- causal steps
- recommendations

If an element is not present in the Evidence Ledger, do not add it
merely because it seems obvious, likely, useful or domain-relevant.


============================================================
INFERENCE BOUNDARY
============================================================

Inference may connect concepts already documented in the evidence.

Inference may NOT create new concepts to bridge them.

VALID:

Evidence documents humanitarian needs in food security and a
national strategy prioritizes food security.

→

[INFERENCE] Addressing documented food-security needs may support
progress toward the stated national food-security priority.

INVALID:

→

[INFERENCE] Cash transfers and agricultural input programmes could
restore livelihoods and improve community resilience.

unless cash transfers, agricultural input programmes, restored
livelihoods and community resilience are themselves supported by
the Evidence Ledger.


============================================================
NO NEW INTERVENTIONS OR OUTCOMES
============================================================

Do not convert a documented NEED into a proposed INTERVENTION.

For example:

"Mine Action needs"

does NOT automatically authorize:

"demining operations"

"Food Security needs"

does NOT automatically authorize:

"food assistance"

"WASH needs"

does NOT automatically authorize:

"WASH service support"

unless those interventions are explicitly documented.

Likewise, do not introduce outcomes such as:

- safe return
- agricultural recovery
- restored livelihoods
- improved stability
- stronger institutions
- increased trust
- restored social cohesion

unless those outcomes are explicitly present in the evidence.

Keep analytical implications at the level supported by the
evidence.


============================================================
CAUSALITY
============================================================

Thematic overlap does not establish causality.

Do not invent causal pathways between:

humanitarian need
→ intervention
→ intermediate outcome
→ development objective

unless every substantive step is supported by the Evidence Ledger.

When the evidence supports only thematic relationship, describe
only thematic relationship.


============================================================
DIRECTNESS
============================================================

Do not say that two bodies of evidence:

- directly align
- directly correspond
- directly map
- directly relate

unless the sources explicitly establish that relationship.

Prefer:

- align with
- relate to
- correspond with
- have thematic overlap with
- can be connected analytically to


============================================================
PRIORITY AND RANKING
============================================================

Do not infer ranking from selected evidence.

Do not call a sector, location, need or issue:

- main
- primary
- largest
- leading
- most important
- top priority
- dominant

unless the supplied evidence establishes that ranking.

If the evidence is a selected subset, describe it as documented
or selected evidence rather than a complete ranking.


============================================================
GEOGRAPHIC CONCENTRATION
============================================================

Do not infer geographic concentration from selected observations.

Do not use:

- concentrated
- concentration
- especially concentrated
- primarily concentrated

unless the evidence explicitly establishes geographic
concentration.

You may say:

- documented in
- reported in
- observed in
- substantial in
- multiple needs are recorded in


============================================================
COMPARATIVE LANGUAGE
============================================================

Words such as:

- larger
- smaller
- higher
- lower
- especially large
- particularly high

require a valid comparison within comparable evidence.

Do not use comparative language merely because a number appears
large in isolation.

When selected records do not establish the relevant comparison,
state the values without ranking them.


============================================================
STRUCTURED DATA
============================================================

For structured evidence preserve exactly:

- geography
- administrative level
- sector
- population category
- population status
- unit
- reference period
- value

Do not silently harmonize different dimensions.

Do not sum Admin2 observations into an Admin1 total unless the
Evidence Ledger explicitly provides that aggregate.

Do not interpret selected sector-location observations as a
complete sector ranking for an Admin1 region.


============================================================
SYSTEM INTERPRETATION CONSTRAINTS
============================================================

A SYSTEM INTERPRETATION CONSTRAINT is an instruction for how the
Knowledge Hub should interpret evidence.

It is NOT source evidence.

Do not write that a source itself imposes a methodological rule
unless the source passage explicitly says so.

For example, if the system constraint says Admin2 observations
must not be summed:

VALID:

[FACT] The supplied evidence contains Admin2 observations and does
not provide an Admin1 aggregate [E11, E12].

[INFERENCE] This analysis therefore does not derive an Admin1
aggregate by summing those observations.

INVALID:

[FACT] The HNO states that these observations must not be summed
[E11, E12].


============================================================
ABSENCE OF EVIDENCE
============================================================

Distinguish:

"The supplied evidence does not provide X"

from:

"X does not exist."

You may identify gaps in the supplied evidence.

Do not infer global absence from retrieval absence.


============================================================
TIME AND FRESHNESS
============================================================

Distinguish:

- reference period
- publication date
- source update date
- retrieval date

Do not describe data as "current" merely because it was recently
retrieved or updated.


============================================================
CITATIONS
============================================================

Every substantive claim must cite the relevant Evidence IDs.

Use only Evidence IDs present in the ledger.

Write IDs individually.

VALID:

[E01]

[E01, E03, E07]

[E11, E12, E13, E14]

INVALID:

[E01–E07]

[E11–E18]

[E11 through E18]

Never create citation ranges.

Do not cite a SYSTEM INTERPRETATION CONSTRAINT as if it were source
evidence.


============================================================
ANSWERING "MAIN" OR "PRIORITY" QUESTIONS
============================================================

If the user asks for "main", "priority", "most important" or
similar ranking but the supplied evidence does not establish a
complete ranking:

do NOT manufacture one.

Instead answer with the principal/documented themes visible in the
supplied evidence and make the evidentiary limitation clear.

Do not allow the wording of the user's question to force a ranking
that the evidence cannot support.


============================================================
STYLE
============================================================

Produce a concise but analytically useful internal draft.

Prefer:

- evidence
- synthesis
- implications
- limitations

over generic introduction.

Do not add rhetorical flourishes.

Do not add suggestions about what the user could ask next.

Do not discuss your own process.


============================================================
FINAL SELF-CHECK
============================================================

Before returning the draft, silently ask:

1. Is every fact supported?
2. Is every synthesis grounded in supported facts?
3. Does every inference use only substantive concepts already
   present in the evidence?
4. Did I invent an intervention?
5. Did I invent an outcome?
6. Did I invent a causal step?
7. Did I infer ranking from selected records?
8. Did I infer geographic concentration?
9. Did I use unsupported comparative language?
10. Did I manufacture an aggregate?
11. Did I treat a system constraint as source evidence?
12. Did I create citation ranges?

If yes to 4–12, repair the draft before returning it.


============================================================
OUTPUT
============================================================

Return only the analytical draft.

Use [FACT], [SYNTHESIS] and [INFERENCE] labels.
"""

    user_input = f"""
USER QUESTION:

{question}


EVIDENCE LEDGER:

{formatted_evidence}
"""

    response = openai_client.responses.create(
        model=model,
        instructions=instructions,
        input=user_input
    )

    return response.output_text.strip()


print("✅ Analysis Agent v2.1 loaded")

✅ Analysis Agent v2.1 loaded


In [ ]:
analysis_v21_test = run_analysis_agent(
    test_question_v2,
    result_v2["evidence"]
)

print(analysis_v21_test)

[FACT] The supplied evidence records large intersectoral humanitarian needs at Admin2 level across Mopti: Djenne (152,595 people in need) [E11]; Konna (363,384) [E12]; Korientze (293,052) [E13]; Mopti (156,243) [E14]; Sofara (133,605) [E15]; Tenenkou (180,964) [E16]; Togueré‑Coumbé (57,333) [E17]; Youwarou (113,636) [E18]. Each observation is an Admin2‑level “people in need” figure for 2025 (reference period in the records) [E11–E18]. 

[FACT] The evidence also documents sector‑specific needs in Mopti (Admin2): Protection in Korientze (291,933 people in need) [E19]; Food Security in Konna (249,711) [E20]; Mine Action in Korientze (247,184) [E21]; Health in Korientze (203,421) [E22]; Water, Sanitation and Hygiene in Konna (171,294) [E23]; Gender‑Based Violence in Korientze (166,910) [E24].

[SYNTHESIS] The sectoral humanitarian needs recorded in Mopti (protection, mine action, food security, health, WASH, GBV) correspond thematically with development issues emphasized in Mali’s long‑ter

## 12 — Factchecker

Audits every substantive claim in the analytical draft against the Evidence Ledger and, where necessary, the original retrieved passages.

Possible verdicts:
- Fully supported
- Partially supported
- Unsupported
- Citation mismatch
- Reasonable inference
- Inference too strong

In [ ]:
# ============================================================
# 12 — FACTCHECKER v2.2
# ============================================================

import json
import re


def run_factchecker(
    question,
    analysis_draft,
    evidence_ledger,
    model="gpt-5-mini"
):
    """
    Audit the analytical draft claim by claim against the
    Evidence Ledger.

    v2.2:
    - novel-content override
    - system-constraint attribution override
    - comparative / concentration override
    - citation-range detection
    - preserves v2.1 directness / necessity / causality rules
    """

    # --------------------------------------------------------
    # FORMAT EVIDENCE
    # --------------------------------------------------------

    evidence_blocks = []

    for item in evidence_ledger:
        evidence_id = item.get("evidence_id")
        source_type = item.get("source_type")

        if source_type == "hdx_hapi":
            block = f"""
EVIDENCE {evidence_id}
SOURCE TYPE: HDX HAPI
RESOURCE: {item.get("document_title")}
ORGANIZATION: {item.get("organization")}
REFERENCE PERIOD: {item.get("reference_period")}
GEOGRAPHY LEVEL: {item.get("admin_level")}
ADMIN1: {item.get("admin1_name")}
ADMIN2: {item.get("admin2_name")}
SECTOR: {item.get("sector")}
POPULATION STATUS: {item.get("population_status")}
VALUE: {item.get("value")}
UNIT: {item.get("unit")}
PASSAGE:
{item.get("passage")}

SYSTEM INTERPRETATION CONSTRAINT:
{item.get("aggregation_note")}
"""
        else:
            block = f"""
EVIDENCE {evidence_id}
SOURCE TYPE: KNOWLEDGE BASE DOCUMENT
DOCUMENT: {item.get("document_title")}
DOCUMENT TYPE: {item.get("document_type")}
ORGANIZATION: {item.get("organization")}
PUBLICATION DATE: {item.get("publication_date")}
PAGE: {item.get("page")}
SECTION: {item.get("section")}
PASSAGE:
{item.get("passage")}
"""

        evidence_blocks.append(block)

    formatted_evidence = "\n\n".join(evidence_blocks)

    valid_evidence_ids = [
        item.get("evidence_id")
        for item in evidence_ledger
        if item.get("evidence_id")
    ]

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    instructions = """
You are the independent Factchecker and Evidence Auditor of the
Mali Knowledge Hub.

You receive:

1. a user question
2. an analytical draft
3. an Evidence Ledger

Audit every substantive claim against the supplied evidence.

Do NOT improve the answer yourself.

Your job is to identify exactly what is supported, what is
analytical interpretation, what is overstated, and what must be
repaired.


============================================================
ALLOWED STATUSES
============================================================

Use exactly one of these statuses for every claim:

FULLY_SUPPORTED

PARTIALLY_SUPPORTED

UNSUPPORTED

CITATION_MISMATCH

REASONABLE_INFERENCE

INFERENCE_TOO_STRONG


============================================================
STATUS DEFINITIONS
============================================================

FULLY_SUPPORTED

The substantive claim is directly supported by the cited evidence.


PARTIALLY_SUPPORTED

Part of the claim is supported, but another substantive component
is not established by the evidence.


UNSUPPORTED

The claim introduces substantive content that is absent from the
supplied evidence.


CITATION_MISMATCH

The underlying proposition may be supportable, but:

- the cited evidence does not support it,
- the wrong source is attributed,
- a system interpretation constraint is attributed to a source,
- or the citation relationship is otherwise incorrect.


REASONABLE_INFERENCE

The claim goes beyond explicit source wording but:

- is clearly presented as inference,
- follows cautiously from the evidence,
- and introduces no unsupported substantive element.


INFERENCE_TOO_STRONG

The claim is inferential but:

- overstates directness,
- necessity,
- causality,
- ranking,
- comparison,
- concentration,
- certainty,
- or another relationship beyond what the evidence establishes.


============================================================
CORE AUDIT PRINCIPLE
============================================================

Synthesis and inference are allowed.

Novel substantive content is not.

An inference may CONNECT concepts already documented in the
Evidence Ledger.

An inference may NOT INVENT substantive concepts in order to
bridge the evidence.


============================================================
NOVEL CONTENT OVERRIDE
============================================================

THIS IS A DETERMINISTIC OVERRIDE.

Inspect every [SYNTHESIS] and [INFERENCE] claim for concrete
substantive elements not present in the Evidence Ledger.

This includes newly introduced:

- interventions
- programmes
- projects
- activities
- implementation approaches
- policy measures
- institutional actions
- actors
- organizations
- locations
- sectors
- target groups
- outcomes
- causal mechanisms
- intermediate outcomes
- recommendations

If a claim introduces such a concrete element and that element is
not supported by the Evidence Ledger:

it MUST NOT receive REASONABLE_INFERENCE.

Use:

UNSUPPORTED

when the novel element is itself a substantive proposition or
example that cannot be grounded in the ledger.

Use:

INFERENCE_TOO_STRONG

when the underlying analytical relationship is reasonable but the
claim creates an unsupported causal mechanism, outcome or
substantive bridge.

Required fix:

remove the novel element and retain only the narrower analytical
relationship supported by the evidence.


Example:

Evidence:
- Food Security needs are documented.
- National strategy prioritizes food security.

Claim:
"Food assistance could restore agricultural livelihoods."

If neither food assistance nor restored agricultural livelihoods
is documented:

→ NOT REASONABLE_INFERENCE.

The valid narrower inference may be:

"Addressing the documented food-security needs may support
progress toward the stated food-security priority."


============================================================
INTERVENTION / OUTCOME OVERRIDE
============================================================

A documented NEED does not automatically establish an
INTERVENTION.

Examples:

Mine Action need
≠ evidence of demining as the proposed intervention

Food Security need
≠ evidence of food assistance

WASH need
≠ evidence of WASH service delivery

Protection need
≠ evidence of protection programming

Likewise, do not accept newly introduced outcomes such as:

- safe return
- agricultural recovery
- restored livelihoods
- improved stability
- increased trust
- stronger institutions
- restored social cohesion

unless those substantive outcomes are supported by the supplied
evidence.

Domain plausibility is not evidence.


============================================================
DIRECTNESS OVERRIDE
============================================================

THIS IS A DETERMINISTIC OVERRIDE.

If a claim uses language such as:

- directly relates
- directly corresponds
- directly aligns
- directly maps
- maps directly
- directly enables

and the evidence does not explicitly establish that direct
relationship:

status MUST be:

INFERENCE_TOO_STRONG

Required fix must remove the directness language.

Prefer formulations such as:

- relates to
- aligns with
- corresponds with
- has thematic overlap with


============================================================
NECESSITY OVERRIDE
============================================================

THIS IS A DETERMINISTIC OVERRIDE.

If a claim uses:

- necessary
- required
- essential
- prerequisite
- must
- indispensable
- cannot occur without
- depends on

and the source evidence does not establish necessity:

status MUST be:

INFERENCE_TOO_STRONG

Modifiers such as:

- likely necessary
- probably required
- potentially essential

do NOT make the claim acceptable.

Required fix should weaken the relationship to language such as:

- may support
- could contribute
- may facilitate
- may be relevant to


============================================================
CAUSALITY OVERRIDE
============================================================

A thematic relationship does not establish causal impact.

Claims asserting that X:

- causes
- produces
- enables
- leads to
- results in
- removes barriers to
- ensures
- delivers

Y require evidence supporting that causal relationship.

If the underlying relationship is plausible but causality is not
established:

use INFERENCE_TOO_STRONG.

If the claim also introduces novel interventions, mechanisms or
outcomes, apply the NOVEL CONTENT OVERRIDE as well.


============================================================
COMPARATIVE / CONCENTRATION OVERRIDE
============================================================

THIS IS A DETERMINISTIC OVERRIDE.

Words implying comparison, ranking or geographic concentration
require evidence that establishes the relevant comparison.

Scrutinize:

- concentrated
- concentration
- especially concentrated
- primarily concentrated
- main
- primary
- largest
- highest
- lowest
- leading
- dominant
- most important
- top priority
- especially large
- particularly high
- disproportionately affected

A selected structured-data observation does NOT by itself
establish these properties.

For example:

Four large sector values from Korientze do NOT establish that
protection needs are "concentrated" in Korientze unless the
evidence contains an appropriate geographic comparison.

If the comparative or concentration language is not established:

status MUST be:

INFERENCE_TOO_STRONG

or PARTIALLY_SUPPORTED if the underlying factual statement and
values are directly supported.

Required fix:

remove the comparative/ranking/concentration language and preserve
the documented observations.


============================================================
PRIORITY / RANKING RULE
============================================================

Do not infer a complete ranking from selected evidence.

If selected sector records are presented as:

- the main humanitarian priorities
- the primary needs
- the largest sectors
- the most important needs

without evidence establishing that ranking:

the claim is not FULLY_SUPPORTED.

Require neutral wording such as:

- documented needs
- selected sector observations
- needs represented in the supplied evidence


============================================================
SYSTEM INTERPRETATION CONSTRAINT OVERRIDE
============================================================

THIS IS A DETERMINISTIC OVERRIDE.

A SYSTEM INTERPRETATION CONSTRAINT is generated by the Knowledge
Hub.

It is NOT source evidence.

If a draft says or implies that:

- the source states the constraint,
- the source includes the constraint,
- the source instructs the analyst to follow the constraint,
- the source explicitly prohibits an operation,

when that information appears only under:

SYSTEM INTERPRETATION CONSTRAINT

the claim MUST NOT be FULLY_SUPPORTED.

Use:

CITATION_MISMATCH

when the substantive methodological rule is being incorrectly
attributed to the source.

Use:

PARTIALLY_SUPPORTED

when the sentence combines a source-supported fact with a
system-generated methodological rule.

Required fix must separate:

1. the source-supported factual observation

from:

2. the Knowledge Hub's analytical handling.

Example:

INVALID:

"The source records state that Admin2 observations must not be
summed [E11]."

REPAIR:

"The supplied evidence contains Admin2 observations and does not
provide an Admin1 aggregate [E11, E12]."

Then separately:

"This analysis therefore does not derive an Admin1 aggregate by
summing those observations."


============================================================
STRUCTURED DATA RULES
============================================================

For structured evidence verify:

- geography
- administrative level
- sector
- population category
- population status
- unit
- reference period
- value

Do not allow silent aggregation across dimensions.

Admin2 values must not be represented as an Admin1 total unless
an explicit Admin1 aggregate exists in the evidence.

Selected sector-location observations must not be treated as a
complete regional sector ranking.


============================================================
AGGREGATION RULE
============================================================

Do not treat absence of an aggregate in the supplied evidence as
proof that no aggregate exists anywhere.

Valid:

"The supplied evidence does not provide an Admin1 aggregate."

Too strong:

"No Admin1 aggregate exists."


============================================================
ABSENCE-OF-EVIDENCE RULE
============================================================

Distinguish:

"not present in the supplied evidence"

from:

"does not exist."

Retrieval absence is not global absence.


============================================================
TIME / FRESHNESS RULE
============================================================

Distinguish:

- reference period
- publication date
- resource update date
- HAPI update date
- retrieval date

Do not call data "current" or "latest" merely because the API
record was recently retrieved or updated.


============================================================
CITATION RANGE OVERRIDE
============================================================

Citation ranges are NOT allowed in the Mali Knowledge Hub.

Examples of invalid citation syntax:

[E11–E18]

[E11-E18]

[E01–E06]

[E11 through E18]

If a claim uses a citation range:

do NOT change the substantive status solely because of citation
format.

Instead:

- retain the correct substantive status
- explicitly mention the invalid citation range in required_fix
- instruct the Grounded Rewriter to expand it into individual IDs

Example:

[E11–E14]

must become:

[E11, E12, E13, E14]

Never assume that an intermediate Evidence ID supports a claim
merely because it lies numerically inside a range. Verify the
individual evidence items.


============================================================
CITATION VALIDITY
============================================================

Every cited Evidence ID must exist in the supplied Evidence Ledger.

Check whether the cited evidence actually supports the claim.

Do not accept citation quantity as evidence quality.


============================================================
RECOMMENDATIONS AND POLICY IMPLICATIONS
============================================================

Recommendations require special scrutiny.

Do not treat a recommendation as evidence-backed merely because
the problem it addresses is documented.

A recommendation may be a reasonable inference only if it does
not introduce unsupported substantive assumptions, interventions,
actors, mechanisms or outcomes.

Otherwise classify accordingly.


============================================================
CLAIM GRANULARITY
============================================================

Audit the draft claim by claim.

A claim may be:

- one sentence
- a logically distinct clause
- a clearly separable analytical proposition

Split compound statements when different parts require different
statuses.

Do not hide an unsupported proposition inside an otherwise
supported paragraph.


============================================================
FINAL SELF-CHECK
============================================================

Before returning the audit, silently verify:

1. Did any inference introduce a concrete element absent from the
   ledger?
2. Did I accidentally accept domain plausibility as evidence?
3. Did I accept an invented intervention?
4. Did I accept an invented outcome?
5. Did I accept an unsupported causal pathway?
6. Did I accept "directly" without explicit evidence?
7. Did I accept necessity language without explicit evidence?
8. Did I accept ranking or concentration from selected records?
9. Did I treat a system constraint as source evidence?
10. Did I notice all citation ranges?
11. Did I verify cited IDs against the actual ledger?

If any answer indicates a missed issue, repair the audit before
returning it.


============================================================
OUTPUT FORMAT
============================================================

Return valid JSON only.

Use exactly this structure:

{
  "overall_assessment": "short assessment",
  "claims": [
    {
      "claim": "exact or clearly identifiable claim",
      "status": "FULLY_SUPPORTED | PARTIALLY_SUPPORTED | UNSUPPORTED | CITATION_MISMATCH | REASONABLE_INFERENCE | INFERENCE_TOO_STRONG",
      "cited_evidence": ["E01", "E02"],
      "reason": "specific explanation",
      "required_fix": "specific repair instruction or null"
    }
  ]
}

Do not wrap the JSON in markdown fences.
Do not add commentary outside the JSON.
"""

    user_input = f"""
USER QUESTION:

{question}


ANALYTICAL DRAFT:

{analysis_draft}


VALID EVIDENCE IDS:

{", ".join(valid_evidence_ids)}


EVIDENCE LEDGER:

{formatted_evidence}
"""

    response = openai_client.responses.create(
        model=model,
        instructions=instructions,
        input=user_input
    )

    raw_output = response.output_text.strip()

    # Remove accidental markdown fences
    raw_output = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_output,
        flags=re.IGNORECASE
    ).strip()

    try:
        return json.loads(raw_output)

    except json.JSONDecodeError as e:
        print("❌ Factchecker returned invalid JSON")
        print(raw_output)
        raise e


print("✅ Factchecker v2.2 loaded")

✅ Factchecker v2.2 loaded


In [ ]:
audit_v22_test = run_factchecker(
    test_question_v2,
    analysis_v21_test,
    result_v2["evidence"]
)

print(json.dumps(
    audit_v22_test,
    indent=2,
    ensure_ascii=False
))

print("\n" + "=" * 70)
print("STATUS COUNTS")
print("=" * 70)

status_counts = {}

for claim in audit_v22_test.get("claims", []):
    status = claim.get("status", "UNKNOWN")
    status_counts[status] = status_counts.get(status, 0) + 1

for status, count in status_counts.items():
    print(f"{status}: {count}")

{
  "overall_assessment": "Most factual sector- and location-level humanitarian figures are supported by the supplied HDX HAPI records. The higher-level syntheses linking those needs to national development priorities are reasonable inferences from the Vision/SNEDD documents, but (a) one synthesis statement overreaches on 'social protection' without a supporting citation and (b) the draft misattributes a system-generated interpretation note to source records. Several claims use an invalid citation range notation and need citation expansion.",
  "claims": [
    {
      "claim": "The supplied evidence records large intersectoral humanitarian needs at Admin2 level across Mopti: Djenne (152,595 people in need) [E11]; Konna (363,384) [E12]; Korientze (293,052) [E13]; Mopti (156,243) [E14]; Sofara (133,605) [E15]; Tenenkou (180,964) [E16]; Togueré‑Coumbé (57,333) [E17]; Youwarou (113,636) [E18]. Each observation is an Admin2‑level “people in need” figure for 2025 (reference period in the rec

## 13 — Grounded Rewriter

Repairs all grounding problems identified by the Factchecker.

It may:
- weaken claims
- remove unsupported claims
- correct citations
- clarify inference

It may not introduce new factual claims.

In [ ]:
# ============================================================
# 13 — GROUNDED REWRITER v2.3
# ============================================================

def run_grounded_rewriter(
    question,
    analysis_draft,
    audit,
    evidence_ledger,
    model="gpt-5-mini"
):
    """
    Repair the analysis draft using the factchecker audit while
    preserving source boundaries, epistemic status and citations.

    v2.3:
    - no new examples/interventions/outcomes
    - no new causal pathways
    - citation IDs may not be compressed into ranges
    - repairs must be achieved by narrowing/rewording existing claims
    - preserves source vs system-constraint distinction
    """

    # --------------------------------------------------------
    # FORMAT EVIDENCE
    # --------------------------------------------------------

    evidence_blocks = []

    for item in evidence_ledger:
        evidence_id = item.get("evidence_id")
        source_type = item.get("source_type")

        if source_type == "hdx_hapi":
            block = f"""
EVIDENCE {evidence_id}
SOURCE TYPE: HDX HAPI
DOCUMENT / RESOURCE: {item.get("document_title")}
ORGANIZATION: {item.get("organization")}
REFERENCE PERIOD: {item.get("reference_period")}
GEOGRAPHY LEVEL: {item.get("admin_level")}
ADMIN1: {item.get("admin1_name")}
ADMIN2: {item.get("admin2_name")}
SECTOR: {item.get("sector")}
POPULATION STATUS: {item.get("population_status")}
VALUE: {item.get("value")}
UNIT: {item.get("unit")}
PASSAGE:
{item.get("passage")}

SYSTEM INTERPRETATION CONSTRAINT:
{item.get("aggregation_note")}
"""
        else:
            block = f"""
EVIDENCE {evidence_id}
SOURCE TYPE: KNOWLEDGE BASE DOCUMENT
DOCUMENT: {item.get("document_title")}
DOCUMENT TYPE: {item.get("document_type")}
ORGANIZATION: {item.get("organization")}
PUBLICATION DATE: {item.get("publication_date")}
PAGE: {item.get("page")}
SECTION: {item.get("section")}
PASSAGE:
{item.get("passage")}
"""

        evidence_blocks.append(block)

    formatted_evidence = "\n\n".join(evidence_blocks)

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    instructions = """
You are the Grounded Rewriter of the Mali Knowledge Hub.

You receive:

1. a user question
2. an analytical draft
3. a factchecker audit
4. an evidence ledger

Your task is to REPAIR the analytical draft.

You are not writing a new answer from scratch.

The factchecker audit has priority over the original draft.


============================================================
CORE RULE
============================================================

Repair claims by:

- narrowing them
- weakening them
- qualifying them
- splitting them
- removing unsupported portions

Do NOT repair a claim by inventing new substantive material.


============================================================
EVIDENCE BOUNDARY
============================================================

Use ONLY information already present in:

- the analytical draft
- the supplied evidence ledger

Do NOT introduce:

- new facts
- new examples
- new interventions
- new programmes
- new implementation actions
- new outcomes
- new causal pathways
- new actors
- new locations
- new statistics
- new recommendations

Even if they seem obvious or reasonable.

If a required fix cannot be made without introducing new
substance, remove or narrow the problematic claim instead.


============================================================
FACTCHECKER PRIORITY
============================================================

Apply every substantive required_fix in the audit.

For claims marked:

FULLY_SUPPORTED
→ preserve unless minor wording changes improve clarity.

PARTIALLY_SUPPORTED
→ keep only the supported portion.

UNSUPPORTED
→ remove or rewrite only if the evidence ledger directly supports
  a narrower version.

CITATION_MISMATCH
→ repair the citation relationship without inventing evidence.

REASONABLE_INFERENCE
→ preserve clearly as an inference.

INFERENCE_TOO_STRONG
→ weaken the claim to the strongest formulation directly allowed
  by the evidence and audit.


============================================================
EPISTEMIC LABELS
============================================================

Use only:

[FACT]
[SYNTHESIS]
[INFERENCE]

Definitions:

[FACT]
A statement directly supported by supplied evidence.

[SYNTHESIS]
A relationship or comparison across supported facts.

[INFERENCE]
A cautious analytical implication that goes beyond explicit
source wording but remains reasonable from the evidence.

Never disguise an inference as a fact.


============================================================
NO NEW EXAMPLES OVERRIDE
============================================================

THIS IS A HARD RULE.

Do not introduce an example that was not already present in the
analytical draft.

This includes examples of:

- interventions
- humanitarian activities
- development activities
- programmes
- implementation approaches
- institutional actions
- expected outcomes
- causal mechanisms

Do not add parenthetical examples to make a claim more concrete.

For example, if the draft says:

"addressing humanitarian needs may support longer-term
development priorities"

you may NOT rewrite this as:

"demining, food assistance and WASH programming may support safe
return and agricultural recovery"

unless all of those interventions and outcomes were already
explicitly present in the original analytical draft and supported
by evidence.

When in doubt, keep the claim abstract.


============================================================
NO NEW CAUSAL PATHWAYS OVERRIDE
============================================================

Do not manufacture a causal chain.

Do not introduce intermediate outcomes such as:

- safe return
- agricultural recovery
- restored livelihoods
- improved stability
- increased trust
- stronger institutions

unless they already appear in the original analytical draft and
are supported by the evidence ledger.

A thematic relationship between humanitarian needs and
development priorities does NOT establish a causal pathway.


============================================================
DIRECTNESS OVERRIDE
============================================================

If the draft uses language such as:

- directly relates
- directly corresponds
- directly aligns
- maps directly
- directly enables

and the evidence does not explicitly establish that direct
relationship:

rewrite using cautious language such as:

- relates to
- can be related to
- aligns with
- corresponds with
- has thematic overlap with

Do not preserve "directly" unless directly documented.


============================================================
NECESSITY OVERRIDE
============================================================

If the draft says:

- necessary
- required
- essential
- prerequisite
- must
- indispensable
- cannot occur without
- depends on

and the evidence does not establish necessity:

rewrite using cautious language such as:

- may support
- could contribute
- may facilitate
- may be relevant to
- could affect

Modifiers such as "likely necessary" do NOT make a necessity
claim acceptable.


============================================================
PRIORITY / RANKING RULE
============================================================

Do not infer ranking from selected structured observations.

Do not describe selected sectors, places or values as:

- main
- primary
- largest
- most important
- leading
- top priority

unless the supplied evidence explicitly establishes that ranking.

Selected examples remain selected examples.


============================================================
GEOGRAPHIC CONCENTRATION RULE
============================================================

Do not infer geographic concentration from selected records.

Do not use:

- concentrated in
- concentrated needs
- especially concentrated
- geographic concentration

unless the supplied source explicitly establishes concentration.

You may say that needs are:

- documented in
- observed in
- reported for
- substantial in

specific Admin2 locations.


============================================================
STRUCTURED DATA RULES
============================================================

Preserve exactly:

- geography
- administrative level
- sector
- population category
- population status
- unit
- reference period
- values

Do not silently harmonize different dimensions.

Do not sum Admin2 values into an Admin1 total unless the evidence
ledger explicitly provides that aggregate.

Do not treat selected sector-location records as a complete
regional sector ranking.


============================================================
SYSTEM-CONSTRAINT REWRITE OVERRIDE
============================================================

THIS IS A DETERMINISTIC REWRITE RULE.

If the draft contains a methodological claim derived from a
SYSTEM INTERPRETATION CONSTRAINT, split it from the
source-supported factual statement.

Never write:

"[FACT] The records are Admin2 observations and must not be
summed into an Admin1 total [E11]."

Instead write:

"[FACT] The supplied records are Admin2-level observations and
the supplied evidence does not provide an Admin1 aggregate
[E11, E12, E13]."

Then state the methodological consequence separately:

"[INFERENCE] This analysis therefore does not derive an Admin1
regional total by summing those observations."

The methodological sentence:

- must NOT claim that the source imposed the rule
- must NOT cite the system interpretation constraint as evidence
- must NOT be included inside the same [FACT] statement
- must be presented as the Knowledge Hub's analytical handling

If the original draft says:

"must not be summed"
or
"cannot be aggregated"

based only on a SYSTEM INTERPRETATION CONSTRAINT,

rewrite this as:

"The supplied evidence does not provide the aggregate."
+
"This analysis therefore does not derive one by summing the
available observations."


============================================================
CITATION RULES
============================================================

Every substantive claim must cite supporting evidence IDs.

Use ONLY evidence IDs that exist in the supplied ledger.

Evidence IDs must be written individually.

VALID:

[E01]

[E01, E03, E07]

[E11, E12, E13, E14, E15, E16, E17, E18]

INVALID:

[E01–E07]

[E11–E18]

[E19–E24]

Never create citation ranges.

Never compress sequential evidence IDs using:

- hyphens
- en dashes
- em dashes
- "through"
- "to"

Never infer intermediate evidence IDs.

If a claim relies on E11, E12, E13, E14, E15, E16, E17 and E18,
write all eight IDs explicitly.

Do not cite a SYSTEM INTERPRETATION CONSTRAINT as source evidence.


============================================================
CITATION PRESERVATION
============================================================

When repairing a claim, preserve the citations from the
analytical draft whenever they still support the repaired claim.

You may remove an evidence ID only when the corresponding
substantive portion of the claim has been removed.

You may add an evidence ID only when that exact evidence item is
already present in the supplied ledger and directly supports the
repaired claim.

Never replace an explicit list with a range.


============================================================
ABSENCE OF EVIDENCE
============================================================

Be precise when describing what is absent.

Prefer:

"The supplied evidence does not provide an Admin1 aggregate."

Do not write:

"No Admin1 aggregate exists."

unless the evidence establishes global absence rather than absence
from the supplied evidence.


============================================================
TIME AND FRESHNESS
============================================================

Distinguish:

- reference period
- publication date
- source update date
- retrieval date

Do not describe a record as "current" merely because it was
recently retrieved or updated.


============================================================
STYLE
============================================================

Produce a coherent analytical draft.

Be:

- concise
- precise
- policy-relevant
- evidence-grounded

Avoid:

- generic introductions
- rhetorical flourish
- unnecessary repetition
- new examples added for readability

This is still an internal verified draft, not the final
presentation layer.


============================================================
FINAL SELF-CHECK
============================================================

Before returning the answer, silently verify:

1. Did I apply every required substantive fix?
2. Did I introduce any new fact?
3. Did I introduce any new example?
4. Did I introduce any new intervention?
5. Did I introduce any new outcome?
6. Did I introduce any new causal pathway?
7. Did I strengthen any claim beyond the evidence?
8. Did I infer ranking or concentration?
9. Did I create a citation range?
10. Did I treat a system constraint as source evidence?
11. Did I manufacture an Admin1 aggregate?
12. Does every cited evidence ID exist in the ledger?

If yes to 2–11, repair before returning.


============================================================
OUTPUT
============================================================

Return only the repaired analytical draft.

Use [FACT], [SYNTHESIS] and [INFERENCE] labels.

Do not include commentary about the rewriting process.
"""

    user_input = f"""
USER QUESTION:

{question}


ORIGINAL ANALYSIS DRAFT:

{analysis_draft}


FACTCHECKER AUDIT:

{audit}


EVIDENCE LEDGER:

{formatted_evidence}
"""

    response = openai_client.responses.create(
        model=model,
        instructions=instructions,
        input=user_input
    )

    return response.output_text.strip()


print("✅ Grounded Rewriter v2.3 loaded")

✅ Grounded Rewriter v2.3 loaded


In [ ]:
grounded_v23_test = run_grounded_rewriter(
    test_question_v2,
    result_v2["debug"]["analysis_draft"],
    result_v2["debug"]["audit"],
    result_v2["evidence"]
)

print(grounded_v23_test)

[FACT] Humanitarian needs in Mopti (admin2 observations, reference period 2025-01-01 to 2025-12-08): intersectoral “people in need” figures reported by the Global HPC/HNO show large affected populations at the district (admin2) level in Mopti region — Konna: 363,384 people (intersectoral) [E12]; Korientze: 293,052 people (intersectoral) [E13]; Mopti (admin2): 156,243 people (intersectoral) [E14]; Tenenkou: 180,964 people (intersectoral) [E16]; Djenne: 152,595 people (intersectoral) [E11]; Sofara: 133,605 people (intersectoral) [E15]; Youwarou: 113,636 people (intersectoral) [E18]; Togueré‑Coumbé: 57,333 people (intersectoral) [E17].

[FACT] The supplied records are Admin2-level observations and the supplied evidence does not provide an Admin1 aggregate for Mopti [E11, E12, E13, E14, E15, E16, E17, E18].

[INFERENCE] This analysis therefore does not derive an Admin1 (Mopti regional) total by summing those Admin2 observations.

[FACT] Sector-specific humanitarian needs documented in Mopt

In [ ]:
print("=" * 70)
print("ORIGINAL ANALYSIS DRAFT")
print("=" * 70)

print(result_v2["debug"]["analysis_draft"])

print("\n" + "=" * 70)
print("FACTCHECKER AUDIT")
print("=" * 70)

print(result_v2["debug"]["audit"])

ORIGINAL ANALYSIS DRAFT
[FACT] Humanitarian needs in Mopti (admin2 observations, reference period 2025-01-01 to 2025-12-08): intersectoral “people in need” figures reported by the HPC/HNO show large affected populations at the district (admin2) level in Mopti region — Konna: 363,384 people (intersectoral) [E12]; Korientze: 293,052 people (intersectoral) [E13]; Mopti (admin2): 156,243 people (intersectoral) [E14]; Tenenkou: 180,964 people (intersectoral) [E16]; Djenne: 152,595 people (intersectoral) [E11]; Sofara: 133,605 people (intersectoral) [E15]; Youwarou: 113,636 people (intersectoral) [E18]; Togueré‑Coumbé: 57,333 people (intersectoral) [E17]. Each of these is an admin2-level HNO observation and must not be aggregated to an admin1 total unless the source provides such an aggregate [E11–E18].

[FACT] Sector-specific humanitarian needs documented in Mopti admin2s include very large food-security and WASH needs in Konna (Food Security: 249,711 people; WASH: 171,294 people) [E20, E23

## 14 — Final Editor

Improves clarity, structure and concision.

**Hard rule:**
The final editor may alter language but may not alter epistemic status.

In [ ]:
# ============================================================
# 14 — FINAL EDITOR v3
# ============================================================

import re


def lock_citations(text):
    """
    Replace evidence citations like [E01] or [E01, E03, E07]
    with immutable placeholders before sending text to the LLM.
    """
    citation_pattern = r"\[[^\]]*E\d{2}[^\]]*\]"

    citations = []

    def replacer(match):
        placeholder = f"<CITATION_{len(citations)+1}>"
        citations.append({
            "placeholder": placeholder,
            "citation": match.group(0)
        })
        return placeholder

    locked_text = re.sub(citation_pattern, replacer, text)

    return locked_text, citations


def restore_citations(text, citations):
    """
    Restore original citation strings exactly.
    """
    restored = text

    for item in citations:
        restored = restored.replace(
            item["placeholder"],
            item["citation"]
        )

    return restored


def validate_citation_placeholders(text, citations):
    """
    Validate that every original citation placeholder survives
    exactly once and that no unknown citation placeholders appear.
    """

    expected = [item["placeholder"] for item in citations]

    found = re.findall(r"<CITATION_\d+>", text)

    missing = [
        placeholder
        for placeholder in expected
        if found.count(placeholder) == 0
    ]

    duplicated = [
        placeholder
        for placeholder in expected
        if found.count(placeholder) > 1
    ]

    unknown = [
        placeholder
        for placeholder in found
        if placeholder not in expected
    ]

    valid = (
        len(missing) == 0
        and len(duplicated) == 0
        and len(unknown) == 0
        and len(found) == len(expected)
    )

    return {
        "valid": valid,
        "missing": missing,
        "duplicated": duplicated,
        "unknown": unknown,
        "expected_count": len(expected),
        "found_count": len(found)
    }


def strip_epistemic_tags(text):
    """
    Safe fallback presentation:
    remove internal epistemic labels without changing substance.
    """
    cleaned = re.sub(
        r"\[(FACT|SYNTHESIS|INFERENCE)\]\s*",
        "",
        text
    )

    return cleaned.strip()


def run_final_editor(
    question,
    grounded_draft,
    model="gpt-5-mini",
    return_debug=False
):
    """
    Conservative final presentation layer.

    v3:
    - citation locking
    - conservative formatting only
    - no fresh synthesis
    - deterministic citation validation
    - safe fallback to grounded draft
    """

    # --------------------------------------------------------
    # 1. LOCK CITATIONS
    # --------------------------------------------------------

    locked_draft, citations = lock_citations(
        grounded_draft
    )

    # --------------------------------------------------------
    # 2. CONSERVATIVE EDITOR PROMPT
    # --------------------------------------------------------

    instructions = """
You are the presentation editor of the Mali Knowledge Hub.

You receive a VERIFIED analytical answer.

The reasoning is finished.

Your task is NOT to analyse, summarize, interpret, synthesize,
or improve the substance.

Your task is only to make the verified answer easier to read.


============================================================
ALLOWED EDITS
============================================================

You may:

- remove internal labels:
  [FACT]
  [SYNTHESIS]
  [INFERENCE]

- add short descriptive headings
- split long paragraphs
- turn existing lists into bullets
- improve punctuation
- fix grammar
- simplify awkward sentence structure
- remove literal or near-literal repetition
- improve spacing and readability


============================================================
FORBIDDEN EDITS
============================================================

Do NOT:

- write a new summary
- write a new key takeaway
- create a new conclusion
- create new synthesis
- combine separate claims into a new claim
- add examples
- add interventions
- add outcomes
- add recommendations
- add causal pathways
- add interpretations
- strengthen or weaken modality
- change geographic scope
- change temporal scope
- change numbers
- infer rankings
- infer concentration
- infer importance
- infer causality
- add substantive wording not already present


============================================================
SUBSTANCE PRESERVATION
============================================================

Every substantive statement in your output must already exist
in the supplied verified draft.

You may reformat an existing statement.

You may lightly copy-edit an existing statement.

You may NOT create a new substantive sentence by combining
information from different sentences.

When uncertain, preserve the original wording.


============================================================
MODALITY
============================================================

Preserve epistemic strength exactly.

Examples:

"may support"
must remain uncertain.

"could constrain"
must remain uncertain.

"can be related to"
must not become
"directly relates to".

"selected observations"
must not become
"main", "primary", "largest", or "concentrated".


============================================================
CITATION PLACEHOLDERS
============================================================

Citation placeholders such as:

<CITATION_1>
<CITATION_2>

are immutable tokens.

You MUST preserve every citation placeholder EXACTLY ONCE.

Do not:

- delete them
- duplicate them
- rename them
- merge them
- move one to support a different substantive claim
- create new citation placeholders

Keep each placeholder attached to the same substantive claim.


============================================================
STRUCTURE
============================================================

You may add headings only when they describe material already
present in the verified draft.

Good examples:

Humanitarian picture
Development priorities
How the needs relate
Analytical implications
Evidence limitation

Do not create:

Key takeaway
Executive summary
Conclusion

unless such a section already exists in the supplied draft.


============================================================
NO EXTRA MATERIAL
============================================================

Return only the edited answer.

Do not add:

- explanations of your editing
- follow-up questions
- offers
- sign-offs
- commentary


============================================================
FINAL CHECK
============================================================

Before returning the answer, silently verify:

1. Every substantive claim existed in the input.
2. No new example was added.
3. No new implication was added.
4. No modality was strengthened.
5. No geography or time scope changed.
6. Every citation placeholder appears exactly once.
7. No citation placeholder was altered.

If you are unsure whether an edit is safe, keep the original
sentence.
"""

    user_input = f"""
USER QUESTION:

{question}


VERIFIED GROUNDED DRAFT:

{locked_draft}
"""

    response = openai_client.responses.create(
        model=model,
        instructions=instructions,
        input=user_input
    )

    edited_locked = response.output_text.strip()

    # --------------------------------------------------------
    # 3. VALIDATE CITATION PLACEHOLDERS
    # --------------------------------------------------------

    citation_validation = validate_citation_placeholders(
        edited_locked,
        citations
    )

    # --------------------------------------------------------
    # 4. FALLBACK IF VALIDATION FAILS
    # --------------------------------------------------------

    if not citation_validation["valid"]:

        fallback_answer = strip_epistemic_tags(
            grounded_draft
        )

        if return_debug:
            return {
                "answer": fallback_answer,
                "used_fallback": True,
                "citation_validation": citation_validation,
                "editor_output_locked": edited_locked
            }

        return fallback_answer

    # --------------------------------------------------------
    # 5. RESTORE EXACT ORIGINAL CITATIONS
    # --------------------------------------------------------

    final_answer = restore_citations(
        edited_locked,
        citations
    )

    # --------------------------------------------------------
    # 6. RETURN
    # --------------------------------------------------------

    if return_debug:
        return {
            "answer": final_answer,
            "used_fallback": False,
            "citation_validation": citation_validation,
            "editor_output_locked": edited_locked
        }

    return final_answer


print("✅ Final Editor v3 loaded")

✅ Final Editor v3 loaded


In [ ]:
final_editor_v3_test = run_final_editor(
    test_question_v2,
    result_v2["debug"]["grounded_draft"],
    return_debug=True
)

print(final_editor_v3_test["answer"])

print("\n" + "=" * 70)
print("FINAL EDITOR DIAGNOSTICS")
print("=" * 70)

print(
    "Used fallback:",
    final_editor_v3_test["used_fallback"]
)

print(
    "Citation validation:",
    final_editor_v3_test["citation_validation"]
)

Humanitarian needs (admin2 observations)
- Intersectoral “people in need” figures in the Global HPC HNO 2025 record large affected populations at the district (admin2) level in Mopti: Konna: 363,384 people; Korientze: 293,052 people; Mopti (admin2): 156,243 people; Tenenkou: 180,964 people; Djenne: 152,595 people; Sofara: 133,605 people; Youwarou: 113,636 people; Togueré‑Coumbé: 57,333 people. [E11–E18]

Scope note
- The supplied HNO records are Admin2‑level observations and the supplied evidence does not provide an Admin1 (Mopti) aggregate. This analysis therefore does not derive an Admin1 regional total by summing the available Admin2 observations.

Sector‑specific needs
- Sector‑specific HNO records for Mopti show very large food‑security and WASH needs in Konna (Food Security: 249,711; WASH: 171,294) and concentrated protection‑related needs in Korientze across multiple sectors (Protection: 291,933; Mine Action: 247,184; Health: 203,421; Gender‑based violence: 166,910). [E19–E24]



### 🧪 14.1 — Final Cached Pipeline Test

In [ ]:
# ============================================================
# FINAL CACHED PIPELINE TEST
# Analysis v2.1 → Factchecker v2.2 already completed
# Now: Grounded Rewriter v2.3 → Final Editor v3
# ============================================================

grounded_final_test = run_grounded_rewriter(
    test_question_v2,
    analysis_v21_test,
    audit_v22_test,
    result_v2["evidence"]
)

print("=" * 70)
print("GROUNDED DRAFT")
print("=" * 70)
print(grounded_final_test)


final_core_test = run_final_editor(
    test_question_v2,
    grounded_final_test,
    return_debug=True
)

print("\n" + "=" * 70)
print("FINAL ANSWER")
print("=" * 70)
print(final_core_test["answer"])

print("\n" + "=" * 70)
print("FINAL EDITOR DIAGNOSTICS")
print("=" * 70)
print("Used fallback:", final_core_test["used_fallback"])
print(
    "Citation validation:",
    final_core_test["citation_validation"]
)

GROUNDED DRAFT
[FACT] The supplied evidence records intersectoral "people in need" figures at Admin2 level across Mopti: Djenne (152,595) [E11]; Konna (363,384) [E12]; Korientze (293,052) [E13]; Mopti (156,243) [E14]; Sofara (133,605) [E15]; Tenenkou (180,964) [E16]; Togueré‑Coumbé (57,333) [E17]; Youwarou (113,636) [E18]. Each observation is an Admin2‑level "people in need" figure for 2025 (reference period recorded in the items) [E11, E12, E13, E14, E15, E16, E17, E18].

[FACT] The evidence also documents sector‑specific humanitarian needs at Admin2 level: Protection in Korientze (291,933) [E19]; Food Security in Konna (249,711) [E20]; Mine Action in Korientze (247,184) [E21]; Health in Korientze (203,421) [E22]; Water, Sanitation and Hygiene in Konna (171,294) [E23]; Gender‑Based Violence in Korientze (166,910) [E24].

[SYNTHESIS] The recorded humanitarian sectors in Mopti (protection, mine action, food security, health, WASH, GBV) have thematic overlap with development issues empha

## 15 — End-to-End Pipeline

Question → Research → Evidence → Analysis → Factcheck → Rewrite → Edit → Final Answer

In [ ]:
# ============================================================
# 15.1 — END-TO-END PIPELINE
# ============================================================

def ask_mali(
    question,
    top_k_per_query=10,
    model="gpt-5-mini",
    return_debug=False
):
    """
    Runs the complete Mali Knowledge Hub v0.2 pipeline.

    Question
    → Research Planner
    → Research Agent
    → Evidence Ledger
    → Analysis Agent
    → Factchecker
    → Grounded Rewriter
    → Final Editor
    """

    # --------------------------------------------------------
    # 1. Research
    # --------------------------------------------------------

    research = run_research_agent(
        question,
        top_k_per_query=top_k_per_query,
        model=model
    )

    # --------------------------------------------------------
    # 2. Evidence Ledger
    # --------------------------------------------------------

    ledger = build_evidence_ledger(
        research
    )

    # --------------------------------------------------------
    # 3. Analysis
    # --------------------------------------------------------

    analysis_draft = run_analysis_agent(
        question,
        ledger,
        model=model
    )

    # --------------------------------------------------------
    # 4. Factcheck
    # --------------------------------------------------------

    audit = run_factchecker(
        question,
        analysis_draft,
        ledger,
        model=model
    )

    # --------------------------------------------------------
    # 5. Grounded Rewrite
    # --------------------------------------------------------

    grounded_draft = run_grounded_rewriter(
        question,
        analysis_draft,
        audit,
        ledger,
        model=model
    )

    # --------------------------------------------------------
    # 6. Final Edit
    # --------------------------------------------------------

    final_answer = run_final_editor(
        question,
        grounded_draft,
        model=model
    )

    # --------------------------------------------------------
    # 7. Return result
    # --------------------------------------------------------

    result = {
        "question": question,
        "answer": final_answer,
        "evidence": ledger
    }

    if return_debug:
        result["debug"] = {
            "research": research,
            "analysis_draft": analysis_draft,
            "audit": audit,
            "grounded_draft": grounded_draft
        }

    return result


print("✅ End-to-End Pipeline loaded")

✅ End-to-End Pipeline loaded


In [ ]:
result_2 = ask_mali(
    """
    What are the main priorities for strengthening local governance
    and territorial development in Mali, and what concrete investments
    or interventions are envisaged to implement them?
    """,
    return_debug=True
)

print(result_2["answer"])

Summary (high level)  
- Strengthening local governance and territorial development in Mali centers on (a) deepening and operationalizing decentralisation and subsidiarity, (b) reinforcing local fiscal and administrative capacities, (c) improving public service delivery and local infrastructure, (d) promoting territorial equity and development of competitive/local economic poles, and (e) linking local governance with peace, security and state leadership. [E02, E03, E06, E07, E09, E10]

1) Stated policy priorities (documented)  
- The national strategy explicitly makes decentralisation, transfer of competences and resources to subnational levels a guiding principle and requires that national orientations be declined at regional and local levels (subsidiarity). [E02]  
- The SNEDD and Vision Mali 2063 identify improving political, administrative and judicial governance, modernising the public administration, and consolidating the role of Collectivités Territoriales as pillars for peace, 

## 16 — External Connectors


### 16.1 — OCHA / HDX HAPI

In [ ]:
# ============================================================
# 16.1 — OCHA / HDX HAPI CONNECTOR
# ============================================================

import requests
from google.colab import userdata


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

HDX_HAPI_APP_IDENTIFIER = userdata.get("HDX_HAPI_APP_IDENTIFIER")

if not HDX_HAPI_APP_IDENTIFIER:
    raise ValueError(
        "❌ HDX_HAPI_APP_IDENTIFIER not found in Colab Secrets."
    )

HDX_HAPI_BASE_URL = "https://hapi.humdata.org/api/v2"


# ------------------------------------------------------------
# Generic HAPI client
# ------------------------------------------------------------

def hapi_get(
    endpoint,
    params=None,
    limit=1000,
    offset=0,
    timeout=30
):
    """
    Generic GET request to HDX HAPI.

    Parameters
    ----------
    endpoint : str
        API endpoint after /api/v2/
        Example:
        "metadata/data-availability"

    params : dict
        Optional HAPI query parameters.

    limit : int
        Maximum number of records to request.

    offset : int
        Pagination offset.

    Returns
    -------
    dict
        Parsed HAPI JSON response.
    """

    if params is None:
        params = {}

    query_params = {
        **params,
        "limit": limit,
        "offset": offset,
        "output_format": "json"
    }

    headers = {
        "X-HDX-HAPI-APP-IDENTIFIER": HDX_HAPI_APP_IDENTIFIER
    }

    url = f"{HDX_HAPI_BASE_URL}/{endpoint.lstrip('/')}"

    response = requests.get(
        url,
        params=query_params,
        headers=headers,
        timeout=timeout
    )

    response.raise_for_status()

    return response.json()


# ------------------------------------------------------------
# Connection test
# ------------------------------------------------------------

def test_hapi_connection():
    """
    Tests whether HDX HAPI is reachable and the app identifier works.
    """

    result = hapi_get(
        "util/version",
        limit=10
    )

    print("✅ HDX HAPI connected")
    print(result)

    return result


print("✅ HDX HAPI connector loaded")

✅ HDX HAPI connector loaded


In [ ]:
hapi_version = test_hapi_connection()

✅ HDX HAPI connected
{'api_version': '0.9.14', 'hapi_sqlalchemy_schema_version': '0.9.17'}


In [ ]:
# ============================================================
# 16.1 TEST — DATA AVAILABILITY FOR MALI
# ============================================================

mali_data_availability = hapi_get(
    "metadata/data-availability",
    params={
        "location_code": "MLI"
    },
    limit=1000
)

print("Top-level keys:", mali_data_availability.keys())

data = mali_data_availability.get("data", [])

print("Available records:", len(data))

for row in data:
    print(
        row.get("category"),
        "|",
        row.get("subcategory"),
        "| Admin1:",
        row.get("admin1_name"),
        "| Admin2:",
        row.get("admin2_name"),
        "| HAPI updated:",
        row.get("hapi_updated_date")
    )

Top-level keys: dict_keys(['data'])
Available records: 463
affected-people | humanitarian-needs | Admin1: Gao | Admin2: Ansongo | HAPI updated: 2026-08-27T02:44:17.868054
affected-people | humanitarian-needs | Admin1: Gao | Admin2: Bourem | HAPI updated: 2026-08-27T02:44:17.868054
affected-people | humanitarian-needs | Admin1: Gao | Admin2: Gao | HAPI updated: 2026-08-27T02:44:17.868054
affected-people | humanitarian-needs | Admin1: Gao | Admin2: None | HAPI updated: 2026-08-27T02:44:17.868054
affected-people | humanitarian-needs | Admin1: Kayes | Admin2: Ambididi | HAPI updated: 2026-08-27T02:44:17.868054
affected-people | humanitarian-needs | Admin1: Kayes | Admin2: Aourou | HAPI updated: 2026-08-27T02:44:17.868054
affected-people | humanitarian-needs | Admin1: Kayes | Admin2: Bafoulebe | HAPI updated: 2026-08-27T02:44:17.868054
affected-people | humanitarian-needs | Admin1: Kayes | Admin2: Diamou | HAPI updated: 2026-08-27T02:44:17.868054
affected-people | humanitarian-needs | Admin

#### 16.1.2 — Humanitarian Needs Test: Mopti


In [ ]:
# ============================================================
# 16.1.2 — HUMANITARIAN NEEDS TEST: MOPTI
# ============================================================

mopti_needs = hapi_get(
    "affected-people/humanitarian-needs",
    params={
        "location_code": "MLI",
        "admin1_name": "Mopti"
    },
    limit=100
)

print("Top-level keys:", mopti_needs.keys())

rows = mopti_needs.get("data", [])

print("Records returned:", len(rows))

if rows:
    print("\nFIRST RECORD:\n")
    print(json.dumps(rows[0], indent=2, ensure_ascii=False))

Top-level keys: dict_keys(['data'])
Records returned: 100

FIRST RECORD:

{
  "location_code": "MLI",
  "location_name": "Mali",
  "admin1_code": "ML05",
  "admin1_name": "Mopti",
  "admin2_code": "ML0501",
  "admin2_name": "Mopti",
  "admin_level": 2,
  "resource_hdx_id": "8e3931a5-452b-4583-9d02-2247a34e397b",
  "sector_code": "HEA",
  "category": "Adult - female",
  "population_status": "INN",
  "population": 32539,
  "reference_period_start": "2024-01-01T00:00:00",
  "reference_period_end": "2024-12-31T23:59:59",
  "sector_name": "Health"
}


#### 16.1.3 — Inspect Humanitarian Needs Schema

In [ ]:
# ============================================================
# 16.1.3 — INSPECT HUMANITARIAN NEEDS SCHEMA
# ============================================================

rows = mopti_needs.get("data", [])

population_statuses = sorted({
    row.get("population_status")
    for row in rows
    if row.get("population_status") is not None
})

sectors = sorted({
    row.get("sector_name")
    for row in rows
    if row.get("sector_name") is not None
})

categories = sorted({
    row.get("category")
    for row in rows
    if row.get("category") is not None
})

admin2s = sorted({
    row.get("admin2_name")
    for row in rows
    if row.get("admin2_name") is not None
})

print("POPULATION STATUSES")
print(population_statuses)

print("\nSECTORS")
for item in sectors:
    print("-", item)

print("\nCATEGORIES")
for item in categories:
    print("-", item)

print("\nADMIN2 LOCATIONS")
for item in admin2s:
    print("-", item)


POPULATION STATUSES
['INN', 'TGT', 'all']

SECTORS
- Cash programming
- Education
- Emergency Shelter and NFI
- Health
- Intersectoral
- Water Sanitation Hygiene

CATEGORIES
- Adult - female
- Adult - female - autres
- Adult - female - pdi
- Adult - female - rapatries
- Adult - male
- Adult - male - autres
- Adult - male - pdi
- Adult - male - rapatries
- Adult_female
- Adult_male
- Autres
- Children  -  male - pdi
- Children  - female - pdi
- Children - female
- Children - female - autres
- Children - female - rapatries
- Children - male
- Children - male - autres
- Children - male -rapatries
- Children_female
- Children_male
- Elderly - female

ADMIN2 LOCATIONS
- Mopti


#### 16.1.4 — Full Humanitarian Needs Retrieval: Mopti

In [ ]:
# ============================================================
# 16.1.4 — FULL HUMANITARIAN NEEDS RETRIEVAL: MOPTI
# ============================================================

mopti_needs_full = hapi_get(
    "affected-people/humanitarian-needs",
    params={
        "location_code": "MLI",
        "admin1_name": "Mopti"
    },
    limit=10000
)

rows_full = mopti_needs_full.get("data", [])

print("Total records:", len(rows_full))

print("\nADMIN2 LOCATIONS")
for name in sorted({
    row.get("admin2_name")
    for row in rows_full
    if row.get("admin2_name")
}):
    print("-", name)

print("\nREFERENCE PERIODS")
for period in sorted({
    (
        row.get("reference_period_start"),
        row.get("reference_period_end")
    )
    for row in rows_full
}):
    print("-", period)

print("\nPOPULATION STATUSES")
for status in sorted({
    row.get("population_status")
    for row in rows_full
    if row.get("population_status")
}):
    print("-", status)

print("\nSECTORS")
for sector in sorted({
    row.get("sector_name")
    for row in rows_full
    if row.get("sector_name")
}):
    print("-", sector)

Total records: 1482

ADMIN2 LOCATIONS
- Djenne
- Konna
- Korientze
- Mopti
- Sofara
- Tenenkou
- Togueré-Coumbé
- Youwarou

REFERENCE PERIODS
- ('2024-01-01T00:00:00', '2024-12-31T23:59:59')
- ('2025-01-01T00:00:00', '2025-12-08T23:59:59')

POPULATION STATUSES
- INN
- TGT
- all

SECTORS
- Cash programming
- Child Protection
- Education
- Emergency Shelter and NFI
- Food Security
- Gender Based Violence
- Health
- Housing, Land and Property
- Intersectoral
- Mine Action
- Multi-sector (unspecified)
- Nutrition
- Protection
- Water Sanitation Hygiene


#### 16.1.5 — Normalize HAPI records into Evidence

In [ ]:
# ============================================================
# 16.1.5 — HAPI → EVIDENCE NORMALIZER
# ============================================================

from datetime import datetime, timezone
import hashlib
import json


HAPI_POPULATION_STATUS_LABELS = {
    "INN": "People in need",
    "TGT": "People targeted",
    "all": "All / source aggregate"
}


def _stable_hapi_evidence_id(row, endpoint):
    """
    Create a stable evidence ID from the substantive record fields.
    """

    identity = {
        "endpoint": endpoint,
        "resource_hdx_id": row.get("resource_hdx_id"),
        "location_code": row.get("location_code"),
        "admin1_code": row.get("admin1_code"),
        "admin2_code": row.get("admin2_code"),
        "sector_code": row.get("sector_code"),
        "category": row.get("category"),
        "population_status": row.get("population_status"),
        "population": row.get("population"),
        "reference_period_start": row.get("reference_period_start"),
        "reference_period_end": row.get("reference_period_end")
    }

    raw = json.dumps(
        identity,
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")

    digest = hashlib.sha256(raw).hexdigest()[:12]

    return f"HAPI-{digest}"


def normalize_humanitarian_need_record(row):
    """
    Convert one HDX HAPI humanitarian-needs record
    into a source-independent evidence object.
    """

    endpoint = "affected-people/humanitarian-needs"

    status_raw = row.get("population_status")

    status_label = HAPI_POPULATION_STATUS_LABELS.get(
        status_raw,
        status_raw
    )

    admin1 = row.get("admin1_name")
    admin2 = row.get("admin2_name")

    if admin2:
        location_text = f"{admin2}, {admin1}, Mali"
    elif admin1:
        location_text = f"{admin1}, Mali"
    else:
        location_text = "Mali"

    sector = row.get("sector_name") or "Unspecified sector"
    category = row.get("category") or "Unspecified population category"
    population = row.get("population")

    period_start = row.get("reference_period_start")
    period_end = row.get("reference_period_end")

    passage = (
        f"{status_label}: {population:,} people; "
        f"sector: {sector}; "
        f"population category: {category}; "
        f"location: {location_text}; "
        f"reference period: {period_start} to {period_end}."
        if isinstance(population, (int, float))
        else
        f"{status_label}; "
        f"sector: {sector}; "
        f"population category: {category}; "
        f"location: {location_text}; "
        f"reference period: {period_start} to {period_end}."
    )

    return {
        # ----------------------------------------------------
        # Common Evidence Ledger fields
        # ----------------------------------------------------

        "evidence_id": _stable_hapi_evidence_id(row, endpoint),

        "source_type": "hdx_hapi",

        "document_id": None,
        "document_title": "HDX HAPI — Humanitarian Needs",
        "document_type": "structured_humanitarian_data",

        "organization": "OCHA / HDX",

        "publication_date": None,
        "version": None,

        "page": None,
        "section": sector,
        "chunk_id": None,

        "similarity": None,
        "retrieved_for": None,

        "passage": passage,

        "evidence_type": "structured_data",
        "supports": None,

        # ----------------------------------------------------
        # HAPI provenance
        # ----------------------------------------------------

        "source_endpoint": endpoint,

        "resource_hdx_id": row.get("resource_hdx_id"),

        "retrieved_at": datetime.now(
            timezone.utc
        ).isoformat(),

        # ----------------------------------------------------
        # Temporal provenance
        # ----------------------------------------------------

        "reference_period_start": period_start,
        "reference_period_end": period_end,

        # ----------------------------------------------------
        # Geography
        # ----------------------------------------------------

        "location_code": row.get("location_code"),
        "location_name": row.get("location_name"),

        "admin_level": row.get("admin_level"),

        "admin1_code": row.get("admin1_code"),
        "admin1_name": admin1,

        "admin2_code": row.get("admin2_code"),
        "admin2_name": admin2,

        # ----------------------------------------------------
        # Indicator
        # ----------------------------------------------------

        "sector_code": row.get("sector_code"),
        "sector_name": sector,

        "population_category": category,

        "population_status": status_raw,
        "population_status_label": status_label,

        "value": population,
        "unit": "people",

        # ----------------------------------------------------
        # Original source record
        # ----------------------------------------------------

        "raw_record": row
    }


def normalize_humanitarian_needs(rows):
    """
    Normalize multiple HAPI humanitarian-needs records.
    """

    return [
        normalize_humanitarian_need_record(row)
        for row in rows
    ]


print("✅ HAPI humanitarian-needs normalizer loaded")

✅ HAPI humanitarian-needs normalizer loaded


#### 16.1.6 — Resolve HDX Resource Metadata

In [ ]:
# ============================================================
# 16.1.6 — HDX RESOURCE METADATA
# ============================================================

def get_hapi_resource_metadata(resource_hdx_id):
    """
    Retrieve HDX HAPI metadata for one resource.
    """
    result = hapi_get(
        "metadata/resource",
        params={
            "resource_hdx_id": resource_hdx_id
        },
        limit=100
    )

    return result.get("data", [])

#### 16.1.7 — Enrich HAPI Evidence with Resource Metadata

In [ ]:
# ============================================================
# 16.1.7 — ENRICH HAPI EVIDENCE WITH RESOURCE METADATA
# ============================================================

def enrich_hapi_evidence(evidence_item, resource_metadata):
    """
    Add HDX resource/dataset/provider provenance
    to one normalized HAPI evidence object.
    """

    if not resource_metadata:
        return evidence_item

    meta = resource_metadata[0]

    return {
        **evidence_item,

        "source_platform": "HDX HAPI",

        "resource_name": meta.get("name"),
        "resource_format": meta.get("format"),
        "resource_update_date": meta.get("update_date"),
        "resource_download_url": meta.get("download_url"),
        "resource_hdx_link": meta.get("hdx_link"),

        "dataset_hdx_id": meta.get("dataset_hdx_id"),
        "dataset_title": meta.get("dataset_hdx_title"),
        "dataset_hdx_link": meta.get("dataset_hdx_link"),

        "provider_name": meta.get("dataset_hdx_provider_name"),
        "provider_stub": meta.get("dataset_hdx_provider_stub"),
        "provider_hdx_link": meta.get("provider_hdx_link"),

        "hapi_updated_date": meta.get("hapi_updated_date")
    }

#### 16.1.8 — Batch Metadata Cache and Enrichment

In [ ]:
# ============================================================
# 16.1.8 — BATCH METADATA CACHE + ENRICHMENT
# ============================================================

HAPI_RESOURCE_METADATA_CACHE = {}


def get_cached_resource_metadata(resource_hdx_id):
    """
    Fetch resource metadata once per unique HDX resource ID
    and reuse it afterwards.
    """

    if not resource_hdx_id:
        return []

    if resource_hdx_id in HAPI_RESOURCE_METADATA_CACHE:
        return HAPI_RESOURCE_METADATA_CACHE[resource_hdx_id]

    metadata = get_hapi_resource_metadata(resource_hdx_id)

    HAPI_RESOURCE_METADATA_CACHE[resource_hdx_id] = metadata

    return metadata


def enrich_hapi_evidence_batch(evidence_items):
    """
    Enrich a list of normalized HAPI evidence records efficiently.

    Metadata is retrieved only once per unique resource_hdx_id.
    """

    enriched_items = []

    resource_ids = {
        item.get("resource_hdx_id")
        for item in evidence_items
        if item.get("resource_hdx_id")
    }

    print(f"Unique HDX resources: {len(resource_ids)}")

    for resource_id in resource_ids:
        get_cached_resource_metadata(resource_id)

    for item in evidence_items:

        resource_id = item.get("resource_hdx_id")

        metadata = (
            HAPI_RESOURCE_METADATA_CACHE.get(resource_id, [])
            if resource_id
            else []
        )

        enriched_items.append(
            enrich_hapi_evidence(
                item,
                metadata
            )
        )

    return enriched_items


print("✅ HAPI metadata cache and batch enrichment loaded")

✅ HAPI metadata cache and batch enrichment loaded


#### 16.1.9 — HAPI Research Tool

In [ ]:
# ============================================================
# 16.1.9 — HAPI HUMANITARIAN NEEDS RESEARCH TOOL
# ============================================================

def research_humanitarian_needs(
    admin1_name=None,
    admin2_name=None,
    sector_name=None,
    population_status="INN",
    latest_only=True,
    limit=10000
):
    """
    Retrieve and normalize humanitarian-needs evidence from HDX HAPI.

    Default behavior:
    - Mali only
    - People in Need (INN)
    - latest available reference period only
    """

    params = {
        "location_code": "MLI"
    }

    if admin1_name:
        params["admin1_name"] = admin1_name

    if admin2_name:
        params["admin2_name"] = admin2_name

    if sector_name:
        params["sector_name"] = sector_name

    if population_status:
        params["population_status"] = population_status

    # --------------------------------------------------------
    # Retrieve raw HAPI records
    # --------------------------------------------------------

    response = hapi_get(
        "affected-people/humanitarian-needs",
        params=params,
        limit=limit
    )

    rows = response.get("data", [])

    if not rows:
        return []

    # --------------------------------------------------------
    # Keep latest reference period if requested
    # --------------------------------------------------------

    if latest_only:

        available_periods = [
            row.get("reference_period_end")
            for row in rows
            if row.get("reference_period_end")
        ]

        if available_periods:

            latest_period_end = max(available_periods)

            rows = [
                row
                for row in rows
                if row.get("reference_period_end") == latest_period_end
            ]

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    evidence = normalize_humanitarian_needs(rows)

    # --------------------------------------------------------
    # Enrich with cached HDX metadata
    # --------------------------------------------------------

    evidence = enrich_hapi_evidence_batch(evidence)

    return evidence


print("✅ HAPI humanitarian-needs research tool loaded")

✅ HAPI humanitarian-needs research tool loaded


In [ ]:
# ============================================================
# 16.1.9 TEST — LATEST MOPTI NEEDS
# ============================================================

latest_mopti_needs = research_humanitarian_needs(
    admin1_name="Mopti"
)

print("Evidence records:", len(latest_mopti_needs))

periods = sorted({
    (
        e.get("reference_period_start"),
        e.get("reference_period_end")
    )
    for e in latest_mopti_needs
})

print("\nREFERENCE PERIODS")
for period in periods:
    print("-", period)

print("\nRESOURCES")
for resource in sorted({
    e.get("resource_name")
    for e in latest_mopti_needs
    if e.get("resource_name")
}):
    print("-", resource)

print("\nEXAMPLE")
if latest_mopti_needs:
    print(latest_mopti_needs[0]["passage"])

Unique HDX resources: 1
Evidence records: 320

REFERENCE PERIODS
- ('2025-01-01T00:00:00', '2025-12-08T23:59:59')

RESOURCES
- Global HPC HNO 2025

EXAMPLE
People in need: 295 people; sector: Education; population category: Adult_female; location: Mopti, Mopti, Mali; reference period: 2025-01-01T00:00:00 to 2025-12-08T23:59:59.


### 16.2 — Research Source Router

In [ ]:
# ============================================================
# 16.2 — RESEARCH SOURCE ROUTER
# ============================================================

def plan_research_v2(
    question,
    model="gpt-5-mini"
):
    """
    Plan research across available source types.

    Current sources:
    - knowledge_base
    - hapi_humanitarian_needs
    """

    instructions = """
You are the Research Planner for the Mali Knowledge Hub.

Your task is NOT to answer the user's question.

Your task is to decide what evidence needs to be retrieved and
which available source is best suited to each retrieval task.

AVAILABLE SOURCES

1. knowledge_base

Use for:
- national strategies
- policy documents
- development priorities
- governance frameworks
- institutional arrangements
- programme documents
- narrative reports
- qualitative analysis
- documented interventions

Required field:
- query


2. hapi_humanitarian_needs

Use for structured humanitarian needs data such as:
- people in need
- people targeted
- humanitarian needs by sector
- humanitarian needs by location
- humanitarian needs by demographic group

Possible fields:
- admin1_name
- admin2_name
- sector_name
- population_status
- latest_only

Default population_status:
INN

Default latest_only:
true


PLANNING RULES

Use the MINIMUM number of retrieval tasks necessary.

Do not create retrieval tasks for:
- comparison
- synthesis
- implications
- tensions
- gaps
- recommendations
- alignment

Those should be derived later from retrieved evidence.

Choose structured HAPI data when the question asks for
quantitative humanitarian-needs information.

Choose the knowledge base when the evidence is primarily
documentary, strategic, institutional or qualitative.

A single user question may require both sources.

Do not invent:
- administrative locations
- sectors
- dates
- organizations
- filters

Only include filters explicitly stated or clearly implied by
the user's question.

Preserve Mali place names exactly as written when possible.

OUTPUT

Return valid JSON only.

Schema:

{
  "research_tasks": [
    {
      "label": "short human-readable label",
      "source": "knowledge_base",
      "query": "semantic retrieval query"
    },
    {
      "label": "short human-readable label",
      "source": "hapi_humanitarian_needs",
      "admin1_name": null,
      "admin2_name": null,
      "sector_name": null,
      "population_status": "INN",
      "latest_only": true
    }
  ]
}

Only include fields relevant to that source.
"""

    response = openai_client.responses.create(
        model=model,
        instructions=instructions,
        input=question
    )

    text = response.output_text.strip()

    text = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    return json.loads(text)


print("✅ Research Source Router loaded")

✅ Research Source Router loaded


In [ ]:
# ============================================================
# 16.2 TEST — HYBRID RESEARCH PLAN
# ============================================================

router_test_question = (
    "What are the main humanitarian needs in Mopti "
    "and how do they relate to Mali's longer-term "
    "development priorities?"
)

router_plan = plan_research_v2(
    router_test_question
)

print(
    json.dumps(
        router_plan,
        indent=2,
        ensure_ascii=False
    )
)

{
  "research_tasks": [
    {
      "label": "Humanitarian needs in Mopti (HAPI data)",
      "source": "hapi_humanitarian_needs",
      "admin1_name": "Mopti",
      "admin2_name": null,
      "sector_name": null,
      "population_status": "INN",
      "latest_only": true
    },
    {
      "label": "Mali national long-term development priorities and strategies",
      "source": "knowledge_base",
      "query": "Mali long-term development priorities national development plan national strategy poverty reduction resilience economic development agriculture education health infrastructure decentralization governance and social protection strategic documents"
    }
  ]
}


#### 16.2.1 — Multi-Source Research Agent

In [ ]:
# ============================================================
# 16.2.1 — MULTI-SOURCE RESEARCH AGENT
# ============================================================

def run_research_agent_v2(
    question,
    top_k_per_query=10,
    model="gpt-5-mini"
):
    """
    Execute a multi-source research plan.

    Supported sources:
    - knowledge_base
    - hapi_humanitarian_needs
    """

    research_plan = plan_research_v2(
        question,
        model=model
    )

    research_tasks = research_plan.get(
        "research_tasks",
        []
    )

    retrieved = []

    # --------------------------------------------------------
    # Execute each research task
    # --------------------------------------------------------

    for task_index, task in enumerate(
        research_tasks,
        start=1
    ):

        task_id = f"Q{task_index}"
        source = task.get("source")

        # ====================================================
        # KNOWLEDGE BASE
        # ====================================================

        if source == "knowledge_base":

            results = search_knowledge_base(
                task["query"],
                match_count=top_k_per_query
            )

            for rank, r in enumerate(
                results,
                start=1
            ):

                retrieved.append({
                    **r,

                    "source_type": "knowledge_base_document",

                    "research_query_id": task_id,
                    "research_query_label": task.get("label"),
                    "research_query": task.get("query"),

                    "retrieval_rank": rank
                })

        # ====================================================
        # HDX HAPI — HUMANITARIAN NEEDS
        # ====================================================

        elif source == "hapi_humanitarian_needs":

            results = research_humanitarian_needs(
                admin1_name=task.get("admin1_name"),
                admin2_name=task.get("admin2_name"),
                sector_name=task.get("sector_name"),
                population_status=task.get(
                    "population_status",
                    "INN"
                ),
                latest_only=task.get(
                    "latest_only",
                    True
                )
            )

            for rank, r in enumerate(
                results,
                start=1
            ):

                retrieved.append({
                    **r,

                    "research_query_id": task_id,
                    "research_query_label": task.get("label"),

                    "research_query": {
                        "source": source,
                        "admin1_name": task.get("admin1_name"),
                        "admin2_name": task.get("admin2_name"),
                        "sector_name": task.get("sector_name"),
                        "population_status": task.get(
                            "population_status",
                            "INN"
                        ),
                        "latest_only": task.get(
                            "latest_only",
                            True
                        )
                    },

                    "retrieval_rank": rank
                })

        else:
            raise ValueError(
                f"Unsupported research source: {source}"
            )

    # --------------------------------------------------------
    # Deduplicate
    # --------------------------------------------------------

    unique = {}

    for r in retrieved:

        source_type = r.get("source_type")

        if source_type == "hdx_hapi":

            key = r.get("evidence_id")

        else:

            key = (
                r.get("id")
                or r.get("chunk_id")
                or (
                    r.get("document_id"),
                    r.get("page_number"),
                    r.get("content")
                )
            )

        retrieval_context = {
            "query_id": r.get("research_query_id"),
            "label": r.get("research_query_label"),
            "query": r.get("research_query"),
            "rank": r.get("retrieval_rank")
        }

        if key not in unique:

            unique[key] = {
                **r,
                "retrieved_for": [
                    retrieval_context
                ]
            }

        else:

            unique[key]["retrieved_for"].append(
                retrieval_context
            )

    unique_results = list(
        unique.values()
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    source_counts = {}

    for item in unique_results:

        source = item.get(
            "source_type",
            "unknown"
        )

        source_counts[source] = (
            source_counts.get(source, 0) + 1
        )

    return {
        "question": question,
        "research_plan": research_plan,
        "results": unique_results,
        "raw_retrieval_count": len(retrieved),
        "unique_evidence_count": len(unique_results),
        "source_counts": source_counts
    }


print("✅ Multi-source Research Agent loaded")

✅ Multi-source Research Agent loaded


In [ ]:
# ============================================================
# 16.2.1 TEST — HYBRID RESEARCH EXECUTION
# ============================================================

hybrid_research = run_research_agent_v2(
    router_test_question,
    top_k_per_query=10
)

print("Raw retrieval count:",
      hybrid_research["raw_retrieval_count"])

print("Unique evidence count:",
      hybrid_research["unique_evidence_count"])

print("\nSOURCE COUNTS")

for source, count in hybrid_research[
    "source_counts"
].items():
    print("-", source, ":", count)

print("\nFIRST 5 EVIDENCE ITEMS")

for item in hybrid_research[
    "results"
][:5]:

    print("\n---")

    print(
        "Source:",
        item.get("source_type")
    )

    print(
        "Task:",
        item.get("research_query_label")
    )

    print(
        "Evidence:",
        (
            item.get("passage")
            or item.get("content")
            or ""
        )[:500]
    )

Unique HDX resources: 1
Raw retrieval count: 330
Unique evidence count: 330

SOURCE COUNTS
- knowledge_base_document : 10
- hdx_hapi : 320

FIRST 5 EVIDENCE ITEMS

---
Source: knowledge_base_document
Task: Mali national development strategies and long-term priorities
Evidence: 335. Les hypothèses spécifiques sur l’économie nationale portent : Au Plan Politique - la poursuite de la stabilité sociopolitique et sécuritaire du pays à travers un maillage du territoire par les forces armées de défense et de sécurité, la mise en œuvre des recommandations et résolutions des Assises Nationales de la Refondation et du Dialogue Inter-Maliens pour la Paix et la Réconciliation Nationale ; - la mise en œuvre du Pacte de croissance et de stabilité sociale ; - la reconstruction ou l’

---
Source: knowledge_base_document
Task: Mali national development strategies and long-term priorities
Evidence: 86. Dans le domaine de la santé, les défis prioritaires identifiés concernent : la mortalité maternelle et

#### 16.2.2 — Structured Evidence Reducer

In [ ]:
# ============================================================
# 16.2.2 — STRUCTURED EVIDENCE REDUCER
# ============================================================

def reduce_hapi_humanitarian_evidence(
    evidence_items,
    max_per_sector=2,
    max_total=30
):
    """
    Reduce large HAPI humanitarian-needs result sets
    into a compact evidence set suitable for analysis.

    Priority:
    1. latest data already selected upstream
    2. intersectoral / aggregate categories
    3. sector-level aggregates
    4. suppress demographic slices unless needed
    """

    if not evidence_items:
        return []

    # --------------------------------------------------------
    # Helpers
    # --------------------------------------------------------

    aggregate_categories = {
        "all",
        "All",
        "TOTAL",
        "Total",
        "total",
        "Autres"
    }

    def is_aggregate_category(item):
        category = str(
            item.get("population_category") or ""
        ).strip()

        return category in aggregate_categories

    def priority_score(item):

        score = 0

        sector = (
            item.get("sector_name") or ""
        ).lower()

        category = (
            item.get("population_category") or ""
        ).lower()

        admin_level = item.get("admin_level")

        # Prefer intersectoral overview
        if "intersectoral" in sector:
            score += 100

        # Prefer aggregate population categories
        if is_aggregate_category(item):
            score += 50

        # Penalize obvious demographic slices
        demographic_terms = [
            "adult",
            "children",
            "elderly",
            "female",
            "male",
            "pdi",
            "rapatr",
            "autres"
        ]

        if any(
            term in category
            for term in demographic_terms
        ):
            score -= 20

        # Prefer broader geography
        if admin_level == 1:
            score += 20
        elif admin_level == 2:
            score += 10

        return score

    # --------------------------------------------------------
    # Group by sector
    # --------------------------------------------------------

    sectors = {}

    for item in evidence_items:

        sector = (
            item.get("sector_name")
            or "Unspecified"
        )

        sectors.setdefault(
            sector,
            []
        ).append(item)

    selected = []

    # --------------------------------------------------------
    # Select strongest records per sector
    # --------------------------------------------------------

    for sector, items in sectors.items():

        ranked = sorted(
            items,
            key=lambda x: (
                priority_score(x),
                x.get("value") or 0
            ),
            reverse=True
        )

        selected.extend(
            ranked[:max_per_sector]
        )

    # --------------------------------------------------------
    # Final ranking
    # --------------------------------------------------------

    selected = sorted(
        selected,
        key=lambda x: (
            priority_score(x),
            x.get("value") or 0
        ),
        reverse=True
    )

    return selected[:max_total]


print("✅ Structured HAPI evidence reducer loaded")

✅ Structured HAPI evidence reducer loaded


In [ ]:
# ============================================================
# 16.2.2 — STRUCTURED EVIDENCE REDUCER v2
# ============================================================

def reduce_hapi_humanitarian_evidence(
    evidence_items,
    max_sector_examples=6
):
    """
    Reduce broad humanitarian-needs results while preserving
    their actual geographic structure.

    For broad geographic questions:
    1. Prefer category='total'
    2. Keep ALL intersectoral totals across available locations
    3. Keep a small number of high-value sector/location records
       as examples of sectoral patterns
    4. Never manufacture geographic aggregates
    """

    if not evidence_items:
        return []

    # --------------------------------------------------------
    # Keep aggregate population records only
    # --------------------------------------------------------

    total_rows = [
        item
        for item in evidence_items
        if str(
            item.get("population_category") or ""
        ).strip().lower() == "total"
    ]

    # Fallback if the source contains no total-category records
    if not total_rows:
        return evidence_items[:30]

    # --------------------------------------------------------
    # 1. Preserve complete intersectoral geographic coverage
    # --------------------------------------------------------

    intersectoral = [
        item
        for item in total_rows
        if item.get("sector_name") == "Intersectoral"
    ]

    intersectoral = sorted(
        intersectoral,
        key=lambda x: (
            x.get("admin1_name") or "",
            x.get("admin2_name") or ""
        )
    )

    # --------------------------------------------------------
    # 2. Build sector-level candidate pool
    # --------------------------------------------------------

    sector_rows = [
        item
        for item in total_rows
        if item.get("sector_name") != "Intersectoral"
    ]

    # Rank individual Admin2 observations by documented value.
    # IMPORTANT:
    # These are NOT regional totals and must not be summed
    # or interpreted as such.
    sector_rows = sorted(
        sector_rows,
        key=lambda x: x.get("value") or 0,
        reverse=True
    )

    # --------------------------------------------------------
    # 3. Select sector examples while maximizing diversity
    # --------------------------------------------------------

    sector_examples = []
    used_sectors = set()

    for item in sector_rows:

        sector = item.get("sector_name")

        if sector not in used_sectors:
            sector_examples.append(item)
            used_sectors.add(sector)

        if len(sector_examples) >= max_sector_examples:
            break

    # --------------------------------------------------------
    # 4. Combine
    # --------------------------------------------------------

    selected = (
        intersectoral
        + sector_examples
    )

    return selected


print("✅ Structured HAPI evidence reducer v2 loaded")


# ============================================================
# 16.2.2 TEST — REDUCE MOPTI HAPI EVIDENCE
# ============================================================

hapi_items = [
    item
    for item in hybrid_research["results"]
    if item.get("source_type") == "hdx_hapi"
]

reduced_hapi = reduce_hapi_humanitarian_evidence(
    hapi_items
)

print("\nBefore:", len(hapi_items))
print("After:", len(reduced_hapi))

print("\nSELECTED HAPI EVIDENCE")

for item in reduced_hapi:
    print(
        "-",
        item.get("sector_name"),
        "|",
        item.get("population_category"),
        "|",
        item.get("admin2_name"),
        "|",
        item.get("value")
    )

✅ Structured HAPI evidence reducer v2 loaded

Before: 320
After: 14

SELECTED HAPI EVIDENCE
- Intersectoral | total | Djenne | 152595
- Intersectoral | total | Konna | 363384
- Intersectoral | total | Korientze | 293052
- Intersectoral | total | Mopti | 156243
- Intersectoral | total | Sofara | 133605
- Intersectoral | total | Tenenkou | 180964
- Intersectoral | total | Togueré-Coumbé | 57333
- Intersectoral | total | Youwarou | 113636
- Protection | total | Korientze | 291933
- Food Security | total | Konna | 249711
- Mine Action | total | Korientze | 247184
- Health | total | Korientze | 203421
- Water Sanitation Hygiene | total | Konna | 171294
- Gender Based Violence | total | Korientze | 166910


#### 16.2.3 — Inspect Geographic Aggregation

In [ ]:
# ============================================================
# 16.2.3 — INSPECT HAPI AGGREGATE GEOGRAPHY
# ============================================================

total_rows = [
    item
    for item in hapi_items
    if str(
        item.get("population_category") or ""
    ).strip().lower() == "total"
]

print("TOTAL-CATEGORY RECORDS:", len(total_rows))


# ------------------------------------------------------------
# Admin-level distribution
# ------------------------------------------------------------

admin_level_counts = {}

for item in total_rows:
    level = item.get("admin_level")
    admin_level_counts[level] = (
        admin_level_counts.get(level, 0) + 1
    )

print("\nADMIN LEVELS")

for level, count in sorted(
    admin_level_counts.items(),
    key=lambda x: str(x[0])
):
    print("-", level, ":", count)


# ------------------------------------------------------------
# Admin1-level records
# ------------------------------------------------------------

admin1_aggregate_rows = [
    item
    for item in total_rows
    if item.get("admin_level") == 1
]

print(
    "\nADMIN1 AGGREGATE RECORDS:",
    len(admin1_aggregate_rows)
)

for item in admin1_aggregate_rows[:30]:
    print(
        "-",
        item.get("sector_name"),
        "|",
        item.get("admin1_name"),
        "|",
        item.get("value")
    )


# ------------------------------------------------------------
# Intersectoral Admin2 totals
# ------------------------------------------------------------

intersectoral_admin2 = [
    item
    for item in total_rows
    if (
        item.get("sector_name") == "Intersectoral"
        and item.get("admin_level") == 2
    )
]

print("\nINTERSECTORAL ADMIN2 TOTALS")

for item in sorted(
    intersectoral_admin2,
    key=lambda x: x.get("value") or 0,
    reverse=True
):
    print(
        "-",
        item.get("admin2_name"),
        "|",
        item.get("value")
    )


# ------------------------------------------------------------
# Number of total-category records by sector
# ------------------------------------------------------------

sector_counts = {}

for item in total_rows:
    sector = item.get("sector_name")
    sector_counts[sector] = (
        sector_counts.get(sector, 0) + 1
    )

print("\nTOTAL RECORDS BY SECTOR")

for sector, count in sorted(
    sector_counts.items()
):
    print("-", sector, ":", count)

TOTAL-CATEGORY RECORDS: 90

ADMIN LEVELS
- 2 : 90

ADMIN1 AGGREGATE RECORDS: 0

INTERSECTORAL ADMIN2 TOTALS
- Konna | 363384
- Korientze | 293052
- Tenenkou | 180964
- Mopti | 156243
- Djenne | 152595
- Sofara | 133605
- Youwarou | 113636
- Togueré-Coumbé | 57333

TOTAL RECORDS BY SECTOR
- Child Protection : 8
- Education : 8
- Emergency Shelter and NFI : 8
- Food Security : 8
- Gender Based Violence : 8
- Health : 8
- Housing, Land and Property : 8
- Intersectoral : 8
- Mine Action : 6
- Multi-sector (unspecified) : 1
- Nutrition : 3
- Protection : 8
- Water Sanitation Hygiene : 8


#### 16.2.4 — Unified Multi-Source Evidence Ledger

In [ ]:
# ============================================================
# 16.2.4 — UNIFIED MULTI-SOURCE EVIDENCE LEDGER
# ============================================================

def build_multisource_evidence_ledger(
    research_output,
    reduced_hapi_items
):
    """
    Build one Evidence Ledger from:
    - document retrieval results
    - reduced HDX HAPI evidence

    Evidence IDs are reassigned centrally so the downstream
    agents receive one unified evidence namespace.
    """

    ledger = []

    # --------------------------------------------------------
    # 1. Knowledge-base evidence
    # --------------------------------------------------------

    document_items = [
        item
        for item in research_output["results"]
        if item.get("source_type") == "knowledge_base_document"
    ]

    for item in document_items:

        ledger.append({
            "source_type": "knowledge_base_document",

            "document_id": item.get("document_id"),
            "document_title": item.get("document_title"),
            "document_type": item.get("document_type"),

            "organization": item.get("organization"),
            "publication_date": item.get("publication_date"),
            "version": item.get("version"),

            "page": item.get("page_number"),
            "section": item.get("section_title"),

            "chunk_id": item.get("chunk_id") or item.get("id"),

            "similarity": item.get("similarity"),
            "retrieved_for": item.get("retrieved_for"),

            "passage": item.get("content"),

            "evidence_type": "document_passage",

            "supports": None
        })

    # --------------------------------------------------------
    # 2. HAPI evidence
    # --------------------------------------------------------

    for item in reduced_hapi_items:

        hapi_item = {
            **item,

            # Remove old HAPI-specific evidence ID.
            # We assign one unified Evidence Ledger ID below.
            "evidence_id": None,

            # Explicit interpretation guardrail
            "aggregation_note": (
                "This is an Admin2-level observation. "
                "It must not be summed with other Admin2 records "
                "or represented as an Admin1 regional total unless "
                "the source explicitly provides such an aggregate."
            )
        }

        ledger.append(hapi_item)

    # --------------------------------------------------------
    # 3. Assign unified evidence IDs
    # --------------------------------------------------------

    for index, item in enumerate(
        ledger,
        start=1
    ):
        item["evidence_id"] = f"E{index:02d}"

    return ledger


print("✅ Unified multi-source Evidence Ledger loaded")

✅ Unified multi-source Evidence Ledger loaded


In [ ]:
# ============================================================
# 16.2.4 TEST — BUILD HYBRID LEDGER
# ============================================================

hybrid_ledger = build_multisource_evidence_ledger(
    hybrid_research,
    reduced_hapi
)

print("Total evidence items:", len(hybrid_ledger))

source_counts = {}

for item in hybrid_ledger:
    source = item.get("source_type")
    source_counts[source] = source_counts.get(source, 0) + 1

print("\nSOURCE COUNTS")

for source, count in source_counts.items():
    print("-", source, ":", count)

print("\nEVIDENCE INDEX")

for item in hybrid_ledger:
    print(
        item["evidence_id"],
        "|",
        item.get("source_type"),
        "|",
        (
            item.get("document_title")
            or item.get("resource_name")
        ),
        "|",
        (
            item.get("page")
            or item.get("admin2_name")
        )
    )

Total evidence items: 24

SOURCE COUNTS
- knowledge_base_document : 10
- hdx_hapi : 14

EVIDENCE INDEX
E01 | knowledge_base_document | Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024-2033) | 64
E02 | knowledge_base_document | Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024-2033) | 19
E03 | knowledge_base_document | Vision Mali 2063 — Mali Kura Ɲɛtaasira ka bɛn san 2063 ma | 11
E04 | knowledge_base_document | Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024-2033) | 18
E05 | knowledge_base_document | Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024-2033) | 16
E06 | knowledge_base_document | Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024-2033) | 7
E07 | knowledge_base_document | Stratégie Nationale pour l’Émergence et le Développement Durable (SNEDD 2024-2033) | 23
E08 | knowledge_base_document | Vision Mali 2063 — Mali Kura Ɲɛtaasira ka bɛn san 2

#### 16.2.5 — First Multi-Source Analysis Test

Test whether the Analysis Agent can correctly synthesize documentary evidence and structured HDX HAPI data while preserving geographic scope, temporal provenance and epistemic status.

In [ ]:
# ============================================================
# 16.2.5 — FIRST MULTI-SOURCE ANALYSIS TEST
# ============================================================

hybrid_analysis = run_analysis_agent(
    router_test_question,
    hybrid_ledger
)

print(hybrid_analysis)

[FACT] Humanitarian needs in Mopti are documented at the Admin2 level in the HNO 2025 (reference period 2025-01-01 to 2025-12-08). Multiple Mopti districts (Admin2) are reported with large numbers of "people in need" in the HNO 2025 records (examples: Konna 363,384 intersectoral; Korientze 293,052 intersectoral; Mopti (town) 156,243 intersectoral; Tenenkou 180,964 intersectoral; Djenne 152,595 intersectoral) [E12, E13, E14, E16, E11]. Note: each HNO observation is an Admin2-level record and must not be summed to create an Admin1 total unless the source explicitly provides such an aggregate [E11].

[FACT] Sector-specific humanitarian needs recorded (selected Admin2 examples):
- Food security: Konna — 249,711 people in need (Food Security, 2025) [E20].  
- Protection: Korientze — 291,933 people in need (Protection, 2025) [E19].  
- Mine action: Korientze — 247,184 people in need (Mine Action, 2025) [E21].  
- Health: Korientze — 203,421 people in need (Health, 2025) [E22].  
- Water, San

#### 16.2.6 — Multi-Source Factcheck Test

In [ ]:
# ============================================================
# 16.2.6 — MULTI-SOURCE FACTCHECK TEST
# ============================================================

hybrid_audit = run_factchecker(
    router_test_question,
    hybrid_analysis,
    hybrid_ledger
)

print(
    json.dumps(
        hybrid_audit,
        indent=2,
        ensure_ascii=False
    )
)

{
  "overall_assessment": "The draft accurately cites HNO 2025 Admin2 data and Mali strategic documents for facts, but two synthesis/inference statements overstate the evidence: one asserts prioritized humanitarian 'primacy' in the documented Admin2s and another claims directness/necessity between humanitarian response and national development goals. Both require wording changes to match what the ledger supports.",
  "claims": [
    {
      "claim": "Humanitarian needs in Mopti are documented at the Admin2 level in the HNO 2025 (reference period 2025-01-01 to 2025-12-08). Multiple Mopti districts (Admin2) are reported with large numbers of \"people in need\" in the HNO 2025 records (examples: Konna 363,384 intersectoral; Korientze 293,052 intersectoral; Mopti (town) 156,243 intersectoral; Tenenkou 180,964 intersectoral; Djenne 152,595 intersectoral) [E12, E13, E14, E16, E11]. Note: each HNO observation is an Admin2-level record and must not be summed to create an Admin1 total unless th

In [ ]:
print(router_test_question)
print()
print(hybrid_analysis[:500])

What are the main humanitarian needs in Mopti and how do they relate to Mali's longer-term development priorities?

[FACT] Humanitarian needs in Mopti are documented at the Admin2 level in the HNO 2025 (reference period 2025-01-01 to 2025-12-08). Multiple Mopti districts (Admin2) are reported with large numbers of "people in need" in the HNO 2025 records (examples: Konna 363,384 intersectoral; Korientze 293,052 intersectoral; Mopti (town) 156,243 intersectoral; Tenenkou 180,964 intersectoral; Djenne 152,595 intersectoral) [E12, E13, E14, E16, E11]. Note: each HNO observation is an Admin2-level record and must


#### 16.2.7 — Multi-Source Grounded Rewrite Test

Test whether the Grounded Rewriter correctly repairs issues identified
by the multi-source Factchecker without introducing new claims or
altering supported evidence.

In [ ]:
hybrid_grounded_draft = run_grounded_rewriter(
    router_test_question,
    hybrid_analysis,
    hybrid_audit,
    hybrid_ledger
)

print(hybrid_grounded_draft)

[FACT] The HNO 2025 records humanitarian needs in Mopti at the Admin2 level (reference period 2025-01-01 to 2025-12-08). Multiple Mopti districts are recorded with large intersectoral "people in need" values: Konna 363,384; Korientze 293,052; Mopti (town) 156,243; Tenenkou 180,964; Djenne 152,595 [E12, E13, E14, E16, E11].

[FACT] Selected sector-specific Admin2 observations in the HNO 2025 include: food security — Konna 249,711 people [E20]; protection — Korientze 291,933 people [E19]; mine action — Korientze 247,184 people [E21]; health — Korientze 203,421 people [E22]; WASH — Konna 171,294 people [E23]; and GBV — Korientze 166,910 people [E24].

[FACT] Mali’s long‑term strategy documents (Vision Mali 2063 and SNEDD 2024–2033) identify development priorities that include: restoring security and social cohesion; addressing food insecurity and strengthening agriculture (LOA/PDA); improving health (maternal/infant mortality, HR and governance gaps); expanding access to water and sanitat

#### 16.2.8 — Multi-Source Final Editor Test

Test whether the Final Editor converts the verified multi-source
draft into a concise, user-facing answer without changing factual
content, citations, modality or epistemic strength.

In [ ]:
hybrid_final_answer = run_final_editor(
    router_test_question,
    hybrid_grounded_draft
)

print(hybrid_final_answer)

Summary answer
- Main humanitarian needs in Mopti are concentrated in specific districts and span protection (including mine action and GBV), food security, health, and WASH. The documented needs in these sectors correspond with several long‑term development priorities set out in Vision Mali 2063 and SNEDD 2024–2033 and therefore may be relevant to longer‑term transformation — while persistent unmet humanitarian needs could also complicate progress on those priorities [E12, E13, E14, E16, E11; E20, E19, E21, E22, E23, E24; E03, E01, E02, E04, E10, E08, E09, E07].

District‑level humanitarian picture (HNO 2025, Admin2 observations)
- Intersectoral “people in need” (selected Admin2 counts, reference period 2025-01-01 to 2025-12-08): Konna 363,384; Korientze 293,052; Mopti (town) 156,243; Tenenkou 180,964; Djenne 152,595 [E12, E13, E14, E16, E11].
- Selected sector caseloads (Admin2): 
  - Food security — Konna: 249,711 people [E20]. 
  - Protection — Korientze: 291,933 people [E19]. 
  -

#### 16.2.9 — End-to-End Multi-Source Pipeline

Integrate source routing, document retrieval, HDX HAPI retrieval,
structured evidence reduction, unified evidence construction,
analysis, factchecking, grounded rewriting and final editing into
a single user-facing pipeline.

In [ ]:
# ============================================================
# 16.2.9 — END-TO-END MULTI-SOURCE PIPELINE
# ============================================================

def ask_mali_v2(
    question,
    top_k_per_query=10,
    model="gpt-5-mini",
    max_sector_examples=6,
    return_debug=False
):
    """
    Mali Knowledge Hub v0.2 — multi-source end-to-end pipeline.

    Pipeline:
    Question
        ↓
    Research Source Router
        ↓
    Knowledge Base + HDX HAPI
        ↓
    Structured Evidence Reduction
        ↓
    Unified Evidence Ledger
        ↓
    Analysis Agent
        ↓
    Factchecker
        ↓
    Grounded Rewriter
        ↓
    Final Editor
        ↓
    User-facing answer
    """

    # --------------------------------------------------------
    # 1. MULTI-SOURCE RESEARCH
    # --------------------------------------------------------

    research = run_research_agent_v2(
        question,
        top_k_per_query=top_k_per_query,
        model=model
    )

    # --------------------------------------------------------
    # 2. EXTRACT + REDUCE HAPI EVIDENCE
    # --------------------------------------------------------

    hapi_items = [
        item
        for item in research["results"]
        if item.get("source_type") == "hdx_hapi"
    ]

    reduced_hapi = reduce_hapi_humanitarian_evidence(
        hapi_items,
        max_sector_examples=max_sector_examples
    )

    # --------------------------------------------------------
    # 3. BUILD UNIFIED MULTI-SOURCE EVIDENCE LEDGER
    # --------------------------------------------------------

    ledger = build_multisource_evidence_ledger(
        research,
        reduced_hapi
    )

    # --------------------------------------------------------
    # 4. ANALYSIS
    # --------------------------------------------------------

    analysis_draft = run_analysis_agent(
        question,
        ledger,
        model=model
    )

    # --------------------------------------------------------
    # 5. FACTCHECK
    # --------------------------------------------------------

    audit = run_factchecker(
        question,
        analysis_draft,
        ledger,
        model=model
    )

    # --------------------------------------------------------
    # 6. GROUNDED REWRITE
    # --------------------------------------------------------

    grounded_draft = run_grounded_rewriter(
        question,
        analysis_draft,
        audit,
        ledger,
        model=model
    )

    # --------------------------------------------------------
    # 7. FINAL EDIT
    # --------------------------------------------------------

    final_answer = run_final_editor(
        question,
        grounded_draft,
        model=model
    )

    # --------------------------------------------------------
    # 8. RESULT
    # --------------------------------------------------------

    result = {
        "question": question,
        "answer": final_answer,
        "evidence": ledger,
        "sources_used": research.get("source_counts", {}),
        "evidence_count": len(ledger)
    }

    # --------------------------------------------------------
    # 9. OPTIONAL DEBUG OUTPUT
    # --------------------------------------------------------

    if return_debug:

        audit_counts = {}

        for claim in audit.get("claims", []):
            status = claim.get("status", "UNKNOWN")
            audit_counts[status] = audit_counts.get(status, 0) + 1

        result["debug"] = {
            "research_plan": research.get("research_plan"),
            "research": research,
            "hapi_before_reduction": len(hapi_items),
            "hapi_after_reduction": len(reduced_hapi),
            "ledger": ledger,
            "analysis_draft": analysis_draft,
            "audit": audit,
            "audit_counts": audit_counts,
            "grounded_draft": grounded_draft
        }

    return result


print("🚀 ask_mali_v2() loaded")

🚀 ask_mali_v2() loaded


#### 16.2.10 — First End-to-End v0.2 Test

Run the complete multi-source pipeline from a raw user question
to the final evidence-grounded answer and inspect the pipeline
diagnostics.

In [ ]:
# ============================================================
# 16.2.10 — FIRST END-TO-END v0.2 TEST
# ============================================================

test_question_v2 = """
What are the main humanitarian needs in Mopti and how do they
relate to Mali's longer-term development priorities?
"""

result_v2 = ask_mali_v2(
    test_question_v2,
    return_debug=True
)

print("\n" + "=" * 70)
print("🇲🇱 MALI KNOWLEDGE HUB v0.2")
print("=" * 70)

print("\nANSWER\n")
print(result_v2["answer"])

print("\n" + "=" * 70)
print("PIPELINE DIAGNOSTICS")
print("=" * 70)

print("\nSources used:")
print(result_v2["sources_used"])

print(
    "\nHAPI evidence:",
    result_v2["debug"]["hapi_before_reduction"],
    "→",
    result_v2["debug"]["hapi_after_reduction"]
)

print(
    "Final evidence ledger:",
    result_v2["evidence_count"],
    "items"
)

print("\nFactcheck:")
for status, count in result_v2["debug"]["audit_counts"].items():
    print(f"  {status}: {count}")

Unique HDX resources: 1

🇲🇱 MALI KNOWLEDGE HUB v0.2

ANSWER

Key takeaway
- Mopti exhibits multi‑sector, location‑specific humanitarian needs in 2025 — large food‑security, WASH, health and protection (including mine‑action and GBV) caseloads concentrated in particular admin2s — and these sectoral needs can be related to Mali’s stated long‑term development priorities (peace/security, reconstruction, resilience, water management, health systems, agriculture) [E11–E24, E01, E03, E04, E06, E09].

Humanitarian picture (admin2 observations, HNO 2025)
- Intersectoral people‑in‑need (admin2): Konna 363,384; Korientze 293,052; Mopti (admin2) 156,243; Tenenkou 180,964; Djenne 152,595; Sofara 133,605; Youwarou 113,636; Togueré‑Coumbé 57,333 [E11–E18].  
  - Note: these are admin2 observations; the supplied evidence does not provide an Admin1 (Mopti) aggregate and this analysis does not sum them [E11–E18].
- Sector‑specific highlights:
  - Konna — Food security: 249,711; WASH: 171,294 [E19–E23].


### 16.3 — FONGIM Connector

Production connector for the FONGIM operational dataset.

Self-contained from a fresh runtime. Extracts, validates and synchronizes the complete FONGIM relational dataset with Supabase.

#### 16.3.1 — Self-contained FONGIM Connector

Production connector for the FONGIM operational dataset.

This section consolidates the configuration, Power BI query definitions, decoding, validation and synchronization logic required to reconstruct the FONGIM connector from a fresh runtime.

It must not depend on exploratory or discovery cells elsewhere in section 16.3.

In [ ]:
# 16.3.1 — FONGIM configuration & query definitions

import requests


# =========================================================
# Source configuration
# =========================================================

FONGIM_SOURCE_URL = "https://www.fongim.org/carte-interactive"

FONGIM_QUERYDATA_URL = (
    "https://wabi-south-africa-north-a-primary-api.analysis.windows.net/"
    "public/reports/querydata?synchronous=true"
)

FONGIM_MODEL_ID = 2218134

FONGIM_HEADERS = {
    "Content-Type": "application/json",
    "X-PowerBI-ResourceKey":
        "3dc6e1ae-910f-42a2-b41d-01aa141e6ee7"
}


# =========================================================
# Generic Power BI payload builder
# =========================================================

def build_fongim_payload(
    entity,
    source_alias,
    fields,
    count=10000
):
    """
    Build a standard FONGIM Power BI query payload.

    fields:
        list of (source_field, output_name)
    """

    selects = []

    for source_field, output_name in fields:

        selects.append({
            "Column": {
                "Expression": {
                    "SourceRef": {
                        "Source": source_alias
                    }
                },
                "Property": source_field
            },
            "Name": output_name
        })

    return {
        "version": "1.0.0",
        "queries": [
            {
                "Query": {
                    "Commands": [
                        {
                            "SemanticQueryDataShapeCommand": {
                                "Query": {
                                    "Version": 2,
                                    "From": [
                                        {
                                            "Name": source_alias,
                                            "Entity": entity,
                                            "Type": 0
                                        }
                                    ],
                                    "Select": selects
                                },
                                "Binding": {
                                    "Primary": {
                                        "Groupings": [
                                            {
                                                "Projections":
                                                    list(
                                                        range(
                                                            len(fields)
                                                        )
                                                    )
                                            }
                                        ]
                                    },
                                    "DataReduction": {
                                        "DataVolume": 6,
                                        "Primary": {
                                            "Window": {
                                                "Count": count
                                            }
                                        }
                                    },
                                    "Version": 1
                                },
                                "ExecutionMetricsKind": 1
                            }
                        }
                    ]
                },
                "QueryId": ""
            }
        ],
        "cancelQueries": [],
        "modelId": FONGIM_MODEL_ID
    }


# =========================================================
# Canonical source schemas
# =========================================================

FONGIM_PROJECT_FIELDS = [
    ("_index", "fongim_project_id"),
    ("Quel est le nom du projet ?", "project_name"),
    ("Date de début de la mise en œuvre", "start_date"),
    ("Date de fin de la mise en œuvre", "end_date"),
    ("Statut", "status"),
    ("Comment qualifiez vous votre projet ?", "project_type"),
    ("Le nombre total de bénéficiares du projet", "beneficiaries"),
    ("Quel est le bailleur ?", "donor"),
    ("Quel le montant du financement", "funding_amount_raw"),
    (
        "Quel(s) est/sont le(s) partenaire(s) du projet",
        "partners_raw"
    ),
    (
        "travaillez-vous avec des partenaires dans ce projet",
        "has_partners"
    )
]

FONGIM_LOCATION_FIELDS = [
    ("ID_projet", "fongim_project_id"),
    ("ID_zone", "fongim_zone_id"),
    ("Région", "region"),
    ("Cercle", "cercle"),
    ("Commune", "commune_raw")
]

FONGIM_SECTOR_FIELDS = [
    ("ID_projet", "fongim_project_id"),
    ("Secteurs", "sector"),
    ("Autres", "sector_other_raw")
]

FONGIM_ORGANIZATION_FIELDS = [
    ("id", "fongim_organization_id"),
    ("nom_org", "organization_name")
]

FONGIM_PARTNER_FIELDS = [
    ("ID_projet", "fongim_project_id"),
    ("Partenaire", "partner_name"),
    ("ID_ONG", "fongim_organization_id")
]

FONGIM_FUNDING_FIELDS = [
    ("id", "funding_id"),
    ("nom_org", "organization_name"),
    ("_index", "funding_index"),
    ("Projet ", "project_name_raw"),
    ("Type", "project_type"),
    ("Secteurs", "sector"),
    ("M2024", "funding_2024"),
    ("M2025", "funding_2025")
]


# =========================================================
# Build standard query payloads
# =========================================================

project_master_payload = build_fongim_payload(
    "Projets",
    "p",
    FONGIM_PROJECT_FIELDS,
    count=5000
)

location_payload = build_fongim_payload(
    "Zones",
    "z",
    FONGIM_LOCATION_FIELDS
)

sector_payload = build_fongim_payload(
    "Secteurs",
    "s",
    FONGIM_SECTOR_FIELDS
)

organization_payload = build_fongim_payload(
    "Organisations",
    "o",
    FONGIM_ORGANIZATION_FIELDS
)

partner_payload = build_fongim_payload(
    "Partenaires",
    "pa",
    FONGIM_PARTNER_FIELDS
)

funding_payload = build_fongim_payload(
    "funding",
    "f",
    FONGIM_FUNDING_FIELDS
)


# =========================================================
# Project → primary organization relationship
#
# This query spans two Power BI entities and therefore
# cannot use the single-entity builder above.
# =========================================================

project_org_payload = {
    "version": "1.0.0",
    "queries": [
        {
            "Query": {
                "Commands": [
                    {
                        "SemanticQueryDataShapeCommand": {
                            "Query": {
                                "Version": 2,
                                "From": [
                                    {
                                        "Name": "p",
                                        "Entity": "Projets",
                                        "Type": 0
                                    },
                                    {
                                        "Name": "o",
                                        "Entity": "Organisations",
                                        "Type": 0
                                    }
                                ],
                                "Select": [
                                    {
                                        "Column": {
                                            "Expression": {
                                                "SourceRef": {
                                                    "Source": "p"
                                                }
                                            },
                                            "Property": "_index"
                                        },
                                        "Name": "fongim_project_id"
                                    },
                                    {
                                        "Column": {
                                            "Expression": {
                                                "SourceRef": {
                                                    "Source": "p"
                                                }
                                            },
                                            "Property":
                                                "Quel est le nom du projet ?"
                                        },
                                        "Name": "project_name"
                                    },
                                    {
                                        "Column": {
                                            "Expression": {
                                                "SourceRef": {
                                                    "Source": "o"
                                                }
                                            },
                                            "Property": "id"
                                        },
                                        "Name": "fongim_organization_id"
                                    },
                                    {
                                        "Column": {
                                            "Expression": {
                                                "SourceRef": {
                                                    "Source": "o"
                                                }
                                            },
                                            "Property": "nom_org"
                                        },
                                        "Name": "organization_name"
                                    }
                                ]
                            },
                            "Binding": {
                                "Primary": {
                                    "Groupings": [
                                        {
                                            "Projections": [0, 1, 2, 3]
                                        }
                                    ]
                                },
                                "DataReduction": {
                                    "DataVolume": 6,
                                    "Primary": {
                                        "Window": {
                                            "Count": 5000
                                        }
                                    }
                                },
                                "Version": 1
                            },
                            "ExecutionMetricsKind": 1
                        }
                    }
                ]
            },
            "QueryId": ""
        }
    ],
    "cancelQueries": [],
    "modelId": FONGIM_MODEL_ID
}


print("✓ FONGIM configuration loaded")
print("✓ 7 canonical query payloads built")

✓ FONGIM configuration loaded
✓ 7 canonical query payloads built


#### 16.3.2 — Power BI Decoder

In [ ]:
import pandas as pd
from datetime import datetime, timezone


def decode_powerbi_rows(response_json):
    """
    Decode the primary Power BI DSR grouping returned by querydata.
    Supports:
    - dictionary encoded strings
    - repeated values via R bitmask
    - Power BI date timestamps
    """

    result = response_json["results"][0]["result"]["data"]

    # Column metadata
    descriptor = result["descriptor"]["Select"]
    column_names = [item["Name"] for item in descriptor]

    ds = result["dsr"]["DS"][0]
    rows = ds["PH"][0]["DM0"]
    value_dicts = ds.get("ValueDicts", {})

    # Schema information is normally supplied on the first row
    schema = rows[0].get("S", [])

    column_meta = []
    for i, col in enumerate(schema):
        column_meta.append({
            "name": col.get("N"),
            "type": col.get("T"),
            "dict": col.get("DN")
        })

    decoded_rows = []
    previous = [None] * len(column_names)

    for row in rows:
        values = [None] * len(column_names)

        repeat_mask = row.get("R", 0)
        raw_values = iter(row.get("C", []))

        for i in range(len(column_names)):

            # Reuse previous value if bit i is set
            if repeat_mask & (1 << i):
                values[i] = previous[i]
                continue

            try:
                value = next(raw_values)
            except StopIteration:
                value = None

            meta = column_meta[i] if i < len(column_meta) else {}

            # Dictionary encoded value
            dict_name = meta.get("dict")
            if dict_name and value is not None:
                dictionary = value_dicts.get(dict_name, [])
                if isinstance(value, int) and value < len(dictionary):
                    value = dictionary[value]

            # Power BI datetime → readable date
            if meta.get("type") == 7 and isinstance(value, (int, float)):
                value = datetime.fromtimestamp(
                    value / 1000,
                    tz=timezone.utc
                ).date().isoformat()

            values[i] = value

        decoded_rows.append(values)
        previous = values

    return pd.DataFrame(decoded_rows, columns=column_names)

#### 16.3.3 — Extract & Validate

In [ ]:
# 16.3.3 — Extract & validate FONGIM source

from datetime import datetime, timezone
import uuid


def run_fongim_sync_extract():
    """
    Pull the current FONGIM operational dataset from the public
    Power BI semantic model and return validated relational tables.

    No Supabase writes are performed here.
    """

    sync_started_at = datetime.now(timezone.utc)
    sync_run_id = str(uuid.uuid4())

    print("=" * 70)
    print("FONGIM SYNC EXTRACT")
    print("=" * 70)
    print("Sync run:", sync_run_id)
    print("Started:", sync_started_at.isoformat())

    # ---------------------------------------------------------
    # Helper
    # ---------------------------------------------------------

    def execute_payload(payload, label):
        response = requests.post(
            FONGIM_QUERYDATA_URL,
            headers=FONGIM_HEADERS,
            json=payload,
            timeout=30
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"{label} failed with HTTP {response.status_code}: "
                f"{response.text[:500]}"
            )

        df = decode_powerbi_rows(response.json())

        print(f"✓ {label}: {len(df)} rows")

        return df

    # ---------------------------------------------------------
    # Extract current source state
    # ---------------------------------------------------------

    projects = execute_payload(
        project_master_payload,
        "Projects"
    )

    locations = execute_payload(
        location_payload,
        "Locations"
    )

    sectors = execute_payload(
        sector_payload,
        "Sectors"
    )

    organizations = execute_payload(
        organization_payload,
        "Organizations"
    )

    project_organizations = execute_payload(
        project_org_payload,
        "Project organizations"
    )

    partners = execute_payload(
        partner_payload,
        "Partners"
    )

    funding_raw = execute_payload(
        funding_payload,
        "Funding"
    )

    # ---------------------------------------------------------
    # Validate canonical project key
    # ---------------------------------------------------------

    project_ids_current = set(
        projects["fongim_project_id"]
    )

    if projects["fongim_project_id"].duplicated().any():
        raise ValueError(
            "FONGIM sync aborted: duplicate project IDs detected."
        )

    # ---------------------------------------------------------
    # Validate child relationships
    # ---------------------------------------------------------

    orphan_locations = (
        set(locations["fongim_project_id"])
        - project_ids_current
    )

    orphan_sectors = (
        set(sectors["fongim_project_id"])
        - project_ids_current
    )

    orphan_partners = (
        set(partners["fongim_project_id"])
        - project_ids_current
    )

    orphan_project_orgs = (
        set(project_organizations["fongim_project_id"])
        - project_ids_current
    )

    if orphan_locations:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(orphan_locations)} orphan location project IDs."
        )

    if orphan_sectors:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(orphan_sectors)} orphan sector project IDs."
        )

    if orphan_partners:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(orphan_partners)} orphan partner project IDs."
        )

    if orphan_project_orgs:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(orphan_project_orgs)} orphan project-organization IDs."
        )

    # ---------------------------------------------------------
    # Validate organization relationships
    # ---------------------------------------------------------

    organization_ids_current = set(
        organizations["fongim_organization_id"]
    )

    project_org_ids_current = set(
        project_organizations["fongim_organization_id"]
    )

    partner_org_ids_current = set(
        partners["fongim_organization_id"].dropna()
    )

    unmatched_project_orgs = (
        project_org_ids_current
        - organization_ids_current
    )

    unmatched_partner_orgs = (
        partner_org_ids_current
        - organization_ids_current
    )

    if unmatched_project_orgs:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(unmatched_project_orgs)} project organization IDs "
            f"missing from Organisations."
        )

    if unmatched_partner_orgs:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(unmatched_partner_orgs)} partner organization IDs "
            f"missing from Organisations."
        )

    # ---------------------------------------------------------
    # Validate one primary organization per project
    # ---------------------------------------------------------

    orgs_per_project = (
        project_organizations
        .groupby("fongim_project_id")["fongim_organization_id"]
        .nunique()
    )

    multi_org_projects = (
        orgs_per_project[orgs_per_project > 1]
    )

    if len(multi_org_projects):
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(multi_org_projects)} projects have "
            f"multiple primary organizations."
        )

    # ---------------------------------------------------------
    # Link funding through validated native project ID
    # ---------------------------------------------------------

    funding = funding_raw.copy()

    funding["funding_index_numeric"] = pd.to_numeric(
        funding["funding_index"],
        errors="coerce"
    )

    project_lookup = projects[
        [
            "fongim_project_id",
            "project_name"
        ]
    ].copy()

    project_lookup["fongim_project_id_numeric"] = pd.to_numeric(
        project_lookup["fongim_project_id"],
        errors="coerce"
    )

    funding = funding.merge(
        project_lookup,
        left_on="funding_index_numeric",
        right_on="fongim_project_id_numeric",
        how="left",
        suffixes=("_funding", "_canonical")
    )

    funding["linkage_status"] = (
        funding["fongim_project_id"]
        .notna()
        .map({
            True: "linked_native_id",
            False: "unlinked_native_id"
        })
    )

    funding = funding.drop(
        columns=["fongim_project_id_numeric"]
    )

    # ---------------------------------------------------------
    # Sync metadata
    # ---------------------------------------------------------

    sync_finished_at = datetime.now(timezone.utc)

    metadata = {
        "sync_run_id": sync_run_id,
        "source_system": "FONGIM",
        "source_url": FONGIM_SOURCE_URL,
        "source_model_id": FONGIM_MODEL_ID,
        "sync_started_at": sync_started_at.isoformat(),
        "sync_finished_at": sync_finished_at.isoformat(),
        "project_count": len(projects),
        "location_count": len(locations),
        "sector_count": len(sectors),
        "organization_count": len(organizations),
        "project_organization_count": len(project_organizations),
        "partner_count": len(partners),
        "funding_row_count": len(funding),
        "funding_project_count": funding["funding_index"].nunique(),
        "funding_linked_project_count": funding.loc[
            funding["linkage_status"] == "linked_native_id",
            "funding_index"
        ].nunique(),
        "funding_unlinked_project_count": funding.loc[
            funding["linkage_status"] == "unlinked_native_id",
            "funding_index"
        ].nunique(),
    }

    print("\n" + "=" * 70)
    print("SYNC VALIDATION COMPLETE")
    print("=" * 70)

    for key, value in metadata.items():
        print(f"{key}: {value}")

    return {
        "metadata": metadata,
        "projects": projects,
        "locations": locations,
        "sectors": sectors,
        "organizations": organizations,
        "project_organizations": project_organizations,
        "partners": partners,
        "funding": funding
    }

#### 16.3.4 — Persistence Helpers

In [ ]:
# 16.3.4 — FONGIM persistence helpers

import hashlib
import pandas as pd
from datetime import datetime


def fongim_db_value(value):
    """Convert pandas / numpy values into Supabase-safe values."""

    if pd.isna(value):
        return None

    if hasattr(value, "item"):
        value = value.item()

    # Pandas timestamps / Python datetimes
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    # Nullable integer IDs may arrive from pandas as floats
    # (e.g. 48.0 because the column also contains NULLs).
    # Convert only mathematically integral floats.
    if isinstance(value, float) and value.is_integer():
        return int(value)

    return value


def fongim_stable_key(*values):
    """Create deterministic SHA-256 key from stable source fields."""

    normalized = []

    for value in values:
        value = fongim_db_value(value)

        if value is None:
            normalized.append("")
        else:
            normalized.append(str(value).strip())

    raw = "||".join(normalized)

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()


def fongim_validate_unique(records, key, dataset_name):

    values = [record[key] for record in records]

    if any(value is None for value in values):
        raise ValueError(
            f"{dataset_name}: NULL key detected in {key}"
        )

    if len(values) != len(set(values)):
        raise ValueError(
            f"{dataset_name}: duplicate {key} detected"
        )


def fongim_upsert_batches(
    table_name,
    records,
    conflict_column,
    batch_size=250
):
    """Upsert records in controlled batches."""

    written = 0

    for start in range(0, len(records), batch_size):

        batch = records[start:start + batch_size]

        (
            supabase
            .table(table_name)
            .upsert(
                batch,
                on_conflict=conflict_column
            )
            .execute()
        )

        written += len(batch)

    return written


def fongim_mark_missing_inactive(
    table_name,
    key_column,
    current_keys,
    sync_time
):
    """
    Mark records no longer present in FONGIM as inactive.

    Never deletes historical records.
    """

    # Paginate explicitly: Supabase may otherwise return
    # only the first 1000 rows.
    existing_keys = set()
    page_size = 1000
    start = 0

    while True:

        result = (
            supabase
            .table(table_name)
            .select(key_column)
            .range(start, start + page_size - 1)
            .execute()
        )

        rows = result.data or []

        for row in rows:
            existing_keys.add(row[key_column])

        if len(rows) < page_size:
            break

        start += page_size

    missing_keys = existing_keys - set(current_keys)

    for key in missing_keys:

        (
            supabase
            .table(table_name)
            .update({
                "is_present_in_source": False,
                "last_synced_at": sync_time
            })
            .eq(key_column, key)
            .execute()
        )

    return len(missing_keys)

#### 16.3.5 — Complete FONGIM Sync

In [ ]:
# 16.3.5 — Complete FONGIM Sync


def run_complete_fongim_sync():

    print("=" * 70)
    print("COMPLETE FONGIM SYNC")
    print("=" * 70)

    # -----------------------------------------------------
    # 1. Extract + validate source
    # -----------------------------------------------------

    print("1/7  Extracting and validating FONGIM...")

    source = run_fongim_sync_extract()

    metadata = source["metadata"]

    sync_run_id = metadata["sync_run_id"]
    sync_time = metadata["sync_finished_at"]

    projects_df = source["projects"].copy()
    organizations_df = source["organizations"].copy()
    project_orgs_df = source["project_organizations"].copy()
    locations_df = source["locations"].copy()
    sectors_df = source["sectors"].copy()
    partners_df = source["partners"].copy()
    funding_df = source["funding"].copy()

    print("     ✓ Source extraction validated")

    # -----------------------------------------------------
    # 2. Prepare projects
    # -----------------------------------------------------

    print("2/7  Preparing canonical records...")

    project_records = []

    for _, row in projects_df.iterrows():

        project_records.append({
            "fongim_project_id":
                fongim_db_value(row["fongim_project_id"]),

            "project_name":
                fongim_db_value(row["project_name"]),

            "start_date":
                fongim_db_value(row["start_date"]),

            "end_date":
                fongim_db_value(row["end_date"]),

            "status":
                fongim_db_value(row["status"]),

            "project_type":
                fongim_db_value(row["project_type"]),

            "beneficiaries":
                fongim_db_value(row["beneficiaries"]),

            "donor":
                fongim_db_value(row["donor"]),

            "funding_amount_raw":
                fongim_db_value(row["funding_amount_raw"]),

            "partners_raw":
                fongim_db_value(row["partners_raw"]),

            "has_partners":
                fongim_db_value(row["has_partners"]),

            "last_seen_at": sync_time,
            "last_synced_at": sync_time,
            "is_present_in_source": True,
            "last_seen_sync_id": sync_run_id,

            "source_system": "FONGIM",
            "source_model_id": FONGIM_MODEL_ID
        })

    fongim_validate_unique(
        project_records,
        "fongim_project_id",
        "Projects"
    )

    # -----------------------------------------------------
    # 3. Prepare organizations
    # -----------------------------------------------------

    organization_records = []

    for _, row in organizations_df.iterrows():

        organization_records.append({
            "fongim_organization_id":
                fongim_db_value(
                    row["fongim_organization_id"]
                ),

            "organization_name":
                fongim_db_value(
                    row["organization_name"]
                ),

            "last_seen_at": sync_time,
            "last_synced_at": sync_time,
            "is_present_in_source": True,
            "last_seen_sync_id": sync_run_id,

            "source_system": "FONGIM",
            "source_model_id": FONGIM_MODEL_ID
        })

    fongim_validate_unique(
        organization_records,
        "fongim_organization_id",
        "Organizations"
    )

    # -----------------------------------------------------
    # 4. Prepare project ↔ organization relationships
    # -----------------------------------------------------

    project_organization_records = []

    for _, row in project_orgs_df.iterrows():

        project_organization_records.append({
            "fongim_project_id":
                fongim_db_value(
                    row["fongim_project_id"]
                ),

            "fongim_organization_id":
                fongim_db_value(
                    row["fongim_organization_id"]
                ),

            "last_seen_at": sync_time,
            "last_synced_at": sync_time,
            "is_present_in_source": True,
            "last_seen_sync_id": sync_run_id
        })

    fongim_validate_unique(
        project_organization_records,
        "fongim_project_id",
        "Project organizations"
    )

    # -----------------------------------------------------
    # 5. Prepare locations
    # -----------------------------------------------------

    location_records = []

    for _, row in locations_df.iterrows():

        project_id = fongim_db_value(
            row["fongim_project_id"]
        )

        zone_id = fongim_db_value(
            row["fongim_zone_id"]
        )

        region = fongim_db_value(row["region"])
        cercle = fongim_db_value(row["cercle"])

        commune = fongim_db_value(
            row["commune_raw"]
        )

        location_records.append({
            "record_key": fongim_stable_key(
                project_id,
                zone_id,
                region,
                cercle,
                commune
            ),

            "fongim_project_id": project_id,
            "fongim_zone_id": zone_id,
            "region": region,
            "cercle": cercle,
            "commune_raw": commune,

            "last_seen_at": sync_time,
            "last_synced_at": sync_time,
            "is_present_in_source": True,
            "last_seen_sync_id": sync_run_id
        })

    fongim_validate_unique(
        location_records,
        "record_key",
        "Locations"
    )

    # -----------------------------------------------------
    # 6. Prepare sectors
    # -----------------------------------------------------

    sector_records = []

    for _, row in sectors_df.iterrows():

        project_id = fongim_db_value(
            row["fongim_project_id"]
        )

        sector = fongim_db_value(
            row["sector"]
        )

        sector_other = fongim_db_value(
            row["sector_other_raw"]
        )

        sector_records.append({
            "record_key": fongim_stable_key(
                project_id,
                sector,
                sector_other
            ),

            "fongim_project_id": project_id,
            "sector": sector,
            "sector_other_raw": sector_other,

            "last_seen_at": sync_time,
            "last_synced_at": sync_time,
            "is_present_in_source": True,
            "last_seen_sync_id": sync_run_id
        })

    fongim_validate_unique(
        sector_records,
        "record_key",
        "Sectors"
    )

    # -----------------------------------------------------
    # 7. Prepare partners
    # -----------------------------------------------------

    partner_records = []

    for _, row in partners_df.iterrows():

        project_id = fongim_db_value(
            row["fongim_project_id"]
        )

        partner_name = fongim_db_value(
            row["partner_name"]
        )

        organization_id = fongim_db_value(
            row["fongim_organization_id"]
        )

        partner_records.append({
            "record_key": fongim_stable_key(
                project_id,
                partner_name,
                organization_id
            ),

            "fongim_project_id": project_id,
            "partner_name": partner_name,

            "fongim_organization_id":
                organization_id,

            "last_seen_at": sync_time,
            "last_synced_at": sync_time,
            "is_present_in_source": True,
            "last_seen_sync_id": sync_run_id
        })

    fongim_validate_unique(
        partner_records,
        "record_key",
        "Partners"
    )

    # -----------------------------------------------------
    # 8. Prepare funding
    # -----------------------------------------------------

    funding_records = []

    funding_keys = []

    for _, row in funding_df.iterrows():

        # Stable source identity.
        # Amounts deliberately excluded because they may change.
        key = fongim_stable_key(
            row["funding_id"],
            row["funding_index"],
            row["sector"]
        )

        funding_keys.append(key)

        funding_records.append({
            "record_key": key,

            "funding_id":
                fongim_db_value(row["funding_id"]),

            "funding_index":
                fongim_db_value(row["funding_index"]),

            "fongim_project_id":
                fongim_db_value(
                    row["fongim_project_id"]
                ),

            "organization_name":
                fongim_db_value(
                    row["organization_name"]
                ),

            "project_name_raw":
                fongim_db_value(
                    row["project_name_raw"]
                ),

            "project_type":
                fongim_db_value(
                    row["project_type"]
                ),

            "sector":
                fongim_db_value(row["sector"]),

            "funding_2024":
                fongim_db_value(
                    row["funding_2024"]
                ),

            "funding_2025":
                fongim_db_value(
                    row["funding_2025"]
                ),

            "linkage_status":
                fongim_db_value(
                    row["linkage_status"]
                ),

            "last_seen_at": sync_time,
            "last_synced_at": sync_time,
            "is_present_in_source": True,
            "last_seen_sync_id": sync_run_id
        })

    fongim_validate_unique(
        funding_records,
        "record_key",
        "Funding"
    )

    # -----------------------------------------------------
    # Cross-table integrity validation BEFORE WRITE
    # -----------------------------------------------------

    project_ids = {
        r["fongim_project_id"]
        for r in project_records
    }

    organization_ids = {
        r["fongim_organization_id"]
        for r in organization_records
    }

    project_child_sets = {
        "Project organizations": {
            r["fongim_project_id"]
            for r in project_organization_records
        },
        "Locations": {
            r["fongim_project_id"]
            for r in location_records
        },
        "Sectors": {
            r["fongim_project_id"]
            for r in sector_records
        },
        "Partners": {
            r["fongim_project_id"]
            for r in partner_records
        }
    }

    for name, ids in project_child_sets.items():

        orphans = ids - project_ids

        if orphans:
            raise ValueError(
                f"{name}: orphan project IDs detected: "
                f"{sorted(orphans)}"
            )

    project_org_ids = {
        r["fongim_organization_id"]
        for r in project_organization_records
    }

    orphan_orgs = project_org_ids - organization_ids

    if orphan_orgs:
        raise ValueError(
            "Project organizations contain orphan "
            f"organization IDs: {sorted(orphan_orgs)}"
        )

    partner_org_ids = {
        r["fongim_organization_id"]
        for r in partner_records
        if r["fongim_organization_id"] is not None
    }

    orphan_partner_orgs = (
        partner_org_ids - organization_ids
    )

    if orphan_partner_orgs:
        raise ValueError(
            "Partners contain orphan organization IDs: "
            f"{sorted(orphan_partner_orgs)}"
        )

    print("     ✓ All seven datasets prepared")
    print("     ✓ Cross-table integrity validated")
    print()
    print("     Projects:", len(project_records))
    print("     Organizations:", len(organization_records))
    print(
        "     Project organizations:",
        len(project_organization_records)
    )
    print("     Locations:", len(location_records))
    print("     Sectors:", len(sector_records))
    print("     Partners:", len(partner_records))
    print("     Funding:", len(funding_records))

    # -----------------------------------------------------
    # Database phase
    # -----------------------------------------------------

    print()
    print("3/7  Registering sync run...")

    sync_run_record = {
        "sync_run_id": sync_run_id,
        "source_system": metadata["source_system"],
        "source_url": metadata["source_url"],
        "source_model_id": metadata["source_model_id"],
        "sync_started_at": metadata["sync_started_at"],
        "sync_finished_at": metadata["sync_finished_at"],
        "status": "sync_started",

        "project_count": len(project_records),
        "location_count": len(location_records),
        "sector_count": len(sector_records),
        "organization_count": len(organization_records),

        "project_organization_count":
            len(project_organization_records),

        "partner_count": len(partner_records),
        "funding_row_count": len(funding_records)
    }

    (
        supabase
        .table("fongim_sync_runs")
        .upsert(
            sync_run_record,
            on_conflict="sync_run_id"
        )
        .execute()
    )

    print("     ✓ Sync run registered")

    try:

        # Dependency-safe order:
        # parent tables first, then relational children.

        print("4/7  Synchronizing parent tables...")

        fongim_upsert_batches(
            "fongim_projects",
            project_records,
            "fongim_project_id"
        )

        fongim_upsert_batches(
            "fongim_organizations",
            organization_records,
            "fongim_organization_id"
        )

        print("     ✓ Parent tables synchronized")

        print("5/7  Synchronizing relational tables...")

        fongim_upsert_batches(
            "fongim_project_organizations",
            project_organization_records,
            "fongim_project_id"
        )

        fongim_upsert_batches(
            "fongim_project_locations",
            location_records,
            "record_key"
        )

        fongim_upsert_batches(
            "fongim_project_sectors",
            sector_records,
            "record_key"
        )

        fongim_upsert_batches(
            "fongim_project_partners",
            partner_records,
            "record_key"
        )

        fongim_upsert_batches(
            "fongim_project_funding",
            funding_records,
            "record_key"
        )

        print("     ✓ Relational tables synchronized")

        # -------------------------------------------------
        # Only AFTER every upsert succeeded:
        # mark disappeared records inactive.
        # -------------------------------------------------

        print("6/7  Reconciling source presence...")

        inactive_counts = {}

        inactive_counts["projects"] = (
            fongim_mark_missing_inactive(
                "fongim_projects",
                "fongim_project_id",
                project_ids,
                sync_time
            )
        )

        inactive_counts["organizations"] = (
            fongim_mark_missing_inactive(
                "fongim_organizations",
                "fongim_organization_id",
                organization_ids,
                sync_time
            )
        )

        inactive_counts["project_organizations"] = (
            fongim_mark_missing_inactive(
                "fongim_project_organizations",
                "fongim_project_id",
                {
                    r["fongim_project_id"]
                    for r in project_organization_records
                },
                sync_time
            )
        )

        inactive_counts["locations"] = (
            fongim_mark_missing_inactive(
                "fongim_project_locations",
                "record_key",
                {
                    r["record_key"]
                    for r in location_records
                },
                sync_time
            )
        )

        inactive_counts["sectors"] = (
            fongim_mark_missing_inactive(
                "fongim_project_sectors",
                "record_key",
                {
                    r["record_key"]
                    for r in sector_records
                },
                sync_time
            )
        )

        inactive_counts["partners"] = (
            fongim_mark_missing_inactive(
                "fongim_project_partners",
                "record_key",
                {
                    r["record_key"]
                    for r in partner_records
                },
                sync_time
            )
        )

        inactive_counts["funding"] = (
            fongim_mark_missing_inactive(
                "fongim_project_funding",
                "record_key",
                {
                    r["record_key"]
                    for r in funding_records
                },
                sync_time
            )
        )

        print("     ✓ Source presence reconciled")

        # -------------------------------------------------
        # Successful completion
        # -------------------------------------------------

        print("7/7  Finalizing sync...")

        (
            supabase
            .table("fongim_sync_runs")
            .update({
                "status": "success"
            })
            .eq("sync_run_id", sync_run_id)
            .execute()
        )

        print("     ✓ Sync marked successful")

        print()
        print("=" * 70)
        print("FONGIM SYNC COMPLETE")
        print("=" * 70)

        print("Sync run:", sync_run_id)
        print("Projects:", len(project_records))
        print("Organizations:", len(organization_records))
        print(
            "Project organizations:",
            len(project_organization_records)
        )
        print("Locations:", len(location_records))
        print("Sectors:", len(sector_records))
        print("Partners:", len(partner_records))
        print("Funding:", len(funding_records))

        print()
        print("Marked inactive:", inactive_counts)

        return {
            "sync_run_id": sync_run_id,
            "status": "success",
            "counts": {
                "projects": len(project_records),
                "organizations": len(organization_records),

                "project_organizations":
                    len(project_organization_records),

                "locations": len(location_records),
                "sectors": len(sector_records),
                "partners": len(partner_records),
                "funding": len(funding_records)
            },
            "marked_inactive": inactive_counts
        }

    except Exception as exc:

        # The run remains auditable if database persistence fails.
        (
            supabase
            .table("fongim_sync_runs")
            .update({
                "status": "failed",
                "error_message": str(exc)
            })
            .eq("sync_run_id", sync_run_id)
            .execute()
        )

        print()
        print("FONGIM SYNC FAILED")
        print("Sync run:", sync_run_id)
        print("Error:", exc)

        raise

#### 16.3.6 — Connector Test

In [ ]:
# 16.3.6 — FONGIM connector extraction test
# Read-only: does NOT write to Supabase.

fongim_test = run_fongim_sync_extract()

FONGIM SYNC EXTRACT
Sync run: d85d68a1-2c47-4ad7-8b6a-21b638eb2612
Started: 2026-08-28T16:15:43.126648+00:00
✓ Projects: 703 rows
✓ Locations: 2767 rows
✓ Sectors: 1369 rows
✓ Organizations: 103 rows
✓ Project organizations: 703 rows
✓ Partners: 1483 rows
✓ Funding: 328 rows

SYNC VALIDATION COMPLETE
sync_run_id: d85d68a1-2c47-4ad7-8b6a-21b638eb2612
source_system: FONGIM
source_url: https://www.fongim.org/carte-interactive
source_model_id: 2218134
sync_started_at: 2026-08-28T16:15:43.126648+00:00
sync_finished_at: 2026-08-28T16:15:51.052426+00:00
project_count: 703
location_count: 2767
sector_count: 1369
organization_count: 103
project_organization_count: 703
partner_count: 1483
funding_row_count: 328
funding_project_count: 166
funding_linked_project_count: 164
funding_unlinked_project_count: 2


In [ ]:
# 16.3.6 — Complete FONGIM synchronization test
# This WILL synchronize the current validated source state with Supabase.

fongim_sync_test = run_complete_fongim_sync()

COMPLETE FONGIM SYNC
1/7  Extracting and validating FONGIM...
FONGIM SYNC EXTRACT
Sync run: faa61394-2d1a-467c-94dd-49cbab781178
Started: 2026-08-28T15:57:33.176269+00:00
✓ Projects: 703 rows
✓ Locations: 2767 rows
✓ Sectors: 1369 rows
✓ Organizations: 103 rows
✓ Project organizations: 703 rows
✓ Partners: 1483 rows
✓ Funding: 328 rows

SYNC VALIDATION COMPLETE
sync_run_id: faa61394-2d1a-467c-94dd-49cbab781178
source_system: FONGIM
source_url: https://www.fongim.org/carte-interactive
source_model_id: 2218134
sync_started_at: 2026-08-28T15:57:33.176269+00:00
sync_finished_at: 2026-08-28T15:57:41.015513+00:00
project_count: 703
location_count: 2767
sector_count: 1369
organization_count: 103
project_organization_count: 703
partner_count: 1483
funding_row_count: 328
funding_project_count: 166
funding_linked_project_count: 164
funding_unlinked_project_count: 2
     ✓ Source extraction validated
2/7  Preparing canonical records...
     ✓ All seven datasets prepared
     ✓ Cross-table integri

## 17 — Runtime Packaging

#### 17.1 — Build Runtime Module

In [ ]:
# 17.1 — Build Runtime Module

from pathlib import Path


runtime_code = r'''
# ============================================================
# MALI KNOWLEDGE HUB — OPERATIONAL RUNTIME v0.3
# ============================================================

import hashlib
import uuid
from datetime import datetime, timezone

import pandas as pd
import requests


# ============================================================
# Runtime clients
# ============================================================

supabase = None
openai_client = None
hdx_hapi_app_identifier = None


def configure_runtime(
    supabase_client,
    openai_client_instance,
    hdx_hapi_app_identifier_instance=None
):
    """
    Attach initialized Supabase, OpenAI and HDX HAPI
    configuration to the runtime.

    The clients and configuration are created by:
    00 — START MALI KNOWLEDGE HUB
    """

    global supabase
    global openai_client
    global hdx_hapi_app_identifier

    supabase = supabase_client
    openai_client = openai_client_instance
    hdx_hapi_app_identifier = hdx_hapi_app_identifier_instance


def _require_supabase():
    if supabase is None:
        raise RuntimeError(
            "Supabase client is not configured. "
            "Call configure_runtime(...) first."
        )


def _require_openai():
    if openai_client is None:
        raise RuntimeError(
            "OpenAI client is not configured. "
            "Call configure_runtime(...) first."
        )


# ============================================================
# Retrieval Engine
# ============================================================

def search_knowledge_base(
    question,
    match_count=12,
    filter_document_ids=None,
    similarity_threshold=None
):
    """
    Search the Knowledge Hub document corpus.

    Optional document filtering allows retrieval to be restricted
    to a defined subset of documents before vector ranking.

    Returns ranked chunks enriched with document metadata
    so provenance is preserved throughout the pipeline.
    """

    _require_supabase()
    _require_openai()


    # --------------------------------------------------------
    # 1. Embed the question
    # --------------------------------------------------------

    embedding_response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )

    query_embedding = embedding_response.data[0].embedding


    # --------------------------------------------------------
    # 2. Vector search in Supabase
    # --------------------------------------------------------

    response = supabase.rpc(
        "match_chunks",
        {
            "query_embedding": query_embedding,
            "match_count": match_count,
            "filter_document_ids": filter_document_ids
        }
    ).execute()

    raw_results = response.data or []


    # --------------------------------------------------------
    # 3. Optional similarity threshold
    # --------------------------------------------------------

    if similarity_threshold is not None:

        raw_results = [
            r
            for r in raw_results
            if (
                r.get("similarity") is not None
                and r.get("similarity") >= similarity_threshold
            )
        ]


    # --------------------------------------------------------
    # 4. Load document metadata once
    # --------------------------------------------------------

    document_ids = list({
        r.get("document_id")
        for r in raw_results
        if r.get("document_id")
    })

    documents_by_id = {}

    if document_ids:

        docs_response = (
            supabase
            .table("documents")
            .select(
                "id,title,organization,publication_date,"
                "valid_from,valid_until,document_type,"
                "language,geographic_scope,status,version"
            )
            .in_("id", document_ids)
            .execute()
        )

        documents_by_id = {
            d["id"]: d
            for d in docs_response.data
        }


    # --------------------------------------------------------
    # 5. Enrich every chunk with provenance
    # --------------------------------------------------------

    results = []

    for r in raw_results:

        doc = documents_by_id.get(
            r.get("document_id"),
            {}
        )

        results.append({
            **r,

            "document_title": doc.get("title"),
            "organization": doc.get("organization"),
            "publication_date": doc.get("publication_date"),
            "valid_from": doc.get("valid_from"),
            "valid_until": doc.get("valid_until"),
            "document_type": doc.get("document_type"),
            "language": doc.get("language"),
            "geographic_scope": doc.get("geographic_scope"),
            "document_status": doc.get("status"),
            "version": doc.get("version")
        })

    return results

# ============================================================
# OCHA / HDX HAPI Connector
# ============================================================

HAPI_POPULATION_STATUS_LABELS = {
    "INN": "People in need",
    "TGT": "People targeted",
    "all": "All / source aggregate"
}

HAPI_RESOURCE_METADATA_CACHE = {}


def _require_hapi():
    if not hdx_hapi_app_identifier:
        raise RuntimeError(
            "HDX HAPI is not configured. "
            "Pass HDX_HAPI_APP_IDENTIFIER to configure_runtime(...)."
        )


def hapi_get(
    endpoint,
    params=None,
    limit=1000,
    offset=0,
    timeout=30
):
    _require_hapi()

    if params is None:
        params = {}

    query_params = {
        **params,
        "limit": limit,
        "offset": offset,
        "output_format": "json"
    }

    headers = {
        "X-HDX-HAPI-APP-IDENTIFIER": hdx_hapi_app_identifier
    }

    url = (
        "https://hapi.humdata.org/api/v2/"
        f"{endpoint.lstrip('/')}"
    )

    response = requests.get(
        url,
        params=query_params,
        headers=headers,
        timeout=timeout
    )

    response.raise_for_status()

    return response.json()


def _stable_hapi_evidence_id(row, endpoint):
    import json

    identity = {
        "endpoint": endpoint,
        "resource_hdx_id": row.get("resource_hdx_id"),
        "location_code": row.get("location_code"),
        "admin1_code": row.get("admin1_code"),
        "admin2_code": row.get("admin2_code"),
        "sector_code": row.get("sector_code"),
        "category": row.get("category"),
        "population_status": row.get("population_status"),
        "population": row.get("population"),
        "reference_period_start": row.get("reference_period_start"),
        "reference_period_end": row.get("reference_period_end")
    }

    raw = json.dumps(
        identity,
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")

    digest = hashlib.sha256(raw).hexdigest()[:12]

    return f"HAPI-{digest}"


def normalize_humanitarian_need_record(row):
    endpoint = "affected-people/humanitarian-needs"

    status_raw = row.get("population_status")

    status_label = HAPI_POPULATION_STATUS_LABELS.get(
        status_raw,
        status_raw
    )

    admin1 = row.get("admin1_name")
    admin2 = row.get("admin2_name")

    if admin2:
        location_text = f"{admin2}, {admin1}, Mali"
    elif admin1:
        location_text = f"{admin1}, Mali"
    else:
        location_text = "Mali"

    sector = row.get("sector_name") or "Unspecified sector"
    category = (
        row.get("category")
        or "Unspecified population category"
    )
    population = row.get("population")

    period_start = row.get("reference_period_start")
    period_end = row.get("reference_period_end")

    if isinstance(population, (int, float)):
        passage = (
            f"{status_label}: {population:,} people; "
            f"sector: {sector}; "
            f"population category: {category}; "
            f"location: {location_text}; "
            f"reference period: "
            f"{period_start} to {period_end}."
        )
    else:
        passage = (
            f"{status_label}; "
            f"sector: {sector}; "
            f"population category: {category}; "
            f"location: {location_text}; "
            f"reference period: "
            f"{period_start} to {period_end}."
        )

    return {
        "evidence_id": _stable_hapi_evidence_id(
            row,
            endpoint
        ),
        "source_type": "hdx_hapi",
        "document_id": None,
        "document_title":
            "HDX HAPI — Humanitarian Needs",
        "document_type":
            "structured_humanitarian_data",
        "organization": "OCHA / HDX",
        "publication_date": None,
        "version": None,
        "page": None,
        "section": sector,
        "chunk_id": None,
        "similarity": None,
        "retrieved_for": None,
        "passage": passage,
        "evidence_type": "structured_data",
        "supports": None,
        "source_endpoint": endpoint,
        "resource_hdx_id":
            row.get("resource_hdx_id"),
        "retrieved_at":
            datetime.now(timezone.utc).isoformat(),
        "reference_period_start": period_start,
        "reference_period_end": period_end,
        "location_code": row.get("location_code"),
        "location_name": row.get("location_name"),
        "admin_level": row.get("admin_level"),
        "admin1_code": row.get("admin1_code"),
        "admin1_name": admin1,
        "admin2_code": row.get("admin2_code"),
        "admin2_name": admin2,
        "sector_code": row.get("sector_code"),
        "sector_name": sector,
        "population_category": category,
        "population_status": status_raw,
        "population_status_label": status_label,
        "value": population,
        "unit": "people",
        "raw_record": row
    }


def normalize_humanitarian_needs(rows):
    return [
        normalize_humanitarian_need_record(row)
        for row in rows
    ]


def get_hapi_resource_metadata(resource_hdx_id):
    result = hapi_get(
        "metadata/resource",
        params={
            "resource_hdx_id": resource_hdx_id
        },
        limit=100
    )

    return result.get("data", [])


def enrich_hapi_evidence(
    evidence_item,
    resource_metadata
):
    if not resource_metadata:
        return evidence_item

    meta = resource_metadata[0]

    return {
        **evidence_item,
        "source_platform": "HDX HAPI",
        "resource_name": meta.get("name"),
        "resource_format": meta.get("format"),
        "resource_update_date":
            meta.get("update_date"),
        "resource_download_url":
            meta.get("download_url"),
        "resource_hdx_link": meta.get("hdx_link"),
        "dataset_hdx_id":
            meta.get("dataset_hdx_id"),
        "dataset_title":
            meta.get("dataset_hdx_title"),
        "dataset_hdx_link":
            meta.get("dataset_hdx_link"),
        "provider_name":
            meta.get(
                "dataset_hdx_provider_name"
            ),
        "provider_stub":
            meta.get(
                "dataset_hdx_provider_stub"
            ),
        "provider_hdx_link":
            meta.get("provider_hdx_link"),
        "hapi_updated_date":
            meta.get("hapi_updated_date")
    }


def get_cached_resource_metadata(resource_hdx_id):
    if not resource_hdx_id:
        return []

    if (
        resource_hdx_id
        in HAPI_RESOURCE_METADATA_CACHE
    ):
        return HAPI_RESOURCE_METADATA_CACHE[
            resource_hdx_id
        ]

    metadata = get_hapi_resource_metadata(
        resource_hdx_id
    )

    HAPI_RESOURCE_METADATA_CACHE[
        resource_hdx_id
    ] = metadata

    return metadata


def enrich_hapi_evidence_batch(evidence_items):
    enriched_items = []

    resource_ids = {
        item.get("resource_hdx_id")
        for item in evidence_items
        if item.get("resource_hdx_id")
    }

    for resource_id in resource_ids:
        get_cached_resource_metadata(resource_id)

    for item in evidence_items:
        resource_id = item.get(
            "resource_hdx_id"
        )

        metadata = (
            HAPI_RESOURCE_METADATA_CACHE.get(
                resource_id,
                []
            )
            if resource_id
            else []
        )

        enriched_items.append(
            enrich_hapi_evidence(
                item,
                metadata
            )
        )

    return enriched_items


def research_humanitarian_needs(
    admin1_name=None,
    admin2_name=None,
    sector_name=None,
    population_status="INN",
    latest_only=True,
    limit=10000
):
    params = {
        "location_code": "MLI"
    }

    if admin1_name:
        params["admin1_name"] = admin1_name

    if admin2_name:
        params["admin2_name"] = admin2_name

    if sector_name:
        params["sector_name"] = sector_name

    if population_status:
        params[
            "population_status"
        ] = population_status

    response = hapi_get(
        "affected-people/humanitarian-needs",
        params=params,
        limit=limit
    )

    rows = response.get("data", [])

    if not rows:
        return []

    if latest_only:
        available_periods = [
            row.get("reference_period_end")
            for row in rows
            if row.get("reference_period_end")
        ]

        if available_periods:
            latest_period_end = max(
                available_periods
            )

            rows = [
                row
                for row in rows
                if row.get(
                    "reference_period_end"
                ) == latest_period_end
            ]

    evidence = normalize_humanitarian_needs(
        rows
    )

    evidence = enrich_hapi_evidence_batch(
        evidence
    )

    return evidence

# ============================================================
# FONGIM Connector — Configuration
# ============================================================

FONGIM_SOURCE_URL = "https://www.fongim.org/carte-interactive"

FONGIM_QUERYDATA_URL = (
    "https://wabi-south-africa-north-a-primary-api.analysis.windows.net/"
    "public/reports/querydata?synchronous=true"
)

FONGIM_MODEL_ID = 2218134

FONGIM_HEADERS = {
    "Content-Type": "application/json",
    "X-PowerBI-ResourceKey":
        "3dc6e1ae-910f-42a2-b41d-01aa141e6ee7"
}


def build_fongim_payload(
    entity,
    source_alias,
    fields,
    count=10000
):
    """
    Build a standard FONGIM Power BI query payload.

    fields:
        list of (source_field, output_name)
    """

    selects = []

    for source_field, output_name in fields:

        selects.append({
            "Column": {
                "Expression": {
                    "SourceRef": {
                        "Source": source_alias
                    }
                },
                "Property": source_field
            },
            "Name": output_name
        })

    return {
        "version": "1.0.0",
        "queries": [
            {
                "Query": {
                    "Commands": [
                        {
                            "SemanticQueryDataShapeCommand": {
                                "Query": {
                                    "Version": 2,
                                    "From": [
                                        {
                                            "Name": source_alias,
                                            "Entity": entity,
                                            "Type": 0
                                        }
                                    ],
                                    "Select": selects
                                },
                                "Binding": {
                                    "Primary": {
                                        "Groupings": [
                                            {
                                                "Projections":
                                                    list(
                                                        range(
                                                            len(fields)
                                                        )
                                                    )
                                            }
                                        ]
                                    },
                                    "DataReduction": {
                                        "DataVolume": 6,
                                        "Primary": {
                                            "Window": {
                                                "Count": count
                                            }
                                        }
                                    },
                                    "Version": 1
                                },
                                "ExecutionMetricsKind": 1
                            }
                        }
                    ]
                },
                "QueryId": ""
            }
        ],
        "cancelQueries": [],
        "modelId": FONGIM_MODEL_ID
    }


FONGIM_PROJECT_FIELDS = [
    ("_index", "fongim_project_id"),
    ("Quel est le nom du projet ?", "project_name"),
    ("Date de début de la mise en œuvre", "start_date"),
    ("Date de fin de la mise en œuvre", "end_date"),
    ("Statut", "status"),
    ("Comment qualifiez vous votre projet ?", "project_type"),
    ("Le nombre total de bénéficiares du projet", "beneficiaries"),
    ("Quel est le bailleur ?", "donor"),
    ("Quel le montant du financement", "funding_amount_raw"),
    (
        "Quel(s) est/sont le(s) partenaire(s) du projet",
        "partners_raw"
    ),
    (
        "travaillez-vous avec des partenaires dans ce projet",
        "has_partners"
    )
]

FONGIM_LOCATION_FIELDS = [
    ("ID_projet", "fongim_project_id"),
    ("ID_zone", "fongim_zone_id"),
    ("Région", "region"),
    ("Cercle", "cercle"),
    ("Commune", "commune_raw")
]

FONGIM_SECTOR_FIELDS = [
    ("ID_projet", "fongim_project_id"),
    ("Secteurs", "sector"),
    ("Autres", "sector_other_raw")
]

FONGIM_ORGANIZATION_FIELDS = [
    ("id", "fongim_organization_id"),
    ("nom_org", "organization_name")
]

FONGIM_PARTNER_FIELDS = [
    ("ID_projet", "fongim_project_id"),
    ("Partenaire", "partner_name"),
    ("ID_ONG", "fongim_organization_id")
]

FONGIM_FUNDING_FIELDS = [
    ("id", "funding_id"),
    ("nom_org", "organization_name"),
    ("_index", "funding_index"),
    ("Projet ", "project_name_raw"),
    ("Type", "project_type"),
    ("Secteurs", "sector"),
    ("M2024", "funding_2024"),
    ("M2025", "funding_2025")
]


project_master_payload = build_fongim_payload(
    "Projets",
    "p",
    FONGIM_PROJECT_FIELDS,
    count=5000
)

location_payload = build_fongim_payload(
    "Zones",
    "z",
    FONGIM_LOCATION_FIELDS
)

sector_payload = build_fongim_payload(
    "Secteurs",
    "s",
    FONGIM_SECTOR_FIELDS
)

organization_payload = build_fongim_payload(
    "Organisations",
    "o",
    FONGIM_ORGANIZATION_FIELDS
)

partner_payload = build_fongim_payload(
    "Partenaires",
    "pa",
    FONGIM_PARTNER_FIELDS
)

funding_payload = build_fongim_payload(
    "funding",
    "f",
    FONGIM_FUNDING_FIELDS
)


project_org_payload = {
    "version": "1.0.0",
    "queries": [
        {
            "Query": {
                "Commands": [
                    {
                        "SemanticQueryDataShapeCommand": {
                            "Query": {
                                "Version": 2,
                                "From": [
                                    {
                                        "Name": "p",
                                        "Entity": "Projets",
                                        "Type": 0
                                    },
                                    {
                                        "Name": "o",
                                        "Entity": "Organisations",
                                        "Type": 0
                                    }
                                ],
                                "Select": [
                                    {
                                        "Column": {
                                            "Expression": {
                                                "SourceRef": {
                                                    "Source": "p"
                                                }
                                            },
                                            "Property": "_index"
                                        },
                                        "Name": "fongim_project_id"
                                    },
                                    {
                                        "Column": {
                                            "Expression": {
                                                "SourceRef": {
                                                    "Source": "p"
                                                }
                                            },
                                            "Property":
                                                "Quel est le nom du projet ?"
                                        },
                                        "Name": "project_name"
                                    },
                                    {
                                        "Column": {
                                            "Expression": {
                                                "SourceRef": {
                                                    "Source": "o"
                                                }
                                            },
                                            "Property": "id"
                                        },
                                        "Name": "fongim_organization_id"
                                    },
                                    {
                                        "Column": {
                                            "Expression": {
                                                "SourceRef": {
                                                    "Source": "o"
                                                }
                                            },
                                            "Property": "nom_org"
                                        },
                                        "Name": "organization_name"
                                    }
                                ]
                            },
                            "Binding": {
                                "Primary": {
                                    "Groupings": [
                                        {
                                            "Projections": [0, 1, 2, 3]
                                        }
                                    ]
                                },
                                "DataReduction": {
                                    "DataVolume": 6,
                                    "Primary": {
                                        "Window": {
                                            "Count": 5000
                                        }
                                    }
                                },
                                "Version": 1
                            },
                            "ExecutionMetricsKind": 1
                        }
                    }
                ]
            },
            "QueryId": ""
        }
    ],
    "cancelQueries": [],
    "modelId": FONGIM_MODEL_ID
}


# ============================================================
# FONGIM Connector — Power BI Decoder
# ============================================================

def decode_powerbi_rows(response_json):
    """
    Decode the primary Power BI DSR grouping returned by querydata.
    Supports:
    - dictionary encoded strings
    - repeated values via R bitmask
    - Power BI date timestamps
    """

    result = response_json["results"][0]["result"]["data"]

    descriptor = result["descriptor"]["Select"]
    column_names = [item["Name"] for item in descriptor]

    ds = result["dsr"]["DS"][0]
    rows = ds["PH"][0]["DM0"]
    value_dicts = ds.get("ValueDicts", {})

    schema = rows[0].get("S", [])

    column_meta = []

    for i, col in enumerate(schema):
        column_meta.append({
            "name": col.get("N"),
            "type": col.get("T"),
            "dict": col.get("DN")
        })

    decoded_rows = []
    previous = [None] * len(column_names)

    for row in rows:
        values = [None] * len(column_names)

        repeat_mask = row.get("R", 0)
        raw_values = iter(row.get("C", []))

        for i in range(len(column_names)):

            if repeat_mask & (1 << i):
                values[i] = previous[i]
                continue

            try:
                value = next(raw_values)
            except StopIteration:
                value = None

            meta = (
                column_meta[i]
                if i < len(column_meta)
                else {}
            )

            dict_name = meta.get("dict")

            if dict_name and value is not None:
                dictionary = value_dicts.get(
                    dict_name,
                    []
                )

                if (
                    isinstance(value, int)
                    and value < len(dictionary)
                ):
                    value = dictionary[value]

            if (
                meta.get("type") == 7
                and isinstance(value, (int, float))
            ):
                value = datetime.fromtimestamp(
                    value / 1000,
                    tz=timezone.utc
                ).date().isoformat()

            values[i] = value

        decoded_rows.append(values)
        previous = values

    return pd.DataFrame(
        decoded_rows,
        columns=column_names
    )


# ============================================================
# FONGIM Connector — Extract & Validate
# ============================================================

def run_fongim_sync_extract():
    """
    Pull the current FONGIM operational dataset from the public
    Power BI semantic model and return validated relational tables.

    No Supabase writes are performed here.
    """

    sync_started_at = datetime.now(timezone.utc)
    sync_run_id = str(uuid.uuid4())

    print("=" * 70)
    print("FONGIM SYNC EXTRACT")
    print("=" * 70)
    print("Sync run:", sync_run_id)
    print("Started:", sync_started_at.isoformat())

    def execute_payload(payload, label):

        response = requests.post(
            FONGIM_QUERYDATA_URL,
            headers=FONGIM_HEADERS,
            json=payload,
            timeout=30
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"{label} failed with HTTP "
                f"{response.status_code}: "
                f"{response.text[:500]}"
            )

        df = decode_powerbi_rows(
            response.json()
        )

        print(
            f"✓ {label}: {len(df)} rows"
        )

        return df

    projects = execute_payload(
        project_master_payload,
        "Projects"
    )

    locations = execute_payload(
        location_payload,
        "Locations"
    )

    sectors = execute_payload(
        sector_payload,
        "Sectors"
    )

    organizations = execute_payload(
        organization_payload,
        "Organizations"
    )

    project_organizations = execute_payload(
        project_org_payload,
        "Project organizations"
    )

    partners = execute_payload(
        partner_payload,
        "Partners"
    )

    funding_raw = execute_payload(
        funding_payload,
        "Funding"
    )

    project_ids_current = set(
        projects["fongim_project_id"]
    )

    if projects[
        "fongim_project_id"
    ].duplicated().any():
        raise ValueError(
            "FONGIM sync aborted: duplicate "
            "project IDs detected."
        )

    orphan_locations = (
        set(locations["fongim_project_id"])
        - project_ids_current
    )

    orphan_sectors = (
        set(sectors["fongim_project_id"])
        - project_ids_current
    )

    orphan_partners = (
        set(partners["fongim_project_id"])
        - project_ids_current
    )

    orphan_project_orgs = (
        set(
            project_organizations[
                "fongim_project_id"
            ]
        )
        - project_ids_current
    )

    if orphan_locations:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(orphan_locations)} orphan "
            "location project IDs."
        )

    if orphan_sectors:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(orphan_sectors)} orphan "
            "sector project IDs."
        )

    if orphan_partners:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(orphan_partners)} orphan "
            "partner project IDs."
        )

    if orphan_project_orgs:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(orphan_project_orgs)} orphan "
            "project-organization IDs."
        )

    organization_ids_current = set(
        organizations[
            "fongim_organization_id"
        ]
    )

    project_org_ids_current = set(
        project_organizations[
            "fongim_organization_id"
        ]
    )

    partner_org_ids_current = set(
        partners[
            "fongim_organization_id"
        ].dropna()
    )

    unmatched_project_orgs = (
        project_org_ids_current
        - organization_ids_current
    )

    unmatched_partner_orgs = (
        partner_org_ids_current
        - organization_ids_current
    )

    if unmatched_project_orgs:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(unmatched_project_orgs)} "
            "project organization IDs missing "
            "from Organisations."
        )

    if unmatched_partner_orgs:
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(unmatched_partner_orgs)} "
            "partner organization IDs missing "
            "from Organisations."
        )

    orgs_per_project = (
        project_organizations
        .groupby(
            "fongim_project_id"
        )["fongim_organization_id"]
        .nunique()
    )

    multi_org_projects = (
        orgs_per_project[
            orgs_per_project > 1
        ]
    )

    if len(multi_org_projects):
        raise ValueError(
            f"FONGIM sync aborted: "
            f"{len(multi_org_projects)} projects "
            "have multiple primary organizations."
        )

    funding = funding_raw.copy()

    funding[
        "funding_index_numeric"
    ] = pd.to_numeric(
        funding["funding_index"],
        errors="coerce"
    )

    project_lookup = projects[
        [
            "fongim_project_id",
            "project_name"
        ]
    ].copy()

    project_lookup[
        "fongim_project_id_numeric"
    ] = pd.to_numeric(
        project_lookup[
            "fongim_project_id"
        ],
        errors="coerce"
    )

    funding = funding.merge(
        project_lookup,
        left_on="funding_index_numeric",
        right_on="fongim_project_id_numeric",
        how="left",
        suffixes=(
            "_funding",
            "_canonical"
        )
    )

    funding["linkage_status"] = (
        funding[
            "fongim_project_id"
        ]
        .notna()
        .map({
            True: "linked_native_id",
            False: "unlinked_native_id"
        })
    )

    funding = funding.drop(
        columns=[
            "fongim_project_id_numeric"
        ]
    )

    sync_finished_at = datetime.now(
        timezone.utc
    )

    metadata = {
        "sync_run_id": sync_run_id,
        "source_system": "FONGIM",
        "source_url": FONGIM_SOURCE_URL,
        "source_model_id": FONGIM_MODEL_ID,
        "sync_started_at":
            sync_started_at.isoformat(),
        "sync_finished_at":
            sync_finished_at.isoformat(),
        "project_count":
            len(projects),
        "location_count":
            len(locations),
        "sector_count":
            len(sectors),
        "organization_count":
            len(organizations),
        "project_organization_count":
            len(project_organizations),
        "partner_count":
            len(partners),
        "funding_row_count":
            len(funding),
        "funding_project_count":
            funding[
                "funding_index"
            ].nunique(),
        "funding_linked_project_count":
            funding.loc[
                funding[
                    "linkage_status"
                ] == "linked_native_id",
                "funding_index"
            ].nunique(),
        "funding_unlinked_project_count":
            funding.loc[
                funding[
                    "linkage_status"
                ] == "unlinked_native_id",
                "funding_index"
            ].nunique(),
    }

    print()
    print("=" * 70)
    print("SYNC VALIDATION COMPLETE")
    print("=" * 70)

    for key, value in metadata.items():
        print(f"{key}: {value}")

    return {
        "metadata":
            metadata,
        "projects":
            projects,
        "locations":
            locations,
        "sectors":
            sectors,
        "organizations":
            organizations,
        "project_organizations":
            project_organizations,
        "partners":
            partners,
        "funding":
            funding
    }


# ============================================================
# FONGIM Connector — Persistence Helpers
# ============================================================

def fongim_db_value(value):
    """
    Convert pandas / numpy values into
    Supabase-safe values.
    """

    if pd.isna(value):
        return None

    if hasattr(value, "item"):
        value = value.item()

    if isinstance(
        value,
        (pd.Timestamp, datetime)
    ):
        return value.isoformat()

    if (
        isinstance(value, float)
        and value.is_integer()
    ):
        return int(value)

    return value


def fongim_stable_key(*values):
    """
    Create deterministic SHA-256 key
    from stable source fields.
    """

    normalized = []

    for value in values:

        value = fongim_db_value(value)

        if value is None:
            normalized.append("")
        else:
            normalized.append(
                str(value).strip()
            )

    raw = "||".join(normalized)

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()


def fongim_validate_unique(
    records,
    key,
    dataset_name
):

    values = [
        record[key]
        for record in records
    ]

    if any(
        value is None
        for value in values
    ):
        raise ValueError(
            f"{dataset_name}: NULL key "
            f"detected in {key}"
        )

    if len(values) != len(set(values)):
        raise ValueError(
            f"{dataset_name}: duplicate "
            f"{key} detected"
        )


def fongim_upsert_batches(
    table_name,
    records,
    conflict_column,
    batch_size=250
):
    """
    Upsert records in controlled batches.
    """

    _require_supabase()

    written = 0

    for start in range(
        0,
        len(records),
        batch_size
    ):

        batch = records[
            start:start + batch_size
        ]

        (
            supabase
            .table(table_name)
            .upsert(
                batch,
                on_conflict=conflict_column
            )
            .execute()
        )

        written += len(batch)

    return written


def fongim_mark_missing_inactive(
    table_name,
    key_column,
    current_keys,
    sync_time
):
    """
    Mark records no longer present
    in FONGIM as inactive.

    Never deletes historical records.
    """

    _require_supabase()

    existing_keys = set()

    page_size = 1000
    start = 0

    while True:

        result = (
            supabase
            .table(table_name)
            .select(key_column)
            .range(
                start,
                start + page_size - 1
            )
            .execute()
        )

        rows = result.data or []

        for row in rows:
            existing_keys.add(
                row[key_column]
            )

        if len(rows) < page_size:
            break

        start += page_size

    missing_keys = (
        existing_keys
        - set(current_keys)
    )

    for key in missing_keys:

        (
            supabase
            .table(table_name)
            .update({
                "is_present_in_source":
                    False,
                "last_synced_at":
                    sync_time
            })
            .eq(
                key_column,
                key
            )
            .execute()
        )

    return len(missing_keys)


# ============================================================
# FONGIM Connector — Complete Synchronization
# ============================================================

def run_complete_fongim_sync():

    _require_supabase()

    print("=" * 70)
    print("COMPLETE FONGIM SYNC")
    print("=" * 70)

    print(
        "1/7  Extracting and validating FONGIM..."
    )

    source = run_fongim_sync_extract()

    metadata = source["metadata"]

    sync_run_id = metadata["sync_run_id"]
    sync_time = metadata["sync_finished_at"]

    projects_df = source[
        "projects"
    ].copy()

    organizations_df = source[
        "organizations"
    ].copy()

    project_orgs_df = source[
        "project_organizations"
    ].copy()

    locations_df = source[
        "locations"
    ].copy()

    sectors_df = source[
        "sectors"
    ].copy()

    partners_df = source[
        "partners"
    ].copy()

    funding_df = source[
        "funding"
    ].copy()

    print(
        "     ✓ Source extraction validated"
    )

    print(
        "2/7  Preparing canonical records..."
    )

    project_records = []

    for _, row in projects_df.iterrows():

        project_records.append({
            "fongim_project_id":
                fongim_db_value(
                    row[
                        "fongim_project_id"
                    ]
                ),

            "project_name":
                fongim_db_value(
                    row["project_name"]
                ),

            "start_date":
                fongim_db_value(
                    row["start_date"]
                ),

            "end_date":
                fongim_db_value(
                    row["end_date"]
                ),

            "status":
                fongim_db_value(
                    row["status"]
                ),

            "project_type":
                fongim_db_value(
                    row["project_type"]
                ),

            "beneficiaries":
                fongim_db_value(
                    row["beneficiaries"]
                ),

            "donor":
                fongim_db_value(
                    row["donor"]
                ),

            "funding_amount_raw":
                fongim_db_value(
                    row[
                        "funding_amount_raw"
                    ]
                ),

            "partners_raw":
                fongim_db_value(
                    row["partners_raw"]
                ),

            "has_partners":
                fongim_db_value(
                    row["has_partners"]
                ),

            "last_seen_at":
                sync_time,
            "last_synced_at":
                sync_time,
            "is_present_in_source":
                True,
            "last_seen_sync_id":
                sync_run_id,

            "source_system":
                "FONGIM",
            "source_model_id":
                FONGIM_MODEL_ID
        })

    fongim_validate_unique(
        project_records,
        "fongim_project_id",
        "Projects"
    )

    organization_records = []

    for _, row in organizations_df.iterrows():

        organization_records.append({
            "fongim_organization_id":
                fongim_db_value(
                    row[
                        "fongim_organization_id"
                    ]
                ),

            "organization_name":
                fongim_db_value(
                    row[
                        "organization_name"
                    ]
                ),

            "last_seen_at":
                sync_time,
            "last_synced_at":
                sync_time,
            "is_present_in_source":
                True,
            "last_seen_sync_id":
                sync_run_id,

            "source_system":
                "FONGIM",
            "source_model_id":
                FONGIM_MODEL_ID
        })

    fongim_validate_unique(
        organization_records,
        "fongim_organization_id",
        "Organizations"
    )

    project_organization_records = []

    for _, row in project_orgs_df.iterrows():

        project_organization_records.append({
            "fongim_project_id":
                fongim_db_value(
                    row[
                        "fongim_project_id"
                    ]
                ),

            "fongim_organization_id":
                fongim_db_value(
                    row[
                        "fongim_organization_id"
                    ]
                ),

            "last_seen_at":
                sync_time,
            "last_synced_at":
                sync_time,
            "is_present_in_source":
                True,
            "last_seen_sync_id":
                sync_run_id
        })

    fongim_validate_unique(
        project_organization_records,
        "fongim_project_id",
        "Project organizations"
    )

    location_records = []

    for _, row in locations_df.iterrows():

        project_id = fongim_db_value(
            row["fongim_project_id"]
        )

        zone_id = fongim_db_value(
            row["fongim_zone_id"]
        )

        region = fongim_db_value(
            row["region"]
        )

        cercle = fongim_db_value(
            row["cercle"]
        )

        commune = fongim_db_value(
            row["commune_raw"]
        )

        location_records.append({
            "record_key":
                fongim_stable_key(
                    project_id,
                    zone_id,
                    region,
                    cercle,
                    commune
                ),

            "fongim_project_id":
                project_id,
            "fongim_zone_id":
                zone_id,
            "region":
                region,
            "cercle":
                cercle,
            "commune_raw":
                commune,

            "last_seen_at":
                sync_time,
            "last_synced_at":
                sync_time,
            "is_present_in_source":
                True,
            "last_seen_sync_id":
                sync_run_id
        })

    fongim_validate_unique(
        location_records,
        "record_key",
        "Locations"
    )

    sector_records = []

    for _, row in sectors_df.iterrows():

        project_id = fongim_db_value(
            row["fongim_project_id"]
        )

        sector = fongim_db_value(
            row["sector"]
        )

        sector_other = fongim_db_value(
            row["sector_other_raw"]
        )

        sector_records.append({
            "record_key":
                fongim_stable_key(
                    project_id,
                    sector,
                    sector_other
                ),

            "fongim_project_id":
                project_id,
            "sector":
                sector,
            "sector_other_raw":
                sector_other,

            "last_seen_at":
                sync_time,
            "last_synced_at":
                sync_time,
            "is_present_in_source":
                True,
            "last_seen_sync_id":
                sync_run_id
        })

    fongim_validate_unique(
        sector_records,
        "record_key",
        "Sectors"
    )

    partner_records = []

    for _, row in partners_df.iterrows():

        project_id = fongim_db_value(
            row["fongim_project_id"]
        )

        partner_name = fongim_db_value(
            row["partner_name"]
        )

        organization_id = fongim_db_value(
            row[
                "fongim_organization_id"
            ]
        )

        partner_records.append({
            "record_key":
                fongim_stable_key(
                    project_id,
                    partner_name,
                    organization_id
                ),

            "fongim_project_id":
                project_id,

            "partner_name":
                partner_name,

            "fongim_organization_id":
                organization_id,

            "last_seen_at":
                sync_time,
            "last_synced_at":
                sync_time,
            "is_present_in_source":
                True,
            "last_seen_sync_id":
                sync_run_id
        })

    fongim_validate_unique(
        partner_records,
        "record_key",
        "Partners"
    )

    funding_records = []

    for _, row in funding_df.iterrows():

        key = fongim_stable_key(
            row["funding_id"],
            row["funding_index"],
            row["sector"]
        )

        funding_records.append({
            "record_key":
                key,

            "funding_id":
                fongim_db_value(
                    row["funding_id"]
                ),

            "funding_index":
                fongim_db_value(
                    row["funding_index"]
                ),

            "fongim_project_id":
                fongim_db_value(
                    row[
                        "fongim_project_id"
                    ]
                ),

            "organization_name":
                fongim_db_value(
                    row[
                        "organization_name"
                    ]
                ),

            "project_name_raw":
                fongim_db_value(
                    row[
                        "project_name_raw"
                    ]
                ),

            "project_type":
                fongim_db_value(
                    row["project_type"]
                ),

            "sector":
                fongim_db_value(
                    row["sector"]
                ),

            "funding_2024":
                fongim_db_value(
                    row["funding_2024"]
                ),

            "funding_2025":
                fongim_db_value(
                    row["funding_2025"]
                ),

            "linkage_status":
                fongim_db_value(
                    row["linkage_status"]
                ),

            "last_seen_at":
                sync_time,
            "last_synced_at":
                sync_time,
            "is_present_in_source":
                True,
            "last_seen_sync_id":
                sync_run_id
        })

    fongim_validate_unique(
        funding_records,
        "record_key",
        "Funding"
    )

    project_ids = {
        r["fongim_project_id"]
        for r in project_records
    }

    organization_ids = {
        r["fongim_organization_id"]
        for r in organization_records
    }

    project_child_sets = {
        "Project organizations": {
            r["fongim_project_id"]
            for r
            in project_organization_records
        },
        "Locations": {
            r["fongim_project_id"]
            for r in location_records
        },
        "Sectors": {
            r["fongim_project_id"]
            for r in sector_records
        },
        "Partners": {
            r["fongim_project_id"]
            for r in partner_records
        }
    }

    for name, ids in (
        project_child_sets.items()
    ):

        orphans = ids - project_ids

        if orphans:
            raise ValueError(
                f"{name}: orphan project IDs "
                f"detected: {sorted(orphans)}"
            )

    project_org_ids = {
        r["fongim_organization_id"]
        for r
        in project_organization_records
    }

    orphan_orgs = (
        project_org_ids
        - organization_ids
    )

    if orphan_orgs:
        raise ValueError(
            "Project organizations contain "
            "orphan organization IDs: "
            f"{sorted(orphan_orgs)}"
        )

    partner_org_ids = {
        r["fongim_organization_id"]
        for r in partner_records
        if r[
            "fongim_organization_id"
        ] is not None
    }

    orphan_partner_orgs = (
        partner_org_ids
        - organization_ids
    )

    if orphan_partner_orgs:
        raise ValueError(
            "Partners contain orphan "
            "organization IDs: "
            f"{sorted(orphan_partner_orgs)}"
        )

    print(
        "     ✓ All seven datasets prepared"
    )
    print(
        "     ✓ Cross-table integrity validated"
    )
    print()

    print(
        "     Projects:",
        len(project_records)
    )

    print(
        "     Organizations:",
        len(organization_records)
    )

    print(
        "     Project organizations:",
        len(
            project_organization_records
        )
    )

    print(
        "     Locations:",
        len(location_records)
    )

    print(
        "     Sectors:",
        len(sector_records)
    )

    print(
        "     Partners:",
        len(partner_records)
    )

    print(
        "     Funding:",
        len(funding_records)
    )

    print()
    print(
        "3/7  Registering sync run..."
    )

    sync_run_record = {
        "sync_run_id":
            sync_run_id,
        "source_system":
            metadata["source_system"],
        "source_url":
            metadata["source_url"],
        "source_model_id":
            metadata["source_model_id"],
        "sync_started_at":
            metadata["sync_started_at"],
        "sync_finished_at":
            metadata["sync_finished_at"],
        "status":
            "sync_started",

        "project_count":
            len(project_records),
        "location_count":
            len(location_records),
        "sector_count":
            len(sector_records),
        "organization_count":
            len(organization_records),

        "project_organization_count":
            len(
                project_organization_records
            ),

        "partner_count":
            len(partner_records),

        "funding_row_count":
            len(funding_records)
    }

    (
        supabase
        .table(
            "fongim_sync_runs"
        )
        .upsert(
            sync_run_record,
            on_conflict="sync_run_id"
        )
        .execute()
    )

    print(
        "     ✓ Sync run registered"
    )

    try:

        print(
            "4/7  Synchronizing "
            "parent tables..."
        )

        fongim_upsert_batches(
            "fongim_projects",
            project_records,
            "fongim_project_id"
        )

        fongim_upsert_batches(
            "fongim_organizations",
            organization_records,
            "fongim_organization_id"
        )

        print(
            "     ✓ Parent tables synchronized"
        )

        print(
            "5/7  Synchronizing "
            "relational tables..."
        )

        fongim_upsert_batches(
            "fongim_project_organizations",
            project_organization_records,
            "fongim_project_id"
        )

        fongim_upsert_batches(
            "fongim_project_locations",
            location_records,
            "record_key"
        )

        fongim_upsert_batches(
            "fongim_project_sectors",
            sector_records,
            "record_key"
        )

        fongim_upsert_batches(
            "fongim_project_partners",
            partner_records,
            "record_key"
        )

        fongim_upsert_batches(
            "fongim_project_funding",
            funding_records,
            "record_key"
        )

        print(
            "     ✓ Relational tables synchronized"
        )

        print(
            "6/7  Reconciling "
            "source presence..."
        )

        inactive_counts = {}

        inactive_counts[
            "projects"
        ] = fongim_mark_missing_inactive(
            "fongim_projects",
            "fongim_project_id",
            project_ids,
            sync_time
        )

        inactive_counts[
            "organizations"
        ] = fongim_mark_missing_inactive(
            "fongim_organizations",
            "fongim_organization_id",
            organization_ids,
            sync_time
        )

        inactive_counts[
            "project_organizations"
        ] = fongim_mark_missing_inactive(
            "fongim_project_organizations",
            "fongim_project_id",
            {
                r["fongim_project_id"]
                for r
                in project_organization_records
            },
            sync_time
        )

        inactive_counts[
            "locations"
        ] = fongim_mark_missing_inactive(
            "fongim_project_locations",
            "record_key",
            {
                r["record_key"]
                for r in location_records
            },
            sync_time
        )

        inactive_counts[
            "sectors"
        ] = fongim_mark_missing_inactive(
            "fongim_project_sectors",
            "record_key",
            {
                r["record_key"]
                for r in sector_records
            },
            sync_time
        )

        inactive_counts[
            "partners"
        ] = fongim_mark_missing_inactive(
            "fongim_project_partners",
            "record_key",
            {
                r["record_key"]
                for r in partner_records
            },
            sync_time
        )

        inactive_counts[
            "funding"
        ] = fongim_mark_missing_inactive(
            "fongim_project_funding",
            "record_key",
            {
                r["record_key"]
                for r in funding_records
            },
            sync_time
        )

        print(
            "     ✓ Source presence reconciled"
        )

        print(
            "7/7  Finalizing sync..."
        )

        (
            supabase
            .table(
                "fongim_sync_runs"
            )
            .update({
                "status":
                    "success"
            })
            .eq(
                "sync_run_id",
                sync_run_id
            )
            .execute()
        )

        print(
            "     ✓ Sync marked successful"
        )

        print()
        print("=" * 70)
        print("FONGIM SYNC COMPLETE")
        print("=" * 70)

        print(
            "Sync run:",
            sync_run_id
        )

        print(
            "Projects:",
            len(project_records)
        )

        print(
            "Organizations:",
            len(organization_records)
        )

        print(
            "Project organizations:",
            len(
                project_organization_records
            )
        )

        print(
            "Locations:",
            len(location_records)
        )

        print(
            "Sectors:",
            len(sector_records)
        )

        print(
            "Partners:",
            len(partner_records)
        )

        print(
            "Funding:",
            len(funding_records)
        )

        print()
        print(
            "Marked inactive:",
            inactive_counts
        )

        return {
            "sync_run_id":
                sync_run_id,

            "status":
                "success",

            "counts": {
                "projects":
                    len(project_records),

                "organizations":
                    len(
                        organization_records
                    ),

                "project_organizations":
                    len(
                        project_organization_records
                    ),

                "locations":
                    len(location_records),

                "sectors":
                    len(sector_records),

                "partners":
                    len(partner_records),

                "funding":
                    len(funding_records)
            },

            "marked_inactive":
                inactive_counts
        }

    except Exception as exc:

        (
            supabase
            .table(
                "fongim_sync_runs"
            )
            .update({
                "status":
                    "failed",
                "error_message":
                    str(exc)
            })
            .eq(
                "sync_run_id",
                sync_run_id
            )
            .execute()
        )

        print()
        print("FONGIM SYNC FAILED")
        print(
            "Sync run:",
            sync_run_id
        )
        print(
            "Error:",
            exc
        )

        raise
'''


runtime_path = Path(
    "/content/knowledge_hub_runtime_v0_3.py"
)

runtime_path.write_text(
    runtime_code,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

assert runtime_path.exists()

compiled = compile(
    runtime_code,
    str(runtime_path),
    "exec"
)

print(
    "✓ Runtime module created:"
)
print(runtime_path)
print()
print(
    f"✓ Size: "
    f"{runtime_path.stat().st_size:,} bytes"
)
print(
    "✓ Python syntax validation passed"
)

✓ Runtime module created:
/content/knowledge_hub_runtime_v0_3.py

✓ Size: 57,912 bytes
✓ Python syntax validation passed


In [ ]:
# 17.1 — Upload runtime module to Supabase Storage

bucket_name = "knowledge-hub-runtime"
storage_path = "knowledge_hub_runtime_v0_3.py"
local_path = "/content/knowledge_hub_runtime_v0_3.py"

with open(local_path, "rb") as f:
    file_bytes = f.read()

supabase.storage.from_(bucket_name).upload(
    path=storage_path,
    file=file_bytes,
    file_options={
        "content-type": "text/x-python",
        "upsert": "true"
    }
)

print("✓ Runtime module uploaded to Supabase Storage")
print(f"✓ Bucket: {bucket_name}")
print(f"✓ Path:   {storage_path}")

✓ Runtime module uploaded to Supabase Storage
✓ Bucket: knowledge-hub-runtime
✓ Path:   knowledge_hub_runtime_v0_3.py


## 18 — Evaluation

Benchmark v0.2 against v0.1 and reference answers.

## 19 — TEST v0.3




In [ ]:
# ============================================================
# THREE-SOURCE BACKEND TEST
# FONGIM × OCHA/HNRP × NATIONAL STRATEGIES
# Read-only — no Supabase writes
# ============================================================

import pandas as pd
from collections import Counter

# ------------------------------------------------------------
# 1. Research question
# ------------------------------------------------------------

question = """
Dans la région de Mopti, que montrent ensemble:

1. les besoins humanitaires documentés dans le HNRP 2026,
2. les priorités des stratégies nationales du Mali,
3. et la présence actuelle des projets recensés par le FONGIM ?

Identifie les principaux alignements, écarts ou angles morts possibles.
Distingue clairement les faits documentés des inférences analytiques.
Ne conclus pas qu'un faible nombre de projets signifie automatiquement
un déficit de financement ou de couverture.
"""

print("=" * 90)
print("QUESTION")
print("=" * 90)
print(question.strip())


# ------------------------------------------------------------
# 2. Pull CURRENT structured FONGIM data
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("1/4 — READING CURRENT FONGIM DATA")
print("=" * 90)

fongim = run_fongim_sync_extract()

projects = fongim["projects"].copy()
locations = fongim["locations"].copy()
sectors = fongim["sectors"].copy()
organizations = fongim["organizations"].copy()
project_orgs = fongim["project_organizations"].copy()

# Normalize region names for filtering
mopti_locations = locations[
    locations["region"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
    == "mopti"
].copy()

mopti_project_ids = set(
    mopti_locations["fongim_project_id"].dropna()
)

mopti_projects = projects[
    projects["fongim_project_id"].isin(mopti_project_ids)
].copy()

mopti_sectors = sectors[
    sectors["fongim_project_id"].isin(mopti_project_ids)
].copy()

mopti_project_orgs = project_orgs[
    project_orgs["fongim_project_id"].isin(mopti_project_ids)
].copy()

mopti_orgs = (
    mopti_project_orgs
    .merge(
        organizations,
        on="fongim_organization_id",
        how="left",
        suffixes=("", "_org")
    )
)

print(f"FONGIM projects in Mopti: {len(mopti_project_ids)}")
print(f"Location records in Mopti: {len(mopti_locations)}")
print(f"Sector links in Mopti: {len(mopti_sectors)}")


# ------------------------------------------------------------
# 3. Build a compact FONGIM evidence summary
# ------------------------------------------------------------

sector_counts = (
    mopti_sectors["sector"]
    .dropna()
    .astype(str)
    .value_counts()
)

cercle_counts = (
    mopti_locations["cercle"]
    .dropna()
    .astype(str)
    .value_counts()
)

org_name_col = None

for candidate in [
    "organization_name",
    "organization_name_org"
]:
    if candidate in mopti_orgs.columns:
        org_name_col = candidate
        break

if org_name_col:
    org_counts = (
        mopti_orgs[org_name_col]
        .dropna()
        .astype(str)
        .value_counts()
    )
else:
    org_counts = pd.Series(dtype=int)

project_examples = (
    mopti_projects[
        [
            "fongim_project_id",
            "project_name",
            "status",
            "project_type",
            "donor"
        ]
    ]
    .head(25)
    .to_dict("records")
)

fongim_evidence = f"""
FONGIM STRUCTURED DATA — REGION DE MOPTI

Unique projects with at least one Mopti location:
{len(mopti_project_ids)}

Location records:
{len(mopti_locations)}

Sector links:
{len(mopti_sectors)}

Projects by cercle:
{cercle_counts.head(20).to_string()}

Most represented FONGIM sectors:
{sector_counts.head(20).to_string()}

Organizations represented:
{org_counts.head(20).to_string()}

Illustrative project records:
{project_examples}

IMPORTANT INTERPRETATION LIMITATIONS:
- These are projects recorded in FONGIM, not necessarily all interventions in Mopti.
- Project counts are not equivalent to funding volume, intensity, effectiveness or population coverage.
- A project may occur in several locations and sectors.
- Absence or low frequency in FONGIM must not automatically be interpreted as an unmet need.
"""


# ------------------------------------------------------------
# 4. Retrieve OCHA/HNRP evidence specifically
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("2/4 — RETRIEVING OCHA / HNRP EVIDENCE")
print("=" * 90)

ocha_query = """
Mopti région cercles besoins humanitaires HNRP 2026 personnes dans le besoin
sécurité alimentaire protection santé eau hygiène assainissement déplacement
accès humanitaire vulnérabilités Mali
"""

ocha_results_raw = search_knowledge_base(
    ocha_query,
    match_count=16
)

ocha_results = [
    r for r in ocha_results_raw
    if (
        "humanitaire" in str(
            r.get("document_title", "")
        ).casefold()
        or "ocha" in str(
            r.get("organization", "")
        ).casefold()
    )
]

print(f"OCHA/HNRP evidence passages retained: {len(ocha_results)}")


# ------------------------------------------------------------
# 5. Retrieve NATIONAL STRATEGY evidence separately
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("3/4 — RETRIEVING NATIONAL STRATEGY EVIDENCE")
print("=" * 90)

strategy_query = """
Mopti développement territorial sécurité alimentaire agriculture santé eau
services sociaux gouvernance décentralisation résilience infrastructures
priorités Vision Mali 2063 SNEDD 2024-2033 projets structurants
"""

strategy_results_raw = search_knowledge_base(
    strategy_query,
    match_count=18
)

strategy_results = [
    r for r in strategy_results_raw
    if "humanitaire" not in str(
        r.get("document_title", "")
    ).casefold()
]

print(
    f"National-strategy evidence passages retained: "
    f"{len(strategy_results)}"
)


# ------------------------------------------------------------
# 6. Create explicit evidence ledger
# ------------------------------------------------------------

evidence_items = []

# FONGIM = one structured evidence item
evidence_items.append({
    "id": "F1",
    "source_type": "FONGIM structured data",
    "source": "FONGIM public project database",
    "page": None,
    "content": fongim_evidence
})

for i, r in enumerate(ocha_results, 1):
    evidence_items.append({
        "id": f"H{i}",
        "source_type": "Humanitarian document",
        "source": r.get("document_title"),
        "page": r.get("page_number"),
        "content": r.get("content", "")
    })

for i, r in enumerate(strategy_results, 1):
    evidence_items.append({
        "id": f"S{i}",
        "source_type": "National strategy",
        "source": r.get("document_title"),
        "page": r.get("page_number"),
        "content": r.get("content", "")
    })


# ------------------------------------------------------------
# 7. Format evidence for analytical model
# ------------------------------------------------------------

evidence_text = ""

for e in evidence_items:

    evidence_text += f"""
[{e['id']}]
SOURCE TYPE: {e['source_type']}
SOURCE: {e['source']}
PAGE: {e['page']}

{e['content']}

---
"""


# ------------------------------------------------------------
# 8. Ask model to reason ACROSS the three source families
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("4/4 — CROSS-SOURCE ANALYSIS")
print("=" * 90)

system_prompt = """
You are the analytical engine of the Mali Knowledge Hub.

You are given three different evidence families:

1. structured FONGIM project data;
2. humanitarian evidence from Mali's HNRP 2026;
3. Mali's national strategies and planning documents.

Your task is to reason ACROSS these evidence families.

STRICT RULES:

- Use ONLY the supplied evidence.
- Do not use outside knowledge.
- Every factual statement must be traceable to an evidence ID.
- Cite FONGIM claims with [F1].
- Cite humanitarian claims with [H#].
- Cite national-strategy claims with [S#].
- Clearly distinguish DOCUMENTED FACT from ANALYTICAL INFERENCE.
- You may identify thematic or geographic alignment when both sides
  are evidenced.
- You may identify a POSSIBLE gap or mismatch when evidence supports
  that interpretation.
- Never infer that a low number of FONGIM projects automatically means:
    * insufficient funding,
    * insufficient coverage,
    * poor performance,
    * or unmet demand.
- Project counts are not funding volumes or impact measures.
- Do not invent interventions, outcomes or causal mechanisms.
- Do not infer complete geographic coverage from selected evidence.
- Preserve uncertainty and data limitations.
- If the available evidence cannot support a comparison, say so.

Structure the answer as:

1. What the humanitarian evidence shows
2. What the national strategies prioritize
3. What FONGIM shows about the intervention landscape
4. Cross-source alignments
5. Possible gaps, tensions or blind spots
6. What cannot yet be concluded from the data

Use French.
"""

response = openai_client.responses.create(
    model="gpt-5-mini",
    instructions=system_prompt,
    input=f"""
QUESTION:

{question}

EVIDENCE LEDGER:

{evidence_text}
"""
)

print("\n")
print(response.output_text)


# ------------------------------------------------------------
# 9. Print provenance summary after the answer
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("EVIDENCE USED")
print("=" * 90)

print(
    f"FONGIM: 1 structured evidence bundle "
    f"covering {len(mopti_project_ids)} projects"
)

print(
    f"HNRP/OCHA: {len(ocha_results)} retrieved passages"
)

print(
    f"National strategies: {len(strategy_results)} retrieved passages"
)

QUESTION
Dans la région de Mopti, que montrent ensemble:

1. les besoins humanitaires documentés dans le HNRP 2026,
2. les priorités des stratégies nationales du Mali,
3. et la présence actuelle des projets recensés par le FONGIM ?

Identifie les principaux alignements, écarts ou angles morts possibles.
Distingue clairement les faits documentés des inférences analytiques.
Ne conclus pas qu'un faible nombre de projets signifie automatiquement
un déficit de financement ou de couverture.

1/4 — READING CURRENT FONGIM DATA
FONGIM SYNC EXTRACT
Sync run: 4a06bee6-75b7-4f57-8e8d-c8112ccdb8df
Started: 2026-08-28T16:59:52.031505+00:00
✓ Projects: 703 rows
✓ Locations: 2767 rows
✓ Sectors: 1369 rows
✓ Organizations: 103 rows
✓ Project organizations: 703 rows
✓ Partners: 1483 rows
✓ Funding: 328 rows

SYNC VALIDATION COMPLETE
sync_run_id: 4a06bee6-75b7-4f57-8e8d-c8112ccdb8df
source_system: FONGIM
source_url: https://www.fongim.org/carte-interactive
source_model_id: 2218134
sync_started_at: 2026-0